In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

import torch
import json
from pathlib import Path
from sklearn.model_selection import train_test_split

ROOT = Path("/kaggle/input")
OUT = Path("/kaggle/working")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

json_path = next(ROOT.rglob(
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
))

TRAIN_IMAGES = json_path.parent / "train_images"

with open(json_path) as f:
    data = json.load(f)

filenames = sorted({
    image["file_name"] for image in data["images"]
})

train_files, val_files = train_test_split(
    filenames,
    test_size=0.2,
    random_state=42
)

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "None")
print("Training images:", len(train_files))
print("Validation images:", len(val_files))
print("Training image folder:", TRAIN_IMAGES)

assert len(train_files) == 565
assert len(val_files) == 142

print("\nV3 INITIALIZATION SUCCESSFUL")


In [ ]:

# ==========================================================
# V3-02 — DYNAMIC FILAMENT CROP DATASET
# No neural-network training in this cell.
# ==========================================================

import random
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw
from collections import defaultdict

SEED = 42
CROP = 256
BATCH_SIZE = 8

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Group expert annotations by annotation-record ID.
annotations_by_image = defaultdict(list)

for ann in data["annotations"]:
    annotations_by_image[ann["image_id"]].append(ann)

train_set = set(train_files)
val_set = set(val_files)

train_records = [
    rec for rec in data["images"]
    if rec["file_name"] in train_set
]

val_records = [
    rec for rec in data["images"]
    if rec["file_name"] in val_set
]

assert train_set.isdisjoint(val_set)

class DynamicSolarDataset(Dataset):

    def __init__(self, records, training=True):
        self.records = records
        self.training = training

        # Two samples per annotation record:
        # 0 = filament-centred
        # 1 = random location
        self.samples_per_record = 2

    def __len__(self):
        return len(self.records) * self.samples_per_record

    def __getitem__(self, index):

        rec_index = index // 2
        crop_type = index % 2

        rec = self.records[rec_index]
        anns = annotations_by_image[rec["id"]]

        # Training: new locations and augmentation on each access.
        # Validation: fixed randomness, reproducible every epoch.
        rng = (
            random
            if self.training
            else random.Random(SEED + index)
        )

        image_path = TRAIN_IMAGES / rec["file_name"]

        with Image.open(image_path) as img:
            image = np.asarray(
                img.convert("L"), dtype=np.uint8
            )

        H, W = image.shape

        if crop_type == 0 and len(anns) > 0:

            # Choose one expert-annotated filament.
            ann = rng.choice(anns)

            x, y, w, h = ann["bbox"]

            # Centre around that filament, with random jitter.
            cx = x + w / 2
            cy = y + h / 2

            jitter_x = rng.uniform(-CROP/4, CROP/4)
            jitter_y = rng.uniform(-CROP/4, CROP/4)

            left = int(np.clip(
                cx - CROP/2 + jitter_x,
                0, W-CROP
            ))

            top = int(np.clip(
                cy - CROP/2 + jitter_y,
                0, H-CROP
            ))

        else:
            # Random crops can contain background or filaments.
            left = rng.randint(0, W-CROP)
            top = rng.randint(0, H-CROP)

        crop = image[
            top:top+CROP,
            left:left+CROP
        ].copy()

        # Build ground-truth binary mask for this crop.
        mask_image = Image.new("L", (CROP, CROP), 0)
        draw = ImageDraw.Draw(mask_image)

        for ann in anns:

            for polygon in ann["segmentation"]:

                points = np.asarray(
                    polygon, dtype=np.float32
                ).reshape(-1, 2)

                local_points = [
                    (float(px-left), float(py-top))
                    for px, py in points
                ]

                if len(local_points) >= 3:
                    draw.polygon(local_points, fill=1)

        mask = np.asarray(
            mask_image, dtype=np.uint8
        ).copy()

        if self.training:

            # Horizontal flip
            if rng.random() < 0.5:
                crop = np.fliplr(crop)
                mask = np.fliplr(mask)

            # Vertical flip
            if rng.random() < 0.5:
                crop = np.flipud(crop)
                mask = np.flipud(mask)

            # Rotation in multiples of 90 degrees.
            k = rng.randint(0, 3)
            crop = np.rot90(crop, k)
            mask = np.rot90(mask, k)

        # Ensure contiguous memory after flips/rotations.
        crop = np.ascontiguousarray(
            crop, dtype=np.float32
        ) / 255.0

        mask = np.ascontiguousarray(
            mask, dtype=np.float32
        )

        x = torch.from_numpy(crop).unsqueeze(0)
        y = torch.from_numpy(mask).unsqueeze(0)

        return x, y


train_dataset = DynamicSolarDataset(
    train_records, training=True
)

val_dataset = DynamicSolarDataset(
    val_records, training=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Verify dataset structure.
x_batch, y_batch = next(iter(train_loader))

print("Training annotation records:", len(train_records))
print("Validation annotation records:", len(val_records))
print("Training crops per epoch:", len(train_dataset))
print("Validation crops:", len(val_dataset))
print("Image batch:", tuple(x_batch.shape))
print("Mask batch:", tuple(y_batch.shape))
print("Sample foreground fraction:",
      round(float(y_batch.mean()), 5))

assert x_batch.shape == y_batch.shape
assert x_batch.shape[1:] == (1, 256, 256)
assert float(x_batch.min()) >= 0
assert float(x_batch.max()) <= 1
assert set(torch.unique(y_batch).tolist()).issubset({0.0, 1.0})

print("\nV3 DYNAMIC DATASET READY")


In [ ]:

# ============================================================
# V3-03 — STRONGER U-NET TRAINING
# Resumable checkpoints + validation after each epoch
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from tqdm.auto import tqdm

OUT = Path("/kaggle/working/v3_training")
OUT.mkdir(parents=True, exist_ok=True)

LATEST = OUT / "latest_training_state.pth"
BEST = OUT / "v3_unet_best.pth"
HISTORY = OUT / "training_history.csv"

MAX_EPOCHS = 20
LR = 3e-4
POS_WEIGHT = 20.0
DICE_THRESHOLD = 0.45

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# 1. STRONGER U-NET
# ------------------------------------------------------------

class DoubleConvV3(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels, out_channels, 3,
                padding=1, bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels, out_channels, 3,
                padding=1, bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNetV3(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = DoubleConvV3(1, 32)
        self.enc2 = DoubleConvV3(32, 64)
        self.enc3 = DoubleConvV3(64, 128)

        self.bottleneck = DoubleConvV3(128, 256)

        self.pool = nn.MaxPool2d(2)

        self.dec3 = DoubleConvV3(256+128, 128)
        self.dec2 = DoubleConvV3(128+64, 64)
        self.dec1 = DoubleConvV3(64+32, 32)

        self.output = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = F.interpolate(
            b, size=e3.shape[2:],
            mode="bilinear", align_corners=False
        )
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = F.interpolate(
            d3, size=e2.shape[2:],
            mode="bilinear", align_corners=False
        )
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = F.interpolate(
            d2, size=e1.shape[2:],
            mode="bilinear", align_corners=False
        )
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.output(d1)


model_v3 = UNetV3().to(device)

optimizer = torch.optim.AdamW(
    model_v3.parameters(),
    lr=LR,
    weight_decay=1e-4
)

positive_weight = torch.tensor(
    [POS_WEIGHT], device=device
)

# ------------------------------------------------------------
# 2. LOSS AND METRICS
# ------------------------------------------------------------

def combined_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        pos_weight=positive_weight
    )

    probabilities = torch.sigmoid(logits)

    intersection = (
        probabilities * targets
    ).sum(dim=(1, 2, 3))

    denominator = (
        probabilities.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
    )

    dice_loss = 1 - (
        (2 * intersection + 1) /
        (denominator + 1)
    ).mean()

    return bce + dice_loss


@torch.no_grad()
def validate(model):
    model.eval()

    total_dice = 0.0
    total_loss = 0.0
    count = 0

    for images, masks in val_loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        logits = model(images)

        loss = combined_loss(logits, masks)

        pred = (
            torch.sigmoid(logits) >= DICE_THRESHOLD
        ).float()

        intersection = (
            pred * masks
        ).sum(dim=(1, 2, 3))

        denominator = (
            pred.sum(dim=(1, 2, 3))
            + masks.sum(dim=(1, 2, 3))
        )

        # Empty predictions against empty masks receive Dice = 1.
        dice = torch.where(
            denominator == 0,
            torch.ones_like(denominator),
            (2 * intersection) / denominator.clamp_min(1)
        )

        batch_count = images.shape[0]

        total_dice += dice.sum().item()
        total_loss += loss.item() * batch_count
        count += batch_count

    return total_loss / count, total_dice / count


# ------------------------------------------------------------
# 3. RESUME CHECKPOINT IF AVAILABLE
# ------------------------------------------------------------

start_epoch = 0
best_dice = -1.0

if LATEST.exists():
    state = torch.load(
        LATEST,
        map_location=device,
        weights_only=False
    )

    model_v3.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])

    start_epoch = int(state["epoch"])
    best_dice = float(state["best_dice"])

    if "torch_rng" in state:
        torch.set_rng_state(state["torch_rng"].cpu())
    if torch.cuda.is_available() and "cuda_rng" in state:
        torch.cuda.set_rng_state_all(state["cuda_rng"])

    if "numpy_rng" in state:
        np.random.set_state(state["numpy_rng"])
    if "python_rng" in state:
        random.setstate(state["python_rng"])

    print(f"Resuming from completed epoch {start_epoch}")

else:
    print("Starting new V3 training.")

print(
    "Model parameters:",
    f"{sum(p.numel() for p in model_v3.parameters()):,}"
)

# ------------------------------------------------------------
# 4. TRAINING LOOP
# ------------------------------------------------------------

history = (
    pd.read_csv(HISTORY).to_dict("records")
    if HISTORY.exists() else []
)

for epoch in range(start_epoch, MAX_EPOCHS):

    model_v3.train()

    running_loss = 0.0
    seen = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{MAX_EPOCHS}"
    )

    for images, masks in progress:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model_v3(images)
        loss = combined_loss(logits, masks)

        loss.backward()
        optimizer.step()

        batch_count = images.shape[0]
        running_loss += loss.item() * batch_count
        seen += batch_count

        progress.set_postfix(
            loss=f"{running_loss / seen:.4f}"
        )

    train_loss = running_loss / seen

    val_loss, val_dice = validate(model_v3)

    improved = val_dice > best_dice

    if improved:
        best_dice = val_dice

        # Atomic checkpoint replacement
        temp_best = OUT / "temp_best.pth"
        torch.save(model_v3.state_dict(), temp_best)
        os.replace(temp_best, BEST)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_dice": val_dice,
        "best_dice": best_dice
    })

    pd.DataFrame(history).to_csv(
        HISTORY, index=False
    )

    state = {
        "epoch": epoch + 1,
        "model": model_v3.state_dict(),
        "optimizer": optimizer.state_dict(),
        "best_dice": best_dice,
        "torch_rng": torch.get_rng_state(),
        "numpy_rng": np.random.get_state(),
        "python_rng": random.getstate()
    }

    if torch.cuda.is_available():
        state["cuda_rng"] = torch.cuda.get_rng_state_all()

    temp_latest = OUT / "temp_training_state.pth"
    torch.save(state, temp_latest)
    os.replace(temp_latest, LATEST)

    print(
        f"\nEpoch {epoch+1}/{MAX_EPOCHS} | "
        f"Train loss={train_loss:.4f} | "
        f"Val loss={val_loss:.4f} | "
        f"Val Dice={val_dice:.4f} | "
        f"Best={best_dice:.4f}"
    )

    print("Best checkpoint saved:", BEST.exists())
    print("-" * 60)

print("\nV3 TRAINING COMPLETED")
print("Best validation Dice:", round(best_dice, 5))
print("Best model:", BEST)
print("History:", HISTORY)


In [ ]:

from pathlib import Path
import shutil
import json
from kaggle.api.kaggle_api_extended import KaggleApi

source = Path("/kaggle/working/v3_training")
backup = Path("/kaggle/working/v3_checkpoint_backup")
backup.mkdir(exist_ok=True)

files = [
    "v3_unet_best.pth",
    "latest_training_state.pth",
    "training_history.csv"
]

for name in files:
    path = source / name
    assert path.exists() and path.stat().st_size > 0, f"Missing: {name}"
    shutil.copy2(path, backup / name)
    print("Preserved:", name, round(path.stat().st_size/1e6, 2), "MB")

metadata = {
    "title": "Solar Filament V3 Training Checkpoints",
    "id": "asaadwahag/solar-filament-v3-checkpoints",
    "licenses": [{"name": "CC0-1.0"}]
}

(backup / "dataset-metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

api = KaggleApi()
api.authenticate()
api.dataset_create_new(str(backup), public=False)

print("\nSUCCESS — V3 model and training history backed up.")
print("Dataset: asaadwahag/solar-filament-v3-checkpoints")


In [ ]:

# V3-04A — Full-image inference on 12 validation photographs
# No training. Resumable if interrupted.

import json
import numpy as np
import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split

ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
CACHE = WORK / "v3_pilot_probability_maps"
CACHE.mkdir(parents=True, exist_ok=True)

# Original dataset and same physical-image split
annotation_path = next(ROOT.rglob(
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
))
TRAIN_IMAGES = annotation_path.parent / "train_images"

with open(annotation_path) as f:
    coco = json.load(f)

filenames = sorted({
    r["file_name"] for r in coco["images"]
})

_, val_files = train_test_split(
    filenames, test_size=0.20, random_state=42
)

rng = np.random.default_rng(42)
pilot_files = sorted(
    rng.choice(sorted(val_files), 12, replace=False).tolist()
)

# Rebuild V3 and load the BEST model, not epoch 20's
# latest training state (which may differ in other runs).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert "UNetV3" in globals(), (
    "Run the V3 architecture-definition cell first. "
    "Do not rerun the training loop."
)

model = UNetV3().to(device)
model.load_state_dict(
    torch.load(
        "/kaggle/working/v3_training/v3_unet_best.pth",
        map_location=device,
        weights_only=True
    )
)
model.eval()

PATCH = 256
STRIDE = 128
BATCH = 8

def predict_v3_full(path):
    image = np.asarray(
        Image.open(path).convert("L"),
        dtype=np.float32
    ) / 255.0

    h, w = image.shape
    assert (h, w) == (2048, 2048)

    positions = [
        (y, x)
        for y in range(0, h-PATCH+1, STRIDE)
        for x in range(0, w-PATCH+1, STRIDE)
    ]

    total = np.zeros((h, w), dtype=np.float32)
    weights = np.zeros((h, w), dtype=np.float32)

    with torch.inference_mode():
        for start in range(0, len(positions), BATCH):
            current = positions[start:start+BATCH]
            patches = np.stack([
                image[y:y+PATCH, x:x+PATCH]
                for y, x in current
            ])

            batch = torch.from_numpy(
                patches.copy()
            ).unsqueeze(1).to(device)

            probs = torch.sigmoid(
                model(batch)
            ).cpu().numpy()[:, 0]

            for prob, (y, x) in zip(probs, current):
                total[y:y+PATCH, x:x+PATCH] += prob
                weights[y:y+PATCH, x:x+PATCH] += 1

    assert np.all(weights > 0)
    return total / weights

print("Device:", device)
print("Pilot images:", len(pilot_files))

for i, filename in enumerate(pilot_files, 1):
    output = CACHE / (Path(filename).stem + ".npy")

    if not output.exists():
        prob = predict_v3_full(TRAIN_IMAGES / filename)
        np.save(output, prob)

    print(f"{i}/12 — {filename}")

print("\nV3 PILOT INFERENCE COMPLETE")
print("Cached maps:", len(list(CACHE.glob("*.npy"))))
print("Cache:", CACHE)


In [ ]:

# V3-04B — Reference-style instance PQ pilot
# Uses all annotation records per photograph.

import json
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageDraw
from collections import defaultdict

ROOT = Path("/kaggle/input")
OUT = Path("/kaggle/working")
CACHE = OUT / "v3_pilot_probability_maps"

json_path = next(ROOT.rglob(
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
))
IMAGE_DIR = json_path.parent / "train_images"

with open(json_path) as f:
    coco = json.load(f)

records = defaultdict(list)
for rec in coco["images"]:
    records[rec["file_name"]].append(rec)

anns = defaultdict(list)
for ann in coco["annotations"]:
    anns[ann["image_id"]].append(ann)

THRESHOLDS = [0.35, 0.45, 0.55]
AREAS = [200, 400, 800]
RADIUS_OPTIONS = [None, 0.90]

configs = [
    (t, a, r)
    for t in THRESHOLDS
    for a in AREAS
    for r in RADIUS_OPTIONS
]

def disk_geometry(filename):
    gray = cv2.imread(
        str(IMAGE_DIR / filename), cv2.IMREAD_GRAYSCALE
    )
    assert gray is not None

    blur = cv2.GaussianBlur(gray, (15, 15), 0)
    _, binary = cv2.threshold(
        blur, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    contours, _ = cv2.findContours(
        binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    assert contours, f"No solar disk: {filename}"

    (cx, cy), radius = cv2.minEnclosingCircle(
        max(contours, key=cv2.contourArea)
    )
    assert 512 < radius < 1126, (
        f"Check disk estimate for {filename}: {radius}"
    )
    return cx, cy, radius


def expert_masks(rec):
    w, h = int(rec["width"]), int(rec["height"])
    output = []

    for ann in anns[rec["id"]]:
        im = Image.new("1", (w, h), 0)
        draw = ImageDraw.Draw(im)

        for poly in ann["segmentation"]:
            pts = np.asarray(poly).reshape(-1, 2)
            if len(pts) >= 3:
                draw.polygon(
                    [tuple(map(float, p)) for p in pts],
                    fill=1
                )

        mask = np.asarray(im, dtype=bool)
        if mask.any():
            output.append(mask)

    return output


rows = []

assert len(pilot_files) == 12
assert len(list(CACHE.glob("*.npy"))) == 12

for k, filename in enumerate(pilot_files, 1):
    prob = np.load(
        CACHE / (Path(filename).stem + ".npy"),
        mmap_mode="r"
    )
    assert prob.shape == (2048, 2048)

    cx, cy, radius = disk_geometry(filename)

    # Evaluate the same prediction against each expert record.
    gt_records = [
        expert_masks(rec) for rec in records[filename]
    ]

    for threshold in THRESHOLDS:
        binary = (prob > threshold).astype(np.uint8)

        n, labels, stats, centroids = (
            cv2.connectedComponentsWithStats(
                binary, connectivity=8
            )
        )

        distances = np.hypot(
            centroids[:, 0] - cx,
            centroids[:, 1] - cy
        ) / radius

        # Candidate IoUs, calculated once per expert record.
        record_hits = []

        for gt_masks in gt_records:
            matches = []

            for gt_index, gt in enumerate(gt_masks):
                intersection = np.bincount(
                    labels[gt].ravel(), minlength=n
                )

                ids = np.flatnonzero(intersection)
                ids = ids[ids != 0]

                if len(ids) == 0:
                    continue

                union = (
                    stats[ids, cv2.CC_STAT_AREA]
                    + int(gt.sum()) - intersection[ids]
                )

                ious = intersection[ids] / union

                for pred_id, iou in zip(ids, ious):
                    if iou > 0.5:
                        matches.append(
                            (int(pred_id), gt_index, float(iou))
                        )

            record_hits.append((len(gt_masks), matches))

        for area in AREAS:
            selected_ids = np.flatnonzero(
                stats[:, cv2.CC_STAT_AREA] >= area
            )
            selected_ids = selected_ids[selected_ids != 0]

            for limit in RADIUS_OPTIONS:
                selected = (
                    selected_ids
                    if limit is None
                    else selected_ids[
                        distances[selected_ids] <= limit
                    ]
                )
                selected_set = set(selected.tolist())

                for record_index, (n_gt, matches) in enumerate(
                    record_hits
                ):
                    hit_preds = set()
                    hit_gts = set()
                    tp = 0
                    iou_sum = 0.0

                    # Match the organisers' hit-counting logic.
                    for pred_id, gt_id, iou in matches:
                        if pred_id in selected_set:
                            tp += 1
                            iou_sum += iou
                            hit_preds.add(pred_id)
                            hit_gts.add(gt_id)

                    rows.append({
                        "filename": filename,
                        "record_index": record_index,
                        "threshold": threshold,
                        "min_area": area,
                        "radius_limit": (
                            "none" if limit is None else "0.90"
                        ),
                        "TP": tp,
                        "FP": len(selected_set) - len(hit_preds),
                        "FN": n_gt - len(hit_gts),
                        "sum_iou": iou_sum
                    })

    print(f"Evaluated {k}/12: {filename}")

details = pd.DataFrame(rows)
details.to_csv(
    OUT / "v3_pilot_pq_details.csv", index=False
)

summary = details.groupby(
    ["threshold", "min_area", "radius_limit"],
    as_index=False
)[["TP", "FP", "FN", "sum_iou"]].sum()

denom = summary.TP + 0.5 * (summary.FP + summary.FN)
summary["PQ_reference_style"] = np.where(
    denom > 0, summary.sum_iou / denom, 0
)

summary = summary.sort_values(
    "PQ_reference_style", ascending=False
).reset_index(drop=True)

summary.to_csv(
    OUT / "v3_pilot_pq_results.csv", index=False
)

print("\nTOP 12 V3 CONFIGURATIONS")
print(summary[[
    "threshold", "min_area", "radius_limit",
    "PQ_reference_style", "TP", "FP", "FN"
]].head(12).to_string(index=False))

print("\nBEST PILOT CONFIGURATION")
print(summary.iloc[0].to_string())

print("\nSaved:")
print(OUT / "v3_pilot_pq_results.csv")
print(OUT / "v3_pilot_pq_details.csv")


In [ ]:

from pathlib import Path
import json
import shutil
from kaggle.api.kaggle_api_extended import KaggleApi

WORK = Path("/kaggle/working")
BACKUP = WORK / "v3_validation_backup"
BACKUP.mkdir(exist_ok=True)

files = [
    "v3_pilot_pq_results.csv",
    "v3_pilot_pq_details.csv",
    "v3_validation_130_summary.csv",
    "v3_validation_130_details.csv"
]

for name in files:
    source = WORK / name
    assert source.exists(), f"Missing file: {name}"
    shutil.copy2(source, BACKUP / name)

experiment = {
    "version": "V3",
    "best_checkpoint": "v3_unet_best.pth",
    "validation_photographs": 130,
    "selected_threshold": 0.55,
    "selected_min_area": 800,
    "selected_radius_filter": None,
    "reference_style_pq": 0.273674,
    "v2_reference_style_pq": 0.093695,
    "status": "Ready for test submission",
    "note": "Local reference-style PQ, not Kaggle public PQ"
}

(BACKUP / "experiment_config.json").write_text(
    json.dumps(experiment, indent=2)
)

metadata = {
    "title": "Solar Filament V3 Validation Results",
    "id": "asaadwahag/solar-filament-v3-validation-results",
    "licenses": [{"name": "CC0-1.0"}]
}

(BACKUP / "dataset-metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

api = KaggleApi()
api.authenticate()

api.dataset_create_new(
    str(BACKUP),
    public=False,
    quiet=False
)

print("SUCCESS — V3 validation results backed up.")
print("Selected V3: threshold=0.55, area=800, no radius filter")


In [ ]:

# V3-04C — Independent 130-image validation
# No training. Reuses trained model and existing functions.

import numpy as np
import pandas as pd
import cv2
from pathlib import Path

OUT = Path("/kaggle/working")
CACHE130 = OUT / "v3_validation_130_cache"
CACHE130.mkdir(exist_ok=True)

RESULT = OUT / "v3_validation_130_details.csv"

remaining = sorted(set(val_files) - set(pilot_files))
assert len(remaining) == 130

assert callable(predict_v3_full)
assert callable(expert_masks)
assert callable(disk_geometry)

CONFIGS = [
    ("A_035_800_r90", 0.35, 800, 0.90),
    ("B_055_800_none", 0.55, 800, None),
    ("C_055_400_r90", 0.55, 400, 0.90)
]

# Resume completed photographs if available
if RESULT.exists():
    previous = pd.read_csv(RESULT)
    completed = set(previous["filename"])
    rows = previous.to_dict("records")
else:
    completed = set()
    rows = []

print("Already evaluated:", len(completed))
print("Remaining photographs:", len(remaining))

for k, filename in enumerate(remaining, 1):

    if filename in completed:
        continue

    cache_file = CACHE130 / (Path(filename).stem + ".npy")

    if cache_file.exists():
        prob = np.load(cache_file, mmap_mode="r")
    else:
        prob = predict_v3_full(TRAIN_IMAGES / filename)
        np.save(cache_file, prob)

    cx, cy, radius = disk_geometry(filename)
    gt_records = [expert_masks(rec) for rec in records[filename]]

    for threshold in sorted(set(c[1] for c in CONFIGS)):

        binary = (prob > threshold).astype(np.uint8)

        n, labels, stats, centroids = (
            cv2.connectedComponentsWithStats(binary, connectivity=8)
        )

        distances = np.hypot(
            centroids[:, 0] - cx,
            centroids[:, 1] - cy
        ) / radius

        # Calculate qualifying IoUs once per annotation record.
        hit_records = []

        for gt_masks in gt_records:
            hits = []

            for gt_id, gt in enumerate(gt_masks):
                intersections = np.bincount(
                    labels[gt].ravel(), minlength=n
                )

                ids = np.flatnonzero(intersections)
                ids = ids[ids != 0]

                if len(ids) == 0:
                    continue

                union = (
                    stats[ids, cv2.CC_STAT_AREA]
                    + int(gt.sum()) - intersections[ids]
                )

                ious = intersections[ids] / union

                for pred_id, iou in zip(ids, ious):
                    if iou > 0.5:
                        hits.append(
                            (int(pred_id), gt_id, float(iou))
                        )

            hit_records.append((len(gt_masks), hits))

        for name, t, area, limit in CONFIGS:
            if t != threshold:
                continue

            selected = np.flatnonzero(
                stats[:, cv2.CC_STAT_AREA] >= area
            )
            selected = selected[selected != 0]

            if limit is not None:
                selected = selected[distances[selected] <= limit]

            selected_set = set(selected.tolist())

            for record_id, (n_gt, hits) in enumerate(hit_records):
                hit_preds = set()
                hit_gts = set()
                tp = 0
                iou_sum = 0.0

                for pred_id, gt_id, iou in hits:
                    if pred_id in selected_set:
                        tp += 1
                        iou_sum += iou
                        hit_preds.add(pred_id)
                        hit_gts.add(gt_id)

                rows.append({
                    "filename": filename,
                    "annotation_record": record_id,
                    "configuration": name,
                    "TP": tp,
                    "FP": len(selected_set) - len(hit_preds),
                    "FN": n_gt - len(hit_gts),
                    "sum_iou": iou_sum
                })

    completed.add(filename)

    # Save every completed image
    pd.DataFrame(rows).to_csv(RESULT, index=False)

    if len(completed) % 10 == 0:
        print(f"Completed {len(completed)}/130")

details = pd.read_csv(RESULT)

summary = details.groupby(
    "configuration", as_index=False
)[["TP", "FP", "FN", "sum_iou"]].sum()

denom = summary.TP + 0.5 * (summary.FP + summary.FN)
summary["PQ_reference_style"] = summary.sum_iou / denom

summary = summary.sort_values(
    "PQ_reference_style", ascending=False
)

summary.to_csv(
    OUT / "v3_validation_130_summary.csv",
    index=False
)

print("\nV3 INDEPENDENT VALIDATION:")
print(summary.to_string(index=False))

print("\nV2 reference-style PQ: 0.093695")
print("Best V3 PQ:", round(float(summary.iloc[0].PQ_reference_style), 6))
print("Photographs completed:", len(completed), "/130")


In [ ]:
print("Kaggle is working again")

In [ ]:
from pathlib import Path

work = Path("/kaggle/working")

print("Best model:", (work / "v3_training/v3_unet_best.pth").exists())

parts = list((work / "v3_test_parts").glob("*.csv"))
print("Completed test predictions:", len(parts), "/180")

print("Final submission:", (work / "submission_v3.csv").exists())

In [ ]:

import time
import torch
from pathlib import Path

print("GPU:", torch.cuda.is_available(), flush=True)

if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0), flush=True)

test_dirs = [
    p for p in Path("/kaggle/input").rglob("test_images")
    if p.is_dir()
]

print("Test folders:", len(test_dirs), flush=True)

assert torch.cuda.is_available(), "GPU is not available."
assert len(test_dirs) >= 1, "Test images not found."

test_images = sorted(test_dirs[0].glob("*.jpeg"))
print("Test images:", len(test_images), flush=True)

assert len(test_images) == 180
assert callable(predict_v3_full), (
    "The inference function is missing. "
    "Do not rerun training."
)

start = time.time()

prob = predict_v3_full(test_images[0])

torch.cuda.synchronize()

elapsed = time.time() - start

print("Prediction shape:", prob.shape, flush=True)
print("One-image inference time:", round(elapsed, 2), "seconds", flush=True)
print("SUCCESS — V3 GPU inference works.", flush=True)


In [ ]:

# V3 — Restore full-image inference function
# No training and no model inference in this cell.

import numpy as np
import torch
from PIL import Image

def predict_v3_full(image_path):
    image = np.asarray(
        Image.open(image_path).convert("L"),
        dtype=np.float32
    ) / 255.0

    h, w = image.shape
    assert (h, w) == (2048, 2048)

    patch_size = 256
    stride = 128
    batch_size = 8

    positions = [
        (y, x)
        for y in range(0, h - patch_size + 1, stride)
        for x in range(0, w - patch_size + 1, stride)
    ]

    total = np.zeros((h, w), dtype=np.float32)
    counts = np.zeros((h, w), dtype=np.float32)

    model.eval()

    with torch.inference_mode():
        for start in range(0, len(positions), batch_size):
            current = positions[start:start + batch_size]

            patches = np.stack([
                image[y:y+patch_size, x:x+patch_size]
                for y, x in current
            ])

            batch = torch.from_numpy(
                patches.copy()
            ).unsqueeze(1).to(device)

            predictions = torch.sigmoid(
                model(batch)
            ).cpu().numpy()[:, 0]

            for p, (y, x) in zip(predictions, current):
                total[y:y+patch_size, x:x+patch_size] += p
                counts[y:y+patch_size, x:x+patch_size] += 1

    return total / counts

print("Inference function restored.")

print("Model exists:", "model" in globals())
print("Device exists:", "device" in globals())


In [ ]:

import torch
from pathlib import Path

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint = Path(
    "/kaggle/working/v3_training/v3_unet_best.pth"
)

print("Device:", device)
print("Checkpoint exists:", checkpoint.exists())
print("UNetV3 class available:", "UNetV3" in globals())

if checkpoint.exists():
    print("Checkpoint size:",
          round(checkpoint.stat().st_size / 1e6, 2),
          "MB")


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNetV3(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = DoubleConv(1, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.bottleneck = DoubleConv(128, 256)

        self.pool = nn.MaxPool2d(2)

        self.dec3 = DoubleConv(384, 128)
        self.dec2 = DoubleConv(192, 64)
        self.dec1 = DoubleConv(96, 32)

        self.output = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))

        d3 = F.interpolate(b, size=e3.shape[2:],
                           mode="bilinear", align_corners=False)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = F.interpolate(d3, size=e2.shape[2:],
                           mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = F.interpolate(d2, size=e1.shape[2:],
                           mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.output(d1)


# Load the saved V3 checkpoint
model = UNetV3().to(device)

state = torch.load(
    checkpoint,
    map_location=device,
    weights_only=True
)

model.load_state_dict(state, strict=True)
model.eval()

print("SUCCESS — V3 checkpoint loaded!")
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Device:", device)
print("Evaluation mode:", not model.training)


In [ ]:

# V3 — Test one image and measure inference time
# No training.

import time
import numpy as np
import torch
from pathlib import Path

test_dirs = [
    p for p in Path("/kaggle/input").rglob("test_images")
    if p.is_dir()
]

assert len(test_dirs) == 1
test_images = sorted(test_dirs[0].glob("*.jpeg"))
assert len(test_images) == 180

assert callable(predict_v3_full)
assert not model.training

print("Starting one-image V3 inference...", flush=True)

start = time.perf_counter()

prob = predict_v3_full(test_images[0])

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

assert prob.shape == (2048, 2048)

print("\nSUCCESS — One image processed!")
print("Image:", test_images[0].name)
print("Time:", round(elapsed, 2), "seconds")
print("Probability range:",
      round(float(prob.min()), 4),
      round(float(prob.max()), 4))
print("Estimated 180-image time:",
      round(elapsed * 180 / 60, 1), "minutes")


In [ ]:

# V3 — Generate resumable Kaggle submission
# No training. Uses the restored best V3 model.

import time
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from pycocotools import mask as mask_utils

WORK = Path("/kaggle/working")
PARTS = WORK / "v3_test_parts"
PARTS.mkdir(parents=True, exist_ok=True)

THRESHOLD = 0.55
MIN_AREA = 800

assert not model.training
assert len(test_images) == 180

def create_predictions(image_path):
    prob = predict_v3_full(image_path)
    binary = (prob > THRESHOLD).astype(np.uint8)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )

    rows = []
    number = 0

    for component_id in range(1, n):
        area = int(stats[component_id, cv2.CC_STAT_AREA])

        if area < MIN_AREA:
            continue

        mask = np.asfortranarray(
            (labels == component_id).astype(np.uint8)
        )

        rle = mask_utils.encode(mask)["counts"].decode("ascii")
        number += 1

        rows.append({
            "filament_id": f"{image_path.stem}_{number}",
            "segmentation_rle": rle
        })

    return rows


overall_start = time.perf_counter()

for index, path in enumerate(test_images, 1):
    part_file = PARTS / (path.stem + ".csv")

    if part_file.exists():
        print(f"{index}/180 — already saved", flush=True)
        continue

    start = time.perf_counter()
    predictions = create_predictions(path)

    pd.DataFrame(
        predictions,
        columns=["filament_id", "segmentation_rle"]
    ).to_csv(part_file, index=False)

    elapsed = time.perf_counter() - start

    print(
        f"{index}/180 — {path.name} — "
        f"{len(predictions)} filaments — {elapsed:.1f}s",
        flush=True
    )

# Combine all completed photograph CSVs
frames = []

for path in test_images:
    file = PARTS / (path.stem + ".csv")
    assert file.exists(), f"Missing {path.name}"
    frames.append(pd.read_csv(file))

submission = pd.concat(frames, ignore_index=True)

assert submission["filament_id"].is_unique
assert submission.notna().all().all()

OUTPUT = WORK / "submission_v3.csv"
submission.to_csv(OUTPUT, index=False)

print("\nSUCCESS — V3 SUBMISSION READY")
print("Predicted instances:", len(submission))
print("Photographs with predictions:",
      submission["filament_id"].str.rsplit(
          "_", n=1
      ).str[0].nunique())
print("File:", OUTPUT)
print("Total runtime:",
      round((time.perf_counter() - overall_start)/60, 2),
      "minutes")


In [ ]:

# V3-07 — Validate and permanently back up submission

from pathlib import Path
import json
import shutil
import pandas as pd
from pycocotools import mask as mask_utils
from kaggle.api.kaggle_api_extended import KaggleApi

WORK = Path("/kaggle/working")
SOURCE = WORK / "submission_v3.csv"
BACKUP = WORK / "v3_submission_backup"
BACKUP.mkdir(exist_ok=True)

df = pd.read_csv(SOURCE)

assert list(df.columns) == [
    "filament_id", "segmentation_rle"
]
assert df["filament_id"].is_unique
assert df.notna().all().all()
assert len(test_images) == 180

test_stems = {p.stem for p in test_images}
pred_stems = set(
    df["filament_id"].str.rsplit("_", n=1).str[0]
)
assert pred_stems.issubset(test_stems)

# Check every RLE entry
for rle in df["segmentation_rle"]:
    mask = mask_utils.decode({
        "size": [2048, 2048],
        "counts": rle.encode("ascii")
    })
    assert mask.shape == (2048, 2048)
    assert int(mask.sum()) >= 800

print("All RLE masks verified:", len(df), flush=True)

shutil.copy2(SOURCE, BACKUP / SOURCE.name)

config = {
    "model": "UNetV3",
    "checkpoint": "v3_unet_best.pth",
    "threshold": 0.55,
    "minimum_area": 800,
    "radius_filter": None,
    "test_photographs": 180,
    "predicted_instances": len(df),
    "images_with_predictions": len(pred_stems),
    "local_validation_PQ": 0.273674
}

(BACKUP / "submission_config.json").write_text(
    json.dumps(config, indent=2)
)

metadata = {
    "title": "Solar Filament V3 Submission",
    "id": "asaadwahag/solar-filament-v3-submission",
    "licenses": [{"name": "CC0-1.0"}]
}

(BACKUP / "dataset-metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

api = KaggleApi()
api.authenticate()

api.dataset_create_new(
    str(BACKUP),
    public=False,
    quiet=False
)

print("\nSUCCESS — V3 submission verified and backed up.")
print("Dataset: asaadwahag/solar-filament-v3-submission")


In [ ]:

from pathlib import Path

root = Path("/kaggle/working")

folders = [
    "v3_pilot_probability_maps",
    "v3_validation_130_cache"
]

for name in folders:
    folder = root / name
    maps = list(folder.glob("*.npy")) if folder.exists() else []

    print(f"{name}: {len(maps)} probability maps")

print("\nV3 checkpoint:",
      (root / "v3_training/v3_unet_best.pth").exists())

print("V3 validation CSV:",
      (root / "v3_validation_130_details.csv").exists())


In [ ]:

# V4-A — Threshold and minimum-area optimization
# 130 validation photos; no training; no CNN inference.

import json
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageDraw
from collections import defaultdict
from sklearn.model_selection import train_test_split

ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
CACHE = WORK / "v3_validation_130_cache"
RESULT = WORK / "v4a_validation_details.csv"

THRESHOLDS = [0.45, 0.50, 0.55, 0.60, 0.65]
AREAS = [600, 800, 1000, 1200, 1600]

# Read annotations
annotation_path = next(ROOT.rglob(
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
))

with open(annotation_path) as f:
    coco = json.load(f)

image_records = defaultdict(list)
for rec in coco["images"]:
    image_records[rec["file_name"]].append(rec)

annotations = defaultdict(list)
for ann in coco["annotations"]:
    annotations[ann["image_id"]].append(ann)

# Reconstruct original 565/142 physical-image split
filenames = sorted(image_records)

_, val_files_v4 = train_test_split(
    filenames, test_size=0.20, random_state=42
)

pilot = set(
    np.random.default_rng(42).choice(
        sorted(val_files_v4), 12, replace=False
    )
)

remaining = sorted(set(val_files_v4) - pilot)
assert len(remaining) == 130

# Check every probability map
for filename in remaining:
    path = CACHE / (Path(filename).stem + ".npy")
    assert path.exists(), f"Missing: {path}"

def get_ground_truth(rec):
    w, h = int(rec["width"]), int(rec["height"])
    masks = []

    for ann in annotations[rec["id"]]:
        canvas = Image.new("1", (w, h), 0)
        drawer = ImageDraw.Draw(canvas)

        for polygon in ann["segmentation"]:
            pts = np.asarray(polygon).reshape(-1, 2)
            if len(pts) >= 3:
                drawer.polygon(
                    [tuple(map(float, p)) for p in pts],
                    fill=1
                )

        mask = np.asarray(canvas, dtype=bool)
        if mask.any():
            masks.append(mask)

    return masks

# Support resuming after an interruption
if RESULT.exists():
    previous = pd.read_csv(RESULT)
    completed = set(previous.filename.unique())
    rows = previous.to_dict("records")
else:
    completed = set()
    rows = []

print("V4-A configurations:", len(THRESHOLDS)*len(AREAS))
print("Previously evaluated images:", len(completed))
print("Starting evaluation...", flush=True)

for index, filename in enumerate(remaining, 1):
    if filename in completed:
        continue

    probability = np.load(
        CACHE / (Path(filename).stem + ".npy"),
        mmap_mode="r"
    )

    gt_records = [
        get_ground_truth(rec)
        for rec in image_records[filename]
    ]

    image_rows = []

    for threshold in THRESHOLDS:
        binary = (probability > threshold).astype(np.uint8)

        n, labels, stats, _ = cv2.connectedComponentsWithStats(
            binary, connectivity=8
        )

        component_areas = stats[:, cv2.CC_STAT_AREA]

        # Calculate qualifying matches for each annotator
        gt_match_records = []

        for gt_masks in gt_records:
            hits = []

            for gt_id, gt in enumerate(gt_masks):
                intersections = np.bincount(
                    labels[gt].ravel(), minlength=n
                )

                pred_ids = np.flatnonzero(intersections)
                pred_ids = pred_ids[pred_ids != 0]

                if not len(pred_ids):
                    continue

                unions = (
                    component_areas[pred_ids]
                    + int(gt.sum())
                    - intersections[pred_ids]
                )

                ious = intersections[pred_ids] / unions

                for pred_id, iou in zip(pred_ids, ious):
                    if iou > 0.5:
                        hits.append(
                            (int(pred_id), gt_id, float(iou))
                        )

            gt_match_records.append((len(gt_masks), hits))

        for area in AREAS:
            selected = set(
                np.flatnonzero(component_areas >= area).tolist()
            )
            selected.discard(0)

            for record_id, (n_gt, hits) in enumerate(
                gt_match_records
            ):
                matched_preds = set()
                matched_gts = set()
                tp = 0
                sum_iou = 0.0

                for pred_id, gt_id, iou in hits:
                    if pred_id in selected:
                        tp += 1
                        sum_iou += iou
                        matched_preds.add(pred_id)
                        matched_gts.add(gt_id)

                image_rows.append({
                    "filename": filename,
                    "record_id": record_id,
                    "threshold": threshold,
                    "min_area": area,
                    "TP": tp,
                    "FP": len(selected) - len(matched_preds),
                    "FN": n_gt - len(matched_gts),
                    "sum_iou": sum_iou
                })

    rows.extend(image_rows)
    completed.add(filename)

    # Save after each full photograph
    pd.DataFrame(rows).to_csv(RESULT, index=False)

    if len(completed) % 10 == 0:
        print(
            f"Completed {len(completed)}/130",
            flush=True
        )

details = pd.DataFrame(rows)

summary = details.groupby(
    ["threshold", "min_area"], as_index=False
)[["TP", "FP", "FN", "sum_iou"]].sum()

denominator = (
    summary.TP + 0.5 * (summary.FP + summary.FN)
)

summary["PQ"] = np.where(
    denominator > 0,
    summary.sum_iou / denominator,
    0
)

summary = summary.sort_values(
    "PQ", ascending=False
).reset_index(drop=True)

summary.to_csv(
    WORK / "v4a_validation_summary.csv",
    index=False
)

print("\nTOP 12 V4-A CONFIGURATIONS")
print(summary.head(12).to_string(index=False))

baseline = summary[
    (summary.threshold == 0.55) &
    (summary.min_area == 800)
].iloc[0]

print("\nREPRODUCED V3 BASELINE PQ:", round(baseline.PQ, 6))
print("EXPECTED BASELINE: 0.273674")
print("BEST V4-A PQ:", round(summary.iloc[0].PQ, 6))

assert abs(baseline.PQ - 0.273674) < 0.00001, (
    "Baseline mismatch — investigate before selecting V4."
)

print("\nSUCCESS — V4-A validation complete.")


In [ ]:

# V4-B — Morphology pilot, 12 photographs
# No training, no CNN inference

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

PILOT_CACHE = Path("/kaggle/working/v3_pilot_probability_maps")
THRESHOLD = 0.55
MIN_AREA = 800

methods = ["baseline", "close3", "close5", "open3_close3"]

def apply_morphology(binary, method):
    if method == "baseline":
        return binary

    k3 = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (3, 3)
    )
    k5 = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (5, 5)
    )

    if method == "close3":
        return cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k3)
    if method == "close5":
        return cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k5)
    if method == "open3_close3":
        opened = cv2.morphologyEx(
            binary, cv2.MORPH_OPEN, k3
        )
        return cv2.morphologyEx(
            opened, cv2.MORPH_CLOSE, k3
        )

    raise ValueError(method)


def evaluate_instances(binary, gt_records, min_area):
    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )

    areas = stats[:, cv2.CC_STAT_AREA]
    selected = set(
        np.flatnonzero(areas >= min_area).tolist()
    )
    selected.discard(0)

    totals = dict(TP=0, FP=0, FN=0, sum_iou=0.0)

    for gt_masks in gt_records:
        matched_preds = set()
        matched_gts = set()
        tp = 0
        iou_sum = 0.0

        for gt_id, gt in enumerate(gt_masks):
            intersections = np.bincount(
                labels[gt].ravel(), minlength=n
            )

            pred_ids = np.flatnonzero(intersections)
            pred_ids = pred_ids[pred_ids != 0]

            for pred_id in pred_ids:
                if int(pred_id) not in selected:
                    continue

                intersection = intersections[pred_id]
                union = (
                    areas[pred_id]
                    + int(gt.sum())
                    - intersection
                )
                iou = intersection / union

                if iou > 0.5:
                    tp += 1
                    iou_sum += iou
                    matched_preds.add(int(pred_id))
                    matched_gts.add(gt_id)

        totals["TP"] += tp
        totals["FP"] += len(selected) - len(matched_preds)
        totals["FN"] += len(gt_masks) - len(matched_gts)
        totals["sum_iou"] += iou_sum

    return totals


pilot_names = sorted(
    f for f in val_files_v4 if f not in set(remaining)
)
assert len(pilot_names) == 12

rows = []

for idx, filename in enumerate(pilot_names, 1):
    probability = np.load(
        PILOT_CACHE / (Path(filename).stem + ".npy"),
        mmap_mode="r"
    )

    binary = (probability > THRESHOLD).astype(np.uint8)
    gt_records = [
        get_ground_truth(rec)
        for rec in image_records[filename]
    ]

    for method in methods:
        processed = apply_morphology(binary, method)
        result = evaluate_instances(
            processed, gt_records, MIN_AREA
        )
        rows.append({
            "filename": filename,
            "method": method,
            **result
        })

    print(f"Completed {idx}/12", flush=True)

details = pd.DataFrame(rows)
summary = details.groupby("method", as_index=False)[
    ["TP", "FP", "FN", "sum_iou"]
].sum()

denom = summary.TP + 0.5 * (summary.FP + summary.FN)
summary["PQ"] = summary.sum_iou / denom
summary = summary.sort_values("PQ", ascending=False)

details.to_csv(
    "/kaggle/working/v4b_pilot_details.csv",
    index=False
)
summary.to_csv(
    "/kaggle/working/v4b_pilot_summary.csv",
    index=False
)

print("\nV4-B MORPHOLOGY RESULTS")
print(summary.to_string(index=False))

baseline = summary.loc[
    summary.method == "baseline", "PQ"
].iloc[0]

print("\nBaseline pilot PQ:", round(baseline, 6))
print("Expected:", 0.252145)

assert abs(baseline - 0.252145) < 0.00001, (
    "Baseline mismatch — stop and investigate."
)


In [ ]:

# V4-B — Independent morphology confirmation
# 130 photos; no training or GPU inference

import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")
CACHE = WORK / "v3_validation_130_cache"
RESULT = WORK / "v4b_validation_130_details.csv"

assert len(remaining) == 130
assert callable(apply_morphology)
assert callable(evaluate_instances)
assert callable(get_ground_truth)

METHODS = ["baseline", "open3_close3"]

# Resume support
if RESULT.exists():
    previous = pd.read_csv(RESULT)
    completed = set(previous.filename.unique())
    rows = previous.to_dict("records")
else:
    completed = set()
    rows = []

for filename in remaining:
    if filename in completed:
        continue

    prob = np.load(
        CACHE / (Path(filename).stem + ".npy"),
        mmap_mode="r"
    )

    binary = (prob > 0.55).astype(np.uint8)

    gt_records = [
        get_ground_truth(rec)
        for rec in image_records[filename]
    ]

    current_rows = []

    for method in METHODS:
        processed = apply_morphology(binary, method)
        result = evaluate_instances(
            processed, gt_records, min_area=800
        )

        current_rows.append({
            "filename": filename,
            "method": method,
            **result
        })

    rows.extend(current_rows)
    completed.add(filename)

    # Save after each completed photograph
    pd.DataFrame(rows).to_csv(RESULT, index=False)

    if len(completed) % 10 == 0:
        print(f"Completed {len(completed)}/130", flush=True)

details = pd.read_csv(RESULT)

assert set(details.filename) == set(remaining)
assert details.groupby("filename").size().eq(2).all()

summary = details.groupby(
    "method", as_index=False
)[["TP", "FP", "FN", "sum_iou"]].sum()

denom = summary.TP + 0.5 * (summary.FP + summary.FN)
summary["PQ"] = summary.sum_iou / denom

summary = summary.sort_values(
    "PQ", ascending=False
).reset_index(drop=True)

summary.to_csv(
    WORK / "v4b_validation_130_summary.csv",
    index=False
)

print("\nV4-B INDEPENDENT VALIDATION RESULTS")
print(summary.to_string(index=False))

baseline = summary.loc[
    summary.method == "baseline", "PQ"
].iloc[0]

assert abs(baseline - 0.273674) < 0.00001, (
    "V3 baseline mismatch — investigate before proceeding."
)

morph_pq = summary.loc[
    summary.method == "open3_close3", "PQ"
].iloc[0]

print("\nV3 baseline PQ:", round(baseline, 6))
print("V4-B morphology PQ:", round(morph_pq, 6))
print("Absolute PQ change:", round(morph_pq - baseline, 6))
print("Relative change (%):",
      round(100 * (morph_pq / baseline - 1), 2))

print("\nSUCCESS — V4-B evaluation complete.")


In [ ]:

# V4 — Back up completed experiments
# No training or model inference.

from pathlib import Path
import shutil
import json
from kaggle.api.kaggle_api_extended import KaggleApi

WORK = Path("/kaggle/working")
BACKUP = WORK / "v4_experiment_backup"
BACKUP.mkdir(exist_ok=True)

files = [
    "v4a_validation_details.csv",
    "v4a_validation_summary.csv",
    "v4b_pilot_details.csv",
    "v4b_pilot_summary.csv",
    "v4b_validation_130_details.csv",
    "v4b_validation_130_summary.csv"
]

for filename in files:
    source = WORK / filename
    assert source.exists(), f"Missing: {filename}"
    shutil.copy2(source, BACKUP / filename)
    print("Preserved:", filename)

config = {
    "model": "V3 UNet",
    "checkpoint": "v3_unet_best.pth",
    "public_V3_PQ": 0.21,
    "local_V3_PQ": 0.273674,
    "V4A_best": {
        "threshold": 0.50,
        "min_area": 800,
        "PQ": 0.276797
    },
    "V4B_best": {
        "threshold": 0.55,
        "min_area": 800,
        "morphology": "open3_close3",
        "PQ": 0.279330
    },
    "validation_images": 130,
    "evaluation": "Reference-style PQ"
}

(BACKUP / "experiment_config.json").write_text(
    json.dumps(config, indent=2)
)

metadata = {
    "title": "Solar Filament V4 Experiments",
    "id": "asaadwahag/solar-filament-v4-experiments",
    "licenses": [{"name": "CC0-1.0"}]
}

(BACKUP / "dataset-metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

api = KaggleApi()
api.authenticate()
api.dataset_create_new(
    str(BACKUP),
    public=False,
    quiet=False
)

print("\nSUCCESS — V4 experiment results backed up.")


In [ ]:

# V4-C1 — Measure predicted component geometry
# Uses V3 pilot maps and V4-B morphology.
# No training or GPU inference.

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

CACHE = Path("/kaggle/working/v3_pilot_probability_maps")
OUTPUT = Path("/kaggle/working/v4c_component_geometry.csv")

pilot_names = sorted(
    set(val_files_v4) - set(remaining)
)
assert len(pilot_names) == 12

rows = []

for index, filename in enumerate(pilot_names, 1):
    prob = np.load(
        CACHE / (Path(filename).stem + ".npy"),
        mmap_mode="r"
    )

    binary = (prob > 0.55).astype(np.uint8)
    processed = apply_morphology(binary, "open3_close3")

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        processed, connectivity=8
    )

    for component_id in range(1, n):
        area = int(stats[component_id, cv2.CC_STAT_AREA])

        if area < 800:
            continue

        x = int(stats[component_id, cv2.CC_STAT_LEFT])
        y = int(stats[component_id, cv2.CC_STAT_TOP])
        w = int(stats[component_id, cv2.CC_STAT_WIDTH])
        h = int(stats[component_id, cv2.CC_STAT_HEIGHT])

        crop = (labels[y:y+h, x:x+w] == component_id).astype(
            np.uint8
        )

        contours, _ = cv2.findContours(
            crop, cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_NONE
        )

        if not contours:
            continue

        contour = max(contours, key=cv2.contourArea)
        perimeter = cv2.arcLength(contour, True)

        hull = cv2.convexHull(contour)
        hull_area = cv2.contourArea(hull)

        solidity = (
            area / hull_area if hull_area > 0 else 1.0
        )
        circularity = (
            4 * np.pi * area / perimeter**2
            if perimeter > 0 else 0.0
        )

        rows.append({
            "filename": filename,
            "component_id": component_id,
            "area": area,
            "width": w,
            "height": h,
            "solidity": solidity,
            "circularity": circularity
        })

    print(f"Processed {index}/12", flush=True)

geometry = pd.DataFrame(rows)
geometry.to_csv(OUTPUT, index=False)

print("\nV4-C GEOMETRY SUMMARY")
print(geometry[
    ["area", "solidity", "circularity"]
].describe().round(3).to_string())

print("\nComponents measured:", len(geometry))
print("Saved:", OUTPUT)


In [ ]:

# V4-C2 — Label components using expert annotations
# A component is "matched" if IoU > 0.5 with at least
# one filament in at least one annotation record.

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")
CACHE = WORK / "v3_pilot_probability_maps"

geometry = pd.read_csv(
    WORK / "v4c_component_geometry.csv"
)

assert len(geometry) == 77

labeled_rows = []

for index, filename in enumerate(
    sorted(geometry.filename.unique()), 1
):
    probability = np.load(
        CACHE / (Path(filename).stem + ".npy"),
        mmap_mode="r"
    )

    binary = (probability > 0.55).astype(np.uint8)
    processed = apply_morphology(
        binary, "open3_close3"
    )

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        processed, connectivity=8
    )

    areas = stats[:, cv2.CC_STAT_AREA]

    matches_by_component = {}
    best_iou = {}

    for rec in image_records[filename]:
        gt_masks = get_ground_truth(rec)

        for gt in gt_masks:
            intersections = np.bincount(
                labels[gt].ravel(),
                minlength=n
            )

            candidate_ids = np.flatnonzero(intersections)
            candidate_ids = candidate_ids[candidate_ids != 0]

            for pred_id in candidate_ids:
                union = (
                    areas[pred_id]
                    + int(gt.sum())
                    - intersections[pred_id]
                )

                iou = float(
                    intersections[pred_id] / union
                )

                pred_id = int(pred_id)

                best_iou[pred_id] = max(
                    best_iou.get(pred_id, 0.0),
                    iou
                )

                if iou > 0.5:
                    matches_by_component[pred_id] = True

    image_geometry = geometry[
        geometry.filename == filename
    ]

    for _, row in image_geometry.iterrows():
        cid = int(row.component_id)

        labeled_rows.append({
            **row.to_dict(),
            "matched": bool(
                matches_by_component.get(cid, False)
            ),
            "best_iou": best_iou.get(cid, 0.0)
        })

    print(f"Labeled {index}/12", flush=True)

labeled = pd.DataFrame(labeled_rows)

assert len(labeled) == len(geometry)

labeled.to_csv(
    WORK / "v4c_labeled_geometry.csv",
    index=False
)

print("\nCOMPONENT CLASS COUNTS")
print(labeled.matched.value_counts().to_string())

print("\nGEOMETRY BY MATCH STATUS")
print(
    labeled.groupby("matched")[
        ["area", "solidity", "circularity"]
    ].agg(["median", "mean"]).round(3).to_string()
)

print("\nQUARTILES BY MATCH STATUS")
for feature in ["solidity", "circularity"]:
    print(f"\n{feature}:")
    print(
        labeled.groupby("matched")[feature]
        .quantile([0.25, 0.50, 0.75])
        .unstack()
        .round(3)
        .to_string()
    )

print("\nSaved: v4c_labeled_geometry.csv")


In [ ]:

# V4-C3: Geometry filtering, 12-image pilot
# No training or CNN inference.

import numpy as np
import pandas as pd
import cv2
from pathlib import Path

WORK = Path("/kaggle/working")
geometry = pd.read_csv(WORK / "v4c_labeled_geometry.csv")

# Candidate rules. None means no shape restriction.
rules = [
    ("baseline", None, None),
    ("circ_035", 0.35, None),
    ("circ_045", 0.45, None),
    ("circ_055", 0.55, None),
    ("solid_065", None, 0.65),
    ("solid_080", None, 0.80),
    ("solid_090", None, 0.90),
    ("circ045_solid090", 0.45, 0.90),
]

rows = []

for filename in sorted(geometry.filename.unique()):
    prob = np.load(
        WORK / "v3_pilot_probability_maps" /
        (Path(filename).stem + ".npy"),
        mmap_mode="r"
    )

    binary = (prob > 0.55).astype(np.uint8)
    processed = apply_morphology(binary, "open3_close3")

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        processed, connectivity=8
    )

    info = geometry[geometry.filename == filename]
    gt_records = [
        get_ground_truth(rec)
        for rec in image_records[filename]
    ]

    for rule_name, max_circ, max_solid in rules:
        allowed = info[info.area >= 800]

        if max_circ is not None:
            allowed = allowed[
                allowed.circularity <= max_circ
            ]

        if max_solid is not None:
            allowed = allowed[
                allowed.solidity <= max_solid
            ]

        selected = set(
            allowed.component_id.astype(int).tolist()
        )

        for rec_id, gt_masks in enumerate(gt_records):
            tp = 0
            sum_iou = 0.0
            hit_preds = set()
            hit_gts = set()

            for gt_id, gt in enumerate(gt_masks):
                intersection = np.bincount(
                    labels[gt].ravel(), minlength=n
                )

                for cid in selected:
                    overlap = intersection[cid]
                    if overlap == 0:
                        continue

                    union = (
                        int(stats[cid, cv2.CC_STAT_AREA])
                        + int(gt.sum())
                        - overlap
                    )
                    iou = overlap / union

                    if iou > 0.5:
                        tp += 1
                        sum_iou += iou
                        hit_preds.add(cid)
                        hit_gts.add(gt_id)

            rows.append({
                "filename": filename,
                "record_id": rec_id,
                "rule": rule_name,
                "TP": tp,
                "FP": len(selected) - len(hit_preds),
                "FN": len(gt_masks) - len(hit_gts),
                "sum_iou": sum_iou
            })

details = pd.DataFrame(rows)

summary = details.groupby(
    "rule", as_index=False
)[["TP", "FP", "FN", "sum_iou"]].sum()

denom = summary.TP + 0.5 * (summary.FP + summary.FN)
summary["PQ"] = summary.sum_iou / denom

summary = summary.sort_values(
    "PQ", ascending=False
).reset_index(drop=True)

details.to_csv(
    WORK / "v4c_pilot_filter_details.csv",
    index=False
)
summary.to_csv(
    WORK / "v4c_pilot_filter_summary.csv",
    index=False
)

print("V4-C3 GEOMETRIC FILTER RESULTS")
print(summary.to_string(index=False))

baseline = summary.loc[
    summary.rule == "baseline", "PQ"
].iloc[0]

print("\nReproduced V4-B pilot baseline:", round(baseline, 6))
print("Expected:", 0.258482)

assert abs(baseline - 0.258482) < 0.00001, (
    "Baseline mismatch. Investigate before proceeding."
)

print("\nSUCCESS — Geometry filtering pilot completed.")


In [ ]:

import pandas as pd
from pathlib import Path

work = Path("/kaggle/working")

old = pd.read_csv(work / "v4b_pilot_details.csv")
new = pd.read_csv(work / "v4c_pilot_filter_details.csv")

old = old[old.method == "open3_close3"]
new = new[new.rule == "baseline"]

compare = old.groupby("filename")[
    ["TP", "FP", "FN", "sum_iou"]
].sum().subtract(
    new.groupby("filename")[
        ["TP", "FP", "FN", "sum_iou"]
    ].sum(),
    fill_value=0
)

print("DIFFERENCES: V4-B minus V4-C")
print(compare[compare.abs().max(axis=1) > 1e-6].to_string())


In [ ]:

# Inspect annotation counts and false negatives
# No model inference

import pandas as pd
from pathlib import Path

problem_files = [
    "20210331231510Mh.jpeg",
    "20211016145050Bh.jpeg"
]

old = pd.read_csv(
    "/kaggle/working/v4b_pilot_details.csv"
)
new = pd.read_csv(
    "/kaggle/working/v4c_pilot_filter_details.csv"
)

for filename in problem_files:
    print("\nPHOTO:", filename)

    records_for_photo = image_records[filename]
    print("Annotation records:", len(records_for_photo))

    for i, rec in enumerate(records_for_photo):
        masks = get_ground_truth(rec)

        print(
            f"  Record {i}: "
            f"{len(annotations[rec['id']])} raw annotations, "
            f"{len(masks)} rasterized masks"
        )

    print("\nV4-B:")
    print(
        old[
            (old.filename == filename) &
            (old.method == "open3_close3")
        ][["TP", "FP", "FN"]].to_string(index=False)
    )

    print("\nV4-C:")
    print(
        new[
            (new.filename == filename) &
            (new.rule == "baseline")
        ][["TP", "FP", "FN"]].to_string(index=False)
    )


In [ ]:

import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")

summary = pd.read_csv(
    WORK / "v4c_pilot_filter_summary.csv"
)

# Both skipped photographs have no eligible predictions.
# They collectively contain four GT filament instances.
corrected = summary.copy()
corrected["FN"] = corrected["FN"] + 4

denom = (
    corrected["TP"]
    + 0.5 * (corrected["FP"] + corrected["FN"])
)

corrected["PQ"] = corrected["sum_iou"] / denom

corrected = corrected.sort_values(
    "PQ", ascending=False
).reset_index(drop=True)

corrected.to_csv(
    WORK / "v4c_pilot_filter_summary_corrected.csv",
    index=False
)

print("CORRECTED V4-C RESULTS")
print(corrected.to_string(index=False))

baseline = corrected.loc[
    corrected.rule == "baseline", "PQ"
].iloc[0]

assert abs(baseline - 0.258482) < 0.00001

print("\nSUCCESS — V4-C baseline agrees with V4-B.")


In [ ]:

# V4-D1 — Component-size diagnostics
# No training, no neural-network inference.

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")
CACHE = WORK / "v3_validation_130_cache"

rows = []

for index, filename in enumerate(remaining, 1):
    prob = np.load(
        CACHE / (Path(filename).stem + ".npy"),
        mmap_mode="r"
    )

    binary = (prob > 0.55).astype(np.uint8)
    processed = apply_morphology(
        binary, "open3_close3"
    )

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        processed, connectivity=8
    )

    for component_id in range(1, n):
        area = int(stats[component_id, cv2.CC_STAT_AREA])

        if area < 800:
            continue

        rows.append({
            "filename": filename,
            "component_id": component_id,
            "area": area,
            "width": int(
                stats[component_id, cv2.CC_STAT_WIDTH]
            ),
            "height": int(
                stats[component_id, cv2.CC_STAT_HEIGHT]
            )
        })

    if index % 20 == 0:
        print(f"Processed {index}/130", flush=True)

components = pd.DataFrame(rows)

components.to_csv(
    WORK / "v4d_component_sizes.csv",
    index=False
)

print("\nV4-D COMPONENT SIZE DISTRIBUTION")
print(
    components["area"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    ).round(1).to_string()
)

print("\nNUMBER OF LARGE COMPONENTS")
for threshold in [2000, 4000, 8000, 16000]:
    print(
        f"Area >= {threshold}:",
        int((components.area >= threshold).sum())
    )

print("\nTotal components:", len(components))
print("Saved: v4d_component_sizes.csv")


In [ ]:

# V4-D2 — Identify potentially merged filament components
# Uses existing V3 probability maps and V4-B morphology

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")
CACHE = WORK / "v3_validation_130_cache"

rows = []

for index, filename in enumerate(remaining, 1):
    prob = np.load(
        CACHE / (Path(filename).stem + ".npy"),
        mmap_mode="r"
    )

    binary = (prob > 0.55).astype(np.uint8)
    processed = apply_morphology(binary, "open3_close3")

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        processed, connectivity=8
    )

    eligible = np.flatnonzero(
        stats[:, cv2.CC_STAT_AREA] >= 800
    )
    eligible = eligible[eligible != 0]

    # Number of meaningful GT overlaps, assessed separately
    # for each annotator record to avoid double counting.
    max_overlaps = {int(cid): 0 for cid in eligible}

    for rec in image_records[filename]:
        gt_masks = get_ground_truth(rec)
        overlap_counts = {int(cid): 0 for cid in eligible}

        for gt in gt_masks:
            gt_area = int(gt.sum())
            if gt_area == 0:
                continue

            intersections = np.bincount(
                labels[gt].ravel(), minlength=n
            )

            candidates = np.flatnonzero(
                (intersections >= 50) &
                (intersections >= 0.10 * gt_area)
            )

            for cid in candidates:
                cid = int(cid)
                if cid in overlap_counts:
                    overlap_counts[cid] += 1

        for cid, count in overlap_counts.items():
            max_overlaps[cid] = max(max_overlaps[cid], count)

    for cid in eligible:
        cid = int(cid)

        rows.append({
            "filename": filename,
            "component_id": cid,
            "area": int(stats[cid, cv2.CC_STAT_AREA]),
            "gt_overlap_count": max_overlaps[cid],
            "potential_merge": max_overlaps[cid] >= 2
        })

    if index % 20 == 0:
        print(f"Processed {index}/130", flush=True)

merged = pd.DataFrame(rows)

assert len(merged) == 1125, (
    f"Expected 1125 components, got {len(merged)}"
)

merged.to_csv(
    WORK / "v4d_merge_diagnostics.csv",
    index=False
)

print("\nV4-D POTENTIAL MERGES")
print("Total components:", len(merged))
print("Potential merges:", int(merged.potential_merge.sum()))

print("\nBY COMPONENT AREA")
for min_area in [800, 2000, 4000, 8000]:
    subset = merged[merged.area >= min_area]
    count = int(subset.potential_merge.sum())
    print(
        f"Area >= {min_area}: "
        f"{count}/{len(subset)} potential merges"
    )

print("\nOVERLAP-COUNT DISTRIBUTION")
print(
    merged.gt_overlap_count.value_counts()
    .sort_index().to_string()
)

print("\nLARGEST POTENTIAL MERGES")
print(
    merged[merged.potential_merge]
    .sort_values("area", ascending=False)
    .head(10)
    .to_string(index=False)
)

print("\nSaved: v4d_merge_diagnostics.csv")


In [ ]:

# V4-E1 — Confidence of matched vs unmatched components
# No training or CNN inference.

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")
CACHE = WORK / "v3_pilot_probability_maps"

labeled = pd.read_csv(
    WORK / "v4c_labeled_geometry.csv"
)

rows = []

for filename in sorted(labeled.filename.unique()):
    prob = np.load(
        CACHE / (Path(filename).stem + ".npy"),
        mmap_mode="r"
    )

    binary = (prob > 0.55).astype(np.uint8)
    processed = apply_morphology(binary, "open3_close3")

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        processed, connectivity=8
    )

    for _, rec in labeled[
        labeled.filename == filename
    ].iterrows():

        cid = int(rec.component_id)

        # Restrict computation to component bounding box
        x = int(stats[cid, cv2.CC_STAT_LEFT])
        y = int(stats[cid, cv2.CC_STAT_TOP])
        w = int(stats[cid, cv2.CC_STAT_WIDTH])
        h = int(stats[cid, cv2.CC_STAT_HEIGHT])

        component = labels[y:y+h, x:x+w] == cid
        probabilities = prob[y:y+h, x:x+w][component]

        rows.append({
            "filename": filename,
            "component_id": cid,
            "matched": bool(rec.matched),
            "area": int(rec.area),
            "mean_confidence": float(probabilities.mean()),
            "median_confidence": float(np.median(probabilities)),
            "p90_confidence": float(
                np.percentile(probabilities, 90)
            )
        })

confidence = pd.DataFrame(rows)

confidence.to_csv(
    WORK / "v4e_confidence_analysis.csv",
    index=False
)

print("V4-E CONFIDENCE BY MATCH STATUS")
print(
    confidence.groupby("matched")[
        ["mean_confidence",
         "median_confidence",
         "p90_confidence"]
    ].agg(["median", "mean"])
    .round(4).to_string()
)

print("\nCOMPONENT COUNTS")
print(confidence.matched.value_counts().to_string())

print("\nSaved: v4e_confidence_analysis.csv")


In [ ]:

# V4-E2 — Component confidence filtering on 12 pilot images
# No retraining; no CNN inference.

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")
CACHE = WORK / "v3_pilot_probability_maps"

pilot_names = sorted(set(val_files_v4) - set(remaining))
assert len(pilot_names) == 12

RULES = [
    ("baseline", None),
    ("mean_080", 0.80),
    ("mean_085", 0.85),
    ("mean_090", 0.90)
]

rows = []

for index, filename in enumerate(pilot_names, 1):

    prob = np.load(
        CACHE / (Path(filename).stem + ".npy"),
        mmap_mode="r"
    )

    binary = (prob > 0.55).astype(np.uint8)
    processed = apply_morphology(binary, "open3_close3")

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        processed, connectivity=8
    )

    gt_records = [
        get_ground_truth(rec)
        for rec in image_records[filename]
    ]

    eligible = np.flatnonzero(
        stats[:, cv2.CC_STAT_AREA] >= 800
    )
    eligible = eligible[eligible != 0]

    # Component confidence, including pixels added by closing
    confidence_by_id = {}
    for cid in eligible:
        cid = int(cid)
        x = int(stats[cid, cv2.CC_STAT_LEFT])
        y = int(stats[cid, cv2.CC_STAT_TOP])
        w = int(stats[cid, cv2.CC_STAT_WIDTH])
        h = int(stats[cid, cv2.CC_STAT_HEIGHT])

        component = labels[y:y+h, x:x+w] == cid
        confidence_by_id[cid] = float(
            prob[y:y+h, x:x+w][component].mean()
        )

    # Evaluate every annotation record, including images
    # with zero eligible predictions.
    for name, cutoff in RULES:
        selected = set(
            int(cid) for cid in eligible
            if cutoff is None or confidence_by_id[int(cid)] >= cutoff
        )

        for record_id, gt_masks in enumerate(gt_records):
            matched_preds = set()
            matched_gts = set()
            tp = 0
            iou_sum = 0.0

            for gt_id, gt in enumerate(gt_masks):
                intersections = np.bincount(
                    labels[gt].ravel(), minlength=n
                )

                for cid in selected:
                    overlap = intersections[cid]
                    if overlap == 0:
                        continue

                    union = (
                        int(stats[cid, cv2.CC_STAT_AREA])
                        + int(gt.sum()) - overlap
                    )
                    iou = overlap / union

                    if iou > 0.5:
                        tp += 1
                        iou_sum += iou
                        matched_preds.add(cid)
                        matched_gts.add(gt_id)

            rows.append({
                "filename": filename,
                "record_id": record_id,
                "rule": name,
                "TP": tp,
                "FP": len(selected) - len(matched_preds),
                "FN": len(gt_masks) - len(matched_gts),
                "sum_iou": iou_sum
            })

    print(f"Completed {index}/12", flush=True)

details = pd.DataFrame(rows)
summary = details.groupby("rule", as_index=False)[
    ["TP", "FP", "FN", "sum_iou"]
].sum()

denom = summary.TP + 0.5 * (summary.FP + summary.FN)
summary["PQ"] = summary.sum_iou / denom

summary = summary.sort_values(
    "PQ", ascending=False
).reset_index(drop=True)

details.to_csv(
    WORK / "v4e_pilot_filter_details.csv",
    index=False
)
summary.to_csv(
    WORK / "v4e_pilot_filter_summary.csv",
    index=False
)

print("\nV4-E2 CONFIDENCE FILTER RESULTS")
print(summary.to_string(index=False))

baseline = summary.loc[
    summary.rule == "baseline", "PQ"
].iloc[0]

print("\nBaseline:", round(baseline, 6))
print("Expected: 0.258482")

assert abs(baseline - 0.258482) < 0.00001, (
    "Baseline mismatch — check scoring before proceeding."
)


In [ ]:

# V4-E3 — Independent 130-image confidence validation
# No training, no GPU inference

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")
CACHE = WORK / "v3_validation_130_cache"
RESULT = WORK / "v4e_validation_130_details.csv"

assert len(remaining) == 130

RULES = ["baseline", "mean_080"]
rows = []

for index, filename in enumerate(remaining, 1):

    prob = np.load(
        CACHE / (Path(filename).stem + ".npy"),
        mmap_mode="r"
    )

    binary = (prob > 0.55).astype(np.uint8)
    processed = apply_morphology(binary, "open3_close3")

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        processed, connectivity=8
    )

    areas = stats[:, cv2.CC_STAT_AREA]

    # Mean model probability for every connected component
    probability_sums = np.bincount(
        labels.ravel(),
        weights=np.asarray(prob).ravel(),
        minlength=n
    )
    mean_confidence = probability_sums / np.maximum(areas, 1)

    eligible = set(
        np.flatnonzero(areas >= 800).tolist()
    )
    eligible.discard(0)

    selected_by_rule = {
        "baseline": eligible,
        "mean_080": {
            cid for cid in eligible
            if mean_confidence[cid] >= 0.80
        }
    }

    gt_records = [
        get_ground_truth(rec)
        for rec in image_records[filename]
    ]

    # Compute IoU matches once per annotation record
    for record_id, gt_masks in enumerate(gt_records):
        hits = []

        for gt_id, gt in enumerate(gt_masks):
            intersections = np.bincount(
                labels[gt].ravel(), minlength=n
            )

            candidates = np.flatnonzero(intersections)
            candidates = candidates[candidates != 0]

            for cid in candidates:
                if int(cid) not in eligible:
                    continue

                union = (
                    int(areas[cid])
                    + int(gt.sum())
                    - intersections[cid]
                )

                iou = float(intersections[cid] / union)

                if iou > 0.5:
                    hits.append((int(cid), gt_id, iou))

        for rule in RULES:
            selected = selected_by_rule[rule]

            valid_hits = [
                (cid, gt_id, iou)
                for cid, gt_id, iou in hits
                if cid in selected
            ]

            rows.append({
                "filename": filename,
                "record_id": record_id,
                "rule": rule,
                "TP": len(valid_hits),
                "FP": len(selected) - len({
                    hit[0] for hit in valid_hits
                }),
                "FN": len(gt_masks) - len({
                    hit[1] for hit in valid_hits
                }),
                "sum_iou": sum(
                    hit[2] for hit in valid_hits
                )
            })

    if index % 10 == 0:
        # Persist partial results
        pd.DataFrame(rows).to_csv(RESULT, index=False)
        print(f"Completed {index}/130", flush=True)

details = pd.DataFrame(rows)
details.to_csv(RESULT, index=False)

assert details.filename.nunique() == 130

summary = details.groupby(
    "rule", as_index=False
)[["TP", "FP", "FN", "sum_iou"]].sum()

denom = summary.TP + 0.5 * (summary.FP + summary.FN)
summary["PQ"] = summary.sum_iou / denom

summary = summary.sort_values(
    "PQ", ascending=False
).reset_index(drop=True)

summary.to_csv(
    WORK / "v4e_validation_130_summary.csv",
    index=False
)

print("\nV4-E3 INDEPENDENT VALIDATION")
print(summary.to_string(index=False))

baseline = summary.loc[
    summary.rule == "baseline", "PQ"
].iloc[0]

assert abs(baseline - 0.279330) < 0.00001, (
    "Baseline mismatch — investigate before proceeding."
)

candidate = summary.loc[
    summary.rule == "mean_080", "PQ"
].iloc[0]

print("\nV4-B baseline:", round(baseline, 6))
print("V4-E confidence filter:", round(candidate, 6))
print("Absolute PQ improvement:", round(candidate - baseline, 6))
print("Relative improvement:",
      round(100 * (candidate / baseline - 1), 2), "%")

print("\nSUCCESS — V4-E3 complete.")


In [ ]:

# V4-F1 — Explore solar-specific preprocessing
# No training, no GPU inference, no file modification.

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("/kaggle/input")
annotation_path = next(ROOT.rglob(
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
))
IMAGE_DIR = annotation_path.parent / "train_images"

assert len(remaining) == 130

sample_files = [
    remaining[0],
    remaining[len(remaining)//2],
    remaining[-1]
]

def preprocess_solar(image):
    image = image.astype(np.float32)

    h, w = image.shape
    yy, xx = np.indices((h, w))

    cx = (w - 1) / 2
    cy = (h - 1) / 2

    radius = np.sqrt((xx-cx)**2 + (yy-cy)**2)
    solar_radius = 900.0

    disk = radius <= solar_radius

    # Estimate broad radial illumination variation.
    radial_bins = np.minimum(
        radius.astype(np.int32), int(solar_radius)
    )
    radial_medians = np.zeros(int(solar_radius)+1)

    for r in range(int(solar_radius)+1):
        values = image[
            (radial_bins == r) & disk
        ]
        radial_medians[r] = (
            np.median(values) if len(values) else 1.0
        )

    radial_medians = cv2.GaussianBlur(
        radial_medians.astype(np.float32).reshape(-1,1),
        (1,51), 0
    ).ravel()

    target = np.median(
        radial_medians[100:700]
    )

    correction = target / np.maximum(
        radial_medians[radial_bins], 1.0
    )

    corrected = image * correction
    corrected = np.clip(corrected, 0, 255).astype(np.uint8)

    # Locally enhance contrast.
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(16,16)
    )
    enhanced = clahe.apply(corrected)

    enhanced[~disk] = 0

    return enhanced

fig, axes = plt.subplots(
    3, 2, figsize=(11, 15)
)

for i, filename in enumerate(sample_files):
    original = cv2.imread(
        str(IMAGE_DIR / filename),
        cv2.IMREAD_GRAYSCALE
    )

    processed = preprocess_solar(original)

    axes[i,0].imshow(original, cmap="gray", vmin=0, vmax=255)
    axes[i,0].set_title(f"Original: {filename}")

    axes[i,1].imshow(processed, cmap="gray", vmin=0, vmax=255)
    axes[i,1].set_title("Radial correction + CLAHE")

    for j in range(2):
        axes[i,j].axis("off")

plt.tight_layout()
plt.show()

print("V4-F1 preprocessing visual comparison complete.")


In [ ]:

# V4-F2 — Compare preprocessing strengths
# No training, no GPU inference.

import cv2
import numpy as np
import matplotlib.pyplot as plt

def radial_correct_only(image):
    image = image.astype(np.float32)
    h, w = image.shape

    yy, xx = np.indices((h, w))
    cx, cy = (w-1)/2, (h-1)/2
    r = np.sqrt((xx-cx)**2 + (yy-cy)**2)

    disk_radius = 900
    disk = r <= disk_radius
    bins = np.minimum(r.astype(np.int32), disk_radius)

    radial_profile = np.zeros(disk_radius+1, dtype=np.float32)

    for k in range(disk_radius+1):
        values = image[(bins == k) & disk]
        radial_profile[k] = np.median(values) if len(values) else 1

    # Smooth radial illumination profile
    radial_profile = cv2.GaussianBlur(
        radial_profile.reshape(-1,1),
        (1,51), 0
    ).ravel()

    target = np.median(radial_profile[100:700])
    corrected = image * (
        target / np.maximum(radial_profile[bins], 1)
    )

    corrected = np.clip(corrected, 0, 255).astype(np.uint8)
    corrected[~disk] = 0

    return corrected

# Select the middle example, which contains several filaments.
filename = sample_files[1]
original = cv2.imread(
    str(IMAGE_DIR / filename),
    cv2.IMREAD_GRAYSCALE
)

radial = radial_correct_only(original)

variants = [("Original", original),
            ("Radial only", radial)]

for strength in [0.5, 1.0, 2.0]:
    clahe = cv2.createCLAHE(
        clipLimit=strength,
        tileGridSize=(16,16)
    )
    enhanced = clahe.apply(radial)
    enhanced[radial == 0] = 0

    variants.append((f"CLAHE {strength}", enhanced))

fig, axes = plt.subplots(1, len(variants), figsize=(20,5))

for ax, (title, image) in zip(axes, variants):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

print("V4-F2 preprocessing comparison complete.")


In [ ]:

# V4-F3 — Compare annotated filament crops
# No training or GPU inference.

import cv2
import numpy as np
import matplotlib.pyplot as plt

# Use the same 2014 image from V4-F2
filename = sample_files[1]

original = cv2.imread(
    str(IMAGE_DIR / filename),
    cv2.IMREAD_GRAYSCALE
)
assert original is not None

radial = radial_correct_only(original)

clahe = cv2.createCLAHE(
    clipLimit=0.5,
    tileGridSize=(16, 16)
)

enhanced = clahe.apply(radial)
enhanced[radial == 0] = 0

# Select three ground-truth filament bounding boxes.
boxes = []

for rec in image_records[filename]:
    for ann in annotations[rec["id"]]:
        x, y, w, h = ann["bbox"]
        if w >= 15 and h >= 10:
            boxes.append((int(x), int(y), int(w), int(h)))

# Prefer larger filaments and avoid duplicate boxes
boxes = sorted(set(boxes), key=lambda b: b[2] * b[3], reverse=True)
boxes = boxes[:3]

assert boxes, "No suitable annotated filament boxes found."

fig, axes = plt.subplots(
    len(boxes), 3,
    figsize=(12, 4 * len(boxes)),
    squeeze=False
)

variants = [
    ("Original", original),
    ("Radial correction", radial),
    ("Radial + CLAHE 0.5", enhanced)
]

for row, (x, y, w, h) in enumerate(boxes):
    padding = max(40, int(0.5 * max(w, h)))

    x1 = max(0, x-padding)
    y1 = max(0, y-padding)
    x2 = min(original.shape[1], x+w+padding)
    y2 = min(original.shape[0], y+h+padding)

    for col, (name, image) in enumerate(variants):
        crop = image[y1:y2, x1:x2]

        axes[row, col].imshow(
            crop, cmap="gray", vmin=0, vmax=255
        )
        axes[row, col].set_title(
            f"{name} — Filament {row+1}"
        )
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

print("V4-F3 filament-crop comparison complete.")


In [ ]:

import cv2
import numpy as np

for filename in sample_files:
    image = cv2.imread(
        str(IMAGE_DIR / filename),
        cv2.IMREAD_GRAYSCALE
    )

    blur = cv2.GaussianBlur(image, (15, 15), 0)
    _, binary = cv2.threshold(
        blur, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    contours, _ = cv2.findContours(
        binary, cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    largest = max(contours, key=cv2.contourArea)
    (cx, cy), radius = cv2.minEnclosingCircle(largest)

    print(
        filename,
        "| center:", (round(cx, 1), round(cy, 1)),
        "| radius:", round(radius, 1)
    )


In [ ]:

# V4-F5 — Check how many annotated filament pixels
# fall outside fixed and estimated solar disks.
# No training or GPU inference.

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

IMAGE_DIR = annotation_path.parent / "train_images"

rows = []

for index, filename in enumerate(remaining, 1):
    image = cv2.imread(
        str(IMAGE_DIR / filename),
        cv2.IMREAD_GRAYSCALE
    )
    assert image is not None

    h, w = image.shape

    blur = cv2.GaussianBlur(image, (15, 15), 0)
    _, binary = cv2.threshold(
        blur, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    contours, _ = cv2.findContours(
        binary, cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    largest = max(contours, key=cv2.contourArea)
    (cx, cy), radius = cv2.minEnclosingCircle(largest)

    yy, xx = np.ogrid[:h, :w]

    fixed_disk = (
        (xx - (w-1)/2)**2 +
        (yy - (h-1)/2)**2 <= 900**2
    )

    estimated_disk = (
        (xx-cx)**2 + (yy-cy)**2 <= radius**2
    )

    # Union of all expert masks for this physical photograph.
    union_gt = np.zeros((h, w), dtype=bool)

    for rec in image_records[filename]:
        for gt in get_ground_truth(rec):
            union_gt |= gt

    total_gt = int(union_gt.sum())

    outside_fixed = int(
        (union_gt & ~fixed_disk).sum()
    )
    outside_estimated = int(
        (union_gt & ~estimated_disk).sum()
    )

    rows.append({
        "filename": filename,
        "cx": cx,
        "cy": cy,
        "radius": radius,
        "GT_pixels": total_gt,
        "outside_fixed": outside_fixed,
        "outside_estimated": outside_estimated,
        "fixed_loss_pct": (
            100 * outside_fixed / total_gt
            if total_gt else 0
        ),
        "estimated_loss_pct": (
            100 * outside_estimated / total_gt
            if total_gt else 0
        )
    })

    if index % 20 == 0:
        print(f"Processed {index}/130", flush=True)

results = pd.DataFrame(rows)

results.to_csv(
    "/kaggle/working/v4f_disk_annotation_check.csv",
    index=False
)

print("\nV4-F5 ANNOTATION PRESERVATION")
print("Photographs:", len(results))
print("Total unique GT pixels (summed by image):",
      results.GT_pixels.sum())

for name, column in [
    ("Fixed radius 900", "outside_fixed"),
    ("Estimated disk", "outside_estimated")
]:
    lost = int(results[column].sum())
    total = int(results.GT_pixels.sum())

    print(
        name,
        "| excluded GT pixels:", lost,
        "| overall percentage:",
        round(100 * lost / total, 4)
    )

print("\nIMAGES WITH ANY GT LOSS")
print("Fixed disk:",
      int((results.outside_fixed > 0).sum()))
print("Estimated disk:",
      int((results.outside_estimated > 0).sum()))

print("\nWORST 5 ESTIMATED-DISK CASES")
print(
    results.sort_values(
        "estimated_loss_pct", ascending=False
    )[[
        "filename", "radius", "GT_pixels",
        "outside_estimated", "estimated_loss_pct"
    ]].head(5).to_string(index=False)
)

print("\nSUCCESS — V4-F5 complete.")


In [ ]:

# V4-F6 — Reproducible solar preprocessing
# No training, no GPU inference.

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

def preprocess_solar_v4(image, margin=12):
    assert image.ndim == 2
    assert image.dtype == np.uint8

    h, w = image.shape

    blur = cv2.GaussianBlur(image, (15, 15), 0)
    _, binary = cv2.threshold(
        blur, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    contours, _ = cv2.findContours(
        binary, cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    assert contours, "Solar disk not detected"

    largest = max(contours, key=cv2.contourArea)
    (cx, cy), radius = cv2.minEnclosingCircle(largest)

    # Geometry sanity check
    assert 800 < radius < 1100, (
        f"Unusual disk radius: {radius}"
    )

    yy, xx = np.indices((h, w), dtype=np.float32)
    r = np.sqrt((xx-cx)**2 + (yy-cy)**2)

    # Avoid unstable correction near the outer limb
    radial_limit = max(1, int(radius))
    bins = np.minimum(
        r.astype(np.int32), radial_limit
    )

    disk = r <= radius + margin

    profile = np.zeros(radial_limit + 1, dtype=np.float32)

    for k in range(radial_limit + 1):
        pixels = image[
            (bins == k) & (r <= radius)
        ]
        profile[k] = (
            np.median(pixels) if len(pixels) else 1
        )

    profile = cv2.GaussianBlur(
        profile.reshape(-1, 1),
        (1, 51), 0
    ).ravel()

    target = np.median(
        profile[int(radius*0.15):int(radius*0.75)]
    )

    gain = target / np.maximum(
        profile[bins], 1.0
    )
    gain = np.clip(gain, 0.7, 1.5)

    corrected = np.clip(
        image.astype(np.float32) * gain,
        0, 255
    ).astype(np.uint8)

    clahe = cv2.createCLAHE(
        clipLimit=0.5,
        tileGridSize=(16, 16)
    )
    enhanced = clahe.apply(corrected)

    enhanced[~disk] = 0

    return enhanced, (cx, cy, radius)


# Check intensity statistics on three validation images
results = []

for filename in sample_files:
    original = cv2.imread(
        str(IMAGE_DIR / filename),
        cv2.IMREAD_GRAYSCALE
    )

    processed, (cx, cy, radius) = preprocess_solar_v4(
        original
    )

    yy, xx = np.ogrid[:original.shape[0], :original.shape[1]]
    disk = (xx-cx)**2 + (yy-cy)**2 <= radius**2

    results.append({
        "filename": filename,
        "radius": round(radius, 1),
        "original_mean": float(original[disk].mean()),
        "processed_mean": float(processed[disk].mean()),
        "original_std": float(original[disk].std()),
        "processed_std": float(processed[disk].std())
    })

stats = pd.DataFrame(results)

print("\nV4-F6 IMAGE STATISTICS")
print(stats.round(2).to_string(index=False))

print("\nSUCCESS — V4-F6 preprocessing function ready.")


In [ ]:

# V4-F7 — Verify preprocessing on training photographs
# No training or neural-network inference.
# Does not modify any source images.

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")

train_files_v4 = sorted(
    set(image_records) - set(val_files_v4)
)

assert len(train_files_v4) == 565

rows = []

for index, filename in enumerate(train_files_v4, 1):
    original = cv2.imread(
        str(IMAGE_DIR / filename),
        cv2.IMREAD_GRAYSCALE
    )
    assert original is not None

    # Check actual V4 preprocessing output.
    processed, (cx, cy, radius) = preprocess_solar_v4(
        original, margin=12
    )

    assert processed.shape == original.shape
    assert processed.dtype == np.uint8

    h, w = original.shape
    yy, xx = np.ogrid[:h, :w]

    retained_disk = (
        (xx - cx)**2 + (yy - cy)**2
        <= (radius + 12)**2
    )

    # Ground-truth union across expert annotation records.
    union_gt = np.zeros((h, w), dtype=bool)

    for rec in image_records[filename]:
        for gt in get_ground_truth(rec):
            union_gt |= gt

    gt_pixels = int(union_gt.sum())
    excluded = int((union_gt & ~retained_disk).sum())

    rows.append({
        "filename": filename,
        "cx": cx,
        "cy": cy,
        "radius": radius,
        "gt_pixels": gt_pixels,
        "excluded_gt_pixels": excluded,
        "excluded_pct": (
            100 * excluded / gt_pixels
            if gt_pixels else 0.0
        )
    })

    if index % 25 == 0:
        print(f"Checked {index}/565", flush=True)

results = pd.DataFrame(rows)

OUTPUT = WORK / "v4f_training_disk_check.csv"
results.to_csv(OUTPUT, index=False)

total_gt = int(results.gt_pixels.sum())
total_excluded = int(results.excluded_gt_pixels.sum())

print("\nV4-F7 TRAINING ANNOTATION CHECK")
print("Training photographs:", len(results))
print("Total annotated pixels:", total_gt)
print("Excluded pixels:", total_excluded)
print(
    "Excluded percentage:",
    round(100 * total_excluded / max(total_gt, 1), 6)
)
print(
    "Images with excluded annotations:",
    int((results.excluded_gt_pixels > 0).sum())
)

print("\nDETECTED RADIUS STATISTICS")
print(
    results.radius.describe().round(2).to_string()
)

print("\nWORST FIVE CASES")
print(
    results.sort_values(
        "excluded_gt_pixels", ascending=False
    )[[
        "filename", "radius",
        "gt_pixels", "excluded_gt_pixels"
    ]].head(5).to_string(index=False)
)

print("\nSaved:", OUTPUT)
print("SUCCESS — V4-F7 completed.")


In [ ]:

# V4-F8 — Cache preprocessed training/validation images
# No training. Safe to resume after interruption.

import cv2
import time
import json
from pathlib import Path

WORK = Path("/kaggle/working")
CACHE = WORK / "v4f_preprocessed_images"
CACHE.mkdir(parents=True, exist_ok=True)

all_files = sorted(image_records.keys())
assert len(all_files) == 707

assert callable(preprocess_solar_v4)

start = time.perf_counter()
created = 0

for index, filename in enumerate(all_files, 1):
    output = CACHE / (Path(filename).stem + ".png")

    if output.exists() and output.stat().st_size > 0:
        continue

    original = cv2.imread(
        str(IMAGE_DIR / filename),
        cv2.IMREAD_GRAYSCALE
    )
    assert original is not None, filename

    processed, geometry = preprocess_solar_v4(
        original, margin=12
    )

    assert processed.shape == original.shape
    assert processed.dtype == original.dtype

    # Temporary file and atomic rename help avoid partial files.
    temporary = CACHE / (Path(filename).stem + ".tmp.png")
    success = cv2.imwrite(str(temporary), processed)
    assert success, filename

    temporary.replace(output)
    created += 1

    if index % 25 == 0:
        print(
            f"Processed {index}/707 | "
            f"New files: {created} | "
            f"Elapsed: {(time.perf_counter()-start)/60:.1f} min",
            flush=True
        )

cached = list(CACHE.glob("*.png"))
assert len(cached) == 707, (
    f"Expected 707 cached images, found {len(cached)}"
)

total_gb = sum(p.stat().st_size for p in cached) / 1e9

metadata = {
    "method": "radial_correction_plus_CLAHE",
    "clahe_clip_limit": 0.5,
    "clahe_tile_grid": [16, 16],
    "disk_margin_pixels": 12,
    "train_photographs": 565,
    "validation_photographs": 142,
    "image_format": "lossless PNG",
    "original_split_random_state": 42
}

(WORK / "v4f_preprocessing_config.json").write_text(
    json.dumps(metadata, indent=2)
)

print("\nSUCCESS — V4-F8 CACHE COMPLETE")
print("Cached photographs:", len(cached))
print("Cache size:", round(total_gb, 2), "GB")
print("Processing time:",
      round((time.perf_counter()-start)/60, 2),
      "minutes")
print("Cache location:", CACHE)


In [ ]:

# V4-F9 — Create and back up preprocessing cache
# No model training or inference.

import json
import shutil
import zipfile
import time
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi

WORK = Path("/kaggle/working")
CACHE = WORK / "v4f_preprocessed_images"
BACKUP = WORK / "v4f_cache_backup"

BACKUP.mkdir(exist_ok=True)

images = sorted(CACHE.glob("*.png"))
assert len(images) == 707, f"Found {len(images)} images"

archive = BACKUP / "v4f_preprocessed_images.zip"

start = time.perf_counter()

print("Creating ZIP archive...", flush=True)

# PNG is already compressed; storing avoids wasting CPU.
with zipfile.ZipFile(
    archive, "w", compression=zipfile.ZIP_STORED,
    allowZip64=True
) as z:
    for i, image in enumerate(images, 1):
        z.write(image, arcname=image.name)

        if i % 100 == 0:
            print(f"Packed {i}/707", flush=True)

with zipfile.ZipFile(archive, "r") as z:
    assert len(z.namelist()) == 707
    assert z.testzip() is None

config = WORK / "v4f_preprocessing_config.json"
assert config.exists()

shutil.copy2(
    config,
    BACKUP / "v4f_preprocessing_config.json"
)

metadata = {
    "title": "Solar Filament V4 Preprocessed Images",
    "id": "asaadwahag/solar-filament-v4-preprocessed",
    "licenses": [{"name": "CC0-1.0"}]
}

(BACKUP / "dataset-metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

print("ZIP size:",
      round(archive.stat().st_size / 1e9, 3),
      "GB", flush=True)

api = KaggleApi()
api.authenticate()

api.dataset_create_new(
    str(BACKUP),
    public=False,
    quiet=False
)

print("\nSUCCESS — V4 preprocessing cache backed up.")
print("Dataset: asaadwahag/solar-filament-v4-preprocessed")
print("Elapsed:",
      round((time.perf_counter()-start)/60, 2),
      "minutes")


In [ ]:

# V4-F10A — Check readiness for fine-tuning
# No training.

from pathlib import Path
import torch

WORK = Path("/kaggle/working")

checks = {
    "UNetV3 architecture": "UNetV3" in globals(),
    "DynamicSolarDataset": "DynamicSolarDataset" in globals(),
    "train_dataset": "train_dataset" in globals(),
    "val_dataset": "val_dataset" in globals(),
    "train_loader": "train_loader" in globals(),
    "val_loader": "val_loader" in globals(),
    "V3 best checkpoint": (
        WORK / "v3_training/v3_unet_best.pth"
    ).exists(),
    "V4 enhanced images": len(list(
        (WORK / "v4f_preprocessed_images").glob("*.png")
    )) == 707,
    "GPU available": torch.cuda.is_available()
}

print("V4-F10 TRAINING READINESS")
for name, available in checks.items():
    print(f"{name}: {available}")

if "DynamicSolarDataset" in globals():
    import inspect
    print("\nDataset constructor:")
    print(inspect.signature(DynamicSolarDataset))

if "train_dataset" in globals():
    print("\nExisting training samples:", len(train_dataset))

if "val_dataset" in globals():
    print("Existing validation samples:", len(val_dataset))

print("\nReadiness check complete.")


In [ ]:

# V4-F10B — Restore model and dynamic training dataset
# No training in this cell.

import json
import random
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image, ImageDraw
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import defaultdict

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK = Path("/kaggle/working")
CACHE = WORK / "v4f_preprocessed_images"

# -------------------------------------------------------
# 1. Exact architecture matching V3 checkpoint
# -------------------------------------------------------

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNetV3(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = DoubleConv(1, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.bottleneck = DoubleConv(128, 256)
        self.pool = nn.MaxPool2d(2)
        self.dec3 = DoubleConv(384, 128)
        self.dec2 = DoubleConv(192, 64)
        self.dec1 = DoubleConv(96, 32)
        self.output = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))

        d3 = F.interpolate(
            b, size=e3.shape[2:],
            mode="bilinear", align_corners=False
        )
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = F.interpolate(
            d3, size=e2.shape[2:],
            mode="bilinear", align_corners=False
        )
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = F.interpolate(
            d2, size=e1.shape[2:],
            mode="bilinear", align_corners=False
        )
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.output(d1)


# -------------------------------------------------------
# 2. Reconstruct physical-image train/validation split
# -------------------------------------------------------

annotation_path = next(Path("/kaggle/input").rglob(
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
))

with open(annotation_path) as f:
    coco = json.load(f)

image_records = coco["images"]
annotations_by_image = defaultdict(list)

for ann in coco["annotations"]:
    annotations_by_image[ann["image_id"]].append(ann)

filenames = sorted(set(
    rec["file_name"] for rec in image_records
))

train_files, val_files = train_test_split(
    filenames,
    test_size=0.20,
    random_state=42
)

train_set = set(train_files)
val_set = set(val_files)

train_records = [
    rec for rec in image_records
    if rec["file_name"] in train_set
]
val_records = [
    rec for rec in image_records
    if rec["file_name"] in val_set
]

assert len(train_files) == 565
assert len(val_files) == 142
assert len(train_records) == 922
assert len(val_records) == 232
assert train_set.isdisjoint(val_set)


# -------------------------------------------------------
# 3. Dynamic filament-centred and random crop dataset
# -------------------------------------------------------

class DynamicSolarDataset(Dataset):
    def __init__(self, records, cache_dir, training=True):
        self.records = records
        self.cache_dir = Path(cache_dir)
        self.training = training
        self.patch = 256

    def __len__(self):
        return len(self.records) * 2

    def __getitem__(self, index):
        rec = self.records[index // 2]
        filename = rec["file_name"]

        rng = random if self.training else random.Random(
            SEED + index
        )

        image_path = self.cache_dir / (
            Path(filename).stem + ".png"
        )

        image = np.asarray(
            Image.open(image_path).convert("L")
        )

        h, w = image.shape
        anns = annotations_by_image[rec["id"]]

        # Alternate between filament-centred and random crops.
        if index % 2 == 0 and anns:
            ann = rng.choice(anns)
            x, y, bw, bh = ann["bbox"]

            cx = x + bw / 2 + rng.randint(-64, 64)
            cy = y + bh / 2 + rng.randint(-64, 64)

            x0 = int(np.clip(
                cx - self.patch / 2, 0, w - self.patch
            ))
            y0 = int(np.clip(
                cy - self.patch / 2, 0, h - self.patch
            ))
        else:
            x0 = rng.randint(0, w - self.patch)
            y0 = rng.randint(0, h - self.patch)

        crop = image[
            y0:y0+self.patch,
            x0:x0+self.patch
        ].copy()

        mask_img = Image.new(
            "L", (self.patch, self.patch), 0
        )
        draw = ImageDraw.Draw(mask_img)

        for ann in anns:
            for polygon in ann["segmentation"]:
                pts = np.asarray(
                    polygon, dtype=np.float32
                ).reshape(-1, 2)

                if len(pts) >= 3:
                    pts[:, 0] -= x0
                    pts[:, 1] -= y0

                    draw.polygon(
                        [tuple(p) for p in pts],
                        fill=1
                    )

        mask = np.asarray(
            mask_img, dtype=np.uint8
        ).copy()

        if self.training:
            if rng.random() < 0.5:
                crop = np.fliplr(crop)
                mask = np.fliplr(mask)

            if rng.random() < 0.5:
                crop = np.flipud(crop)
                mask = np.flipud(mask)

            k = rng.randint(0, 3)
            crop = np.rot90(crop, k)
            mask = np.rot90(mask, k)

        image_tensor = torch.from_numpy(
            np.ascontiguousarray(crop, dtype=np.float32)
        ).unsqueeze(0) / 255.0

        mask_tensor = torch.from_numpy(
            np.ascontiguousarray(mask, dtype=np.float32)
        ).unsqueeze(0)

        return image_tensor, mask_tensor


train_dataset = DynamicSolarDataset(
    train_records, CACHE, training=True
)
val_dataset = DynamicSolarDataset(
    val_records, CACHE, training=False
)

# num_workers=0 initially, for easier debugging and
# to avoid the earlier multiprocessing warnings.
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# -------------------------------------------------------
# 4. Restore V3 pretrained weights into a NEW model
# -------------------------------------------------------

checkpoint = WORK / "v3_training/v3_unet_best.pth"
assert checkpoint.exists()

model_v4 = UNetV3().to(DEVICE)

model_v4.load_state_dict(
    torch.load(
        checkpoint,
        map_location=DEVICE,
        weights_only=True
    ),
    strict=True
)
model_v4.eval()

# -------------------------------------------------------
# 5. Dataset sanity check
# -------------------------------------------------------

images, masks = next(iter(train_loader))

print("V4-F10B READY")
print("Device:", DEVICE)
print("Training records:", len(train_records))
print("Validation records:", len(val_records))
print("Training samples per epoch:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Batch images:", tuple(images.shape))
print("Batch masks:", tuple(masks.shape))
print("Mask unique values:", torch.unique(masks).tolist())
print("Foreground percentage:",
      round(float(masks.mean()) * 100, 3))
print("Model parameters:",
      sum(p.numel() for p in model_v4.parameters()))
print("V3 weights loaded: True")


In [ ]:

# V4-F10C — Five-epoch fine-tuning
# Starts from V3 weights; does NOT overwrite V3.
# Saves checkpoint after every epoch.

import os
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

OUT = Path("/kaggle/working/v4f_training")
OUT.mkdir(parents=True, exist_ok=True)

BEST_PATH = OUT / "v4f_unet_best.pth"
LATEST_PATH = OUT / "v4f_latest_state.pth"
HISTORY_PATH = OUT / "v4f_training_history.csv"

EPOCHS = 5
LR = 3e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_v4 = model_v4.to(DEVICE)

optimizer = torch.optim.AdamW(
    model_v4.parameters(),
    lr=LR,
    weight_decay=1e-4
)

bce = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([20.0], device=DEVICE)
)

def combined_loss(logits, target):
    loss_bce = bce(logits, target)

    probs = torch.sigmoid(logits)
    intersection = (probs * target).sum(dim=(1, 2, 3))
    denominator = (
        probs.sum(dim=(1, 2, 3))
        + target.sum(dim=(1, 2, 3))
    )

    dice_loss = 1 - (
        (2 * intersection + 1)
        / (denominator + 1)
    ).mean()

    return loss_bce + dice_loss


def validation_dice(model, loader, threshold=0.45):
    model.eval()
    dice_total = 0.0
    count = 0

    with torch.inference_mode():
        for images, masks in loader:
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)

            logits = model(images)
            predictions = (
                torch.sigmoid(logits) > threshold
            ).float()

            intersection = (
                predictions * masks
            ).sum(dim=(1, 2, 3))

            denominator = (
                predictions.sum(dim=(1, 2, 3))
                + masks.sum(dim=(1, 2, 3))
            )

            dice = (
                (2 * intersection + 1)
                / (denominator + 1)
            )

            dice_total += float(dice.sum().item())
            count += len(images)

    return dice_total / max(count, 1)


# Atomic save: write temp file, then rename.
def save_atomic(obj, destination):
    temp = destination.with_suffix(".tmp")
    torch.save(obj, temp)
    os.replace(temp, destination)


history = []
start_epoch = 1
best_dice = -1.0

# Resume automatically if previously interrupted.
if LATEST_PATH.exists():
    state = torch.load(
        LATEST_PATH,
        map_location=DEVICE,
        weights_only=False
    )

    model_v4.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])

    if "torch_rng" in state:
        torch.set_rng_state(state["torch_rng"].cpu())

    if "numpy_rng" in state:
        np.random.set_state(state["numpy_rng"])

    if "python_rng" in state:
        random.setstate(state["python_rng"])

    if "cuda_rng" in state and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda_rng"])

    start_epoch = state["epoch"] + 1
    best_dice = state["best_dice"]

    if HISTORY_PATH.exists():
        history = pd.read_csv(
            HISTORY_PATH
        ).to_dict("records")

    print(
        f"Resuming from epoch {start_epoch}",
        flush=True
    )

print("Starting V4-F fine-tuning...", flush=True)
print("Training epochs:", EPOCHS)
print("Learning rate:", LR)
print("Starting epoch:", start_epoch)

for epoch in range(start_epoch, EPOCHS + 1):
    start = time.perf_counter()
    model_v4.train()

    running_loss = 0.0
    examples = 0

    for images, masks in train_loader:
        images = images.to(
            DEVICE, non_blocking=True
        )
        masks = masks.to(
            DEVICE, non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        logits = model_v4(images)
        loss = combined_loss(logits, masks)

        loss.backward()
        optimizer.step()

        batch_size = len(images)
        running_loss += (
            float(loss.item()) * batch_size
        )
        examples += batch_size

    train_loss = running_loss / examples
    val_dice = validation_dice(
        model_v4, val_loader, threshold=0.45
    )

    improved = val_dice > best_dice

    if improved:
        best_dice = val_dice
        save_atomic(
            model_v4.state_dict(),
            BEST_PATH
        )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_dice": val_dice,
        "best_dice": best_dice,
        "lr": LR,
        "elapsed_sec": time.perf_counter() - start
    })

    pd.DataFrame(history).to_csv(
        HISTORY_PATH, index=False
    )

    state = {
        "epoch": epoch,
        "model": model_v4.state_dict(),
        "optimizer": optimizer.state_dict(),
        "best_dice": best_dice,
        "torch_rng": torch.get_rng_state(),
        "numpy_rng": np.random.get_state(),
        "python_rng": random.getstate(),
        "cuda_rng": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        )
    }

    save_atomic(state, LATEST_PATH)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss {train_loss:.4f} | "
        f"Val Dice {val_dice:.4f} | "
        f"Best {best_dice:.4f} | "
        f"{history[-1]['elapsed_sec']:.1f}s",
        flush=True
    )

print("\nSUCCESS — V4-F fine-tuning complete.")
print("Best enhanced-image Dice:", round(best_dice, 6))
print("Best checkpoint:", BEST_PATH)
print("Latest resumable state:", LATEST_PATH)
print("History:", HISTORY_PATH)


In [ ]:

# V4-F11 — Back up fine-tuned model
# No training or inference.

from pathlib import Path
import shutil
import json
from kaggle.api.kaggle_api_extended import KaggleApi

WORK = Path("/kaggle/working")
SOURCE = WORK / "v4f_training"
BACKUP = WORK / "v4f_checkpoint_backup"
BACKUP.mkdir(exist_ok=True)

files = [
    (SOURCE / "v4f_unet_best.pth", "v4f_unet_best.pth"),
    (SOURCE / "v4f_latest_state.pth", "v4f_latest_state.pth"),
    (SOURCE / "v4f_training_history.csv", "v4f_training_history.csv"),
    (
        WORK / "v4f_preprocessing_config.json",
        "v4f_preprocessing_config.json"
    )
]

for source, name in files:
    assert source.exists() and source.stat().st_size > 0, name
    shutil.copy2(source, BACKUP / name)
    print("Preserved:", name)

experiment = {
    "model": "UNetV3",
    "initialization": "v3_unet_best.pth",
    "input": "Radial correction + CLAHE 0.5",
    "training_epochs": 5,
    "learning_rate": 0.00003,
    "best_epoch": 1,
    "best_validation_dice": 0.672855,
    "note": "Full-image instance PQ not yet evaluated"
}

(BACKUP / "experiment_metadata.json").write_text(
    json.dumps(experiment, indent=2)
)

metadata = {
    "title": "Solar Filament V4F Checkpoints",
    "id": "asaadwahag/solar-filament-v4f-checkpoints",
    "licenses": [{"name": "CC0-1.0"}]
}

(BACKUP / "dataset-metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

api = KaggleApi()
api.authenticate()

api.dataset_create_new(
    str(BACKUP),
    public=False,
    quiet=False
)

print("\nSUCCESS — V4-F checkpoint backed up.")
print("Dataset: asaadwahag/solar-filament-v4f-checkpoints")


In [ ]:

# V4-F12A — Inference on 12 pilot validation images
# No training; resumable within the same session.

import time
import json
import numpy as np
import torch
from pathlib import Path
from PIL import Image

WORK = Path("/kaggle/working")
ENHANCED = WORK / "v4f_preprocessed_images"
CACHE = WORK / "v4f_pilot_probability_maps"
CACHE.mkdir(exist_ok=True)

CHECKPOINT = WORK / "v4f_training/v4f_unet_best.pth"

assert CHECKPOINT.exists()
assert len(list(ENHANCED.glob("*.png"))) == 707
assert "UNetV3" in globals()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_v4 = UNetV3().to(DEVICE)
model_v4.load_state_dict(
    torch.load(
        CHECKPOINT,
        map_location=DEVICE,
        weights_only=True
    ),
    strict=True
)
model_v4.eval()

# Recreate the same 12-image pilot split used for V3.
from sklearn.model_selection import train_test_split

all_files = sorted({
    rec["file_name"] for rec in image_records
}) if isinstance(image_records, list) else sorted(image_records)

_, val_files_check = train_test_split(
    all_files, test_size=0.20, random_state=42
)

pilot_files_v4 = sorted(
    np.random.default_rng(42).choice(
        sorted(val_files_check), 12, replace=False
    ).tolist()
)

def infer_full_v4(image_path):
    image = np.asarray(
        Image.open(image_path).convert("L"),
        dtype=np.float32
    ) / 255.0

    h, w = image.shape
    assert (h, w) == (2048, 2048)

    PATCH = 256
    STRIDE = 128
    BATCH = 8

    positions = [
        (y, x)
        for y in range(0, h-PATCH+1, STRIDE)
        for x in range(0, w-PATCH+1, STRIDE)
    ]

    total = np.zeros((h, w), dtype=np.float32)
    counts = np.zeros((h, w), dtype=np.float32)

    with torch.inference_mode():
        for start in range(0, len(positions), BATCH):
            batch_positions = positions[start:start+BATCH]

            patches = np.stack([
                image[y:y+PATCH, x:x+PATCH]
                for y, x in batch_positions
            ])

            tensor = torch.from_numpy(
                patches.copy()
            ).unsqueeze(1).to(DEVICE)

            probabilities = torch.sigmoid(
                model_v4(tensor)
            ).cpu().numpy()[:, 0]

            for prob, (y, x) in zip(
                probabilities, batch_positions
            ):
                total[y:y+PATCH, x:x+PATCH] += prob
                counts[y:y+PATCH, x:x+PATCH] += 1

    return total / counts


start = time.perf_counter()

for i, filename in enumerate(pilot_files_v4, 1):
    output = CACHE / (Path(filename).stem + ".npy")

    if not output.exists():
        input_path = ENHANCED / (
            Path(filename).stem + ".png"
        )
        assert input_path.exists()

        probability = infer_full_v4(input_path)
        np.save(output, probability)

    print(f"Processed {i}/12 — {filename}", flush=True)

print("\nSUCCESS — V4-F PILOT INFERENCE COMPLETE")
print("Cached maps:", len(list(CACHE.glob("*.npy"))))
print("Elapsed minutes:",
      round((time.perf_counter()-start)/60, 2))


In [ ]:

# V4-F12B FIX — Restore ground-truth mask function
# No training, no inference.

import numpy as np
from PIL import Image, ImageDraw

def get_ground_truth(rec):
    """
    Convert COCO polygon annotations into a list
    of full-resolution binary instance masks.
    """
    width = int(rec["width"])
    height = int(rec["height"])

    instance_masks = []

    for ann in annotations_by_image[rec["id"]]:
        mask = Image.new("L", (width, height), 0)
        draw = ImageDraw.Draw(mask)

        for polygon in ann["segmentation"]:
            points = np.asarray(
                polygon, dtype=np.float32
            ).reshape(-1, 2)

            if len(points) >= 3:
                draw.polygon(
                    [tuple(p) for p in points],
                    fill=1
                )

        instance_masks.append(
            np.asarray(mask, dtype=bool)
        )

    return instance_masks


# The earlier comparison expects filename -> records.
# Rebuild that mapping without modifying the original list.
from collections import defaultdict

records_by_filename = defaultdict(list)

for rec in image_records:
    records_by_filename[rec["file_name"]].append(rec)

# Quick validation
test_filename = pilot_files_v4[0]
test_records = records_by_filename[test_filename]

print("Ground-truth function restored.")
print("Test photograph:", test_filename)
print("Annotation records:", len(test_records))
print(
    "Instance masks:",
    sum(len(get_ground_truth(rec)) for rec in test_records)
)
print("SUCCESS — Ready to evaluate PQ.")


In [ ]:

# ============================================================
# V4-F12B — COMPLETE CORRECTED PILOT PQ COMPARISON
# V3 vs V4-F | 12 validation images
# No training. Uses existing cached probability maps.
# ============================================================

import cv2
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image, ImageDraw
from collections import defaultdict

WORK = Path("/kaggle/working")

CACHES = {
    "V3": WORK / "v3_pilot_probability_maps",
    "V4F": WORK / "v4f_pilot_probability_maps"
}

THRESHOLD = 0.55
MIN_AREA = 800
CONFIDENCE_CUTOFF = 0.80

# ------------------------------------------------------------
# 1. Restore filename -> COCO records mapping
# ------------------------------------------------------------

records_by_filename = defaultdict(list)

for rec in image_records:
    records_by_filename[rec["file_name"]].append(rec)

print("Physical images:", len(records_by_filename))

# ------------------------------------------------------------
# 2. Restore ground-truth instance masks
# ------------------------------------------------------------

def get_ground_truth(rec):
    width = int(rec["width"])
    height = int(rec["height"])

    masks = []

    for ann in annotations_by_image[rec["id"]]:
        mask_image = Image.new(
            "L", (width, height), 0
        )

        draw = ImageDraw.Draw(mask_image)

        for polygon in ann["segmentation"]:
            points = np.asarray(
                polygon, dtype=np.float32
            ).reshape(-1, 2)

            if len(points) >= 3:
                draw.polygon(
                    [tuple(p) for p in points],
                    fill=1
                )

        masks.append(
            np.asarray(mask_image, dtype=bool)
        )

    return masks

# ------------------------------------------------------------
# 3. Morphological post-processing
# ------------------------------------------------------------

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (3, 3)
)

def morphology_open_close(binary):
    opened = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        kernel
    )

    closed = cv2.morphologyEx(
        opened,
        cv2.MORPH_CLOSE,
        kernel
    )

    return closed

# ------------------------------------------------------------
# 4. Calculate reference-style instance PQ
# ------------------------------------------------------------

def score_components(labels, stats, selected, gt_records):
    n = len(stats)
    areas = stats[:, cv2.CC_STAT_AREA]

    totals = {
        "TP": 0,
        "FP": 0,
        "FN": 0,
        "sum_iou": 0.0
    }

    for gt_masks in gt_records:
        matched_preds = set()
        matched_gts = set()

        for gt_id, gt in enumerate(gt_masks):
            intersections = np.bincount(
                labels[gt].ravel(),
                minlength=n
            )

            ids = np.flatnonzero(intersections)
            ids = ids[ids != 0]

            gt_area = int(gt.sum())

            for cid in ids:
                cid = int(cid)

                if cid not in selected:
                    continue

                intersection = int(intersections[cid])

                union = (
                    int(areas[cid])
                    + gt_area
                    - intersection
                )

                if union <= 0:
                    continue

                iou = intersection / union

                if iou > 0.5:
                    totals["TP"] += 1
                    totals["sum_iou"] += float(iou)

                    matched_preds.add(cid)
                    matched_gts.add(gt_id)

        totals["FP"] += (
            len(selected) - len(matched_preds)
        )

        totals["FN"] += (
            len(gt_masks) - len(matched_gts)
        )

    return totals

# ------------------------------------------------------------
# 5. Verify pilot filenames and cached probability maps
# ------------------------------------------------------------

assert len(pilot_files_v4) == 12

for filename in pilot_files_v4:
    assert filename in records_by_filename, filename

    for model_name, cache_dir in CACHES.items():
        path = cache_dir / (
            Path(filename).stem + ".npy"
        )

        assert path.exists(), (
            f"Missing {model_name} probability map: {path}"
        )

print("All 12 pilot images and both model caches verified.")

# ------------------------------------------------------------
# 6. Run V3 / V4-F comparison
# ------------------------------------------------------------

rows = []

for i, filename in enumerate(pilot_files_v4, 1):

    gt_records = [
        get_ground_truth(rec)
        for rec in records_by_filename[filename]
    ]

    assert gt_records, filename

    for model_name, cache_dir in CACHES.items():

        path = cache_dir / (
            Path(filename).stem + ".npy"
        )

        prob = np.load(
            path,
            mmap_mode="r"
        )

        assert prob.shape == (2048, 2048)

        binary = (
            prob > THRESHOLD
        ).astype(np.uint8)

        for method in [
            "raw",
            "morphology"
        ]:

            if method == "raw":
                processed = binary
            else:
                processed = morphology_open_close(
                    binary
                )

            n, labels, stats, _ = (
                cv2.connectedComponentsWithStats(
                    processed,
                    connectivity=8
                )
            )

            areas = stats[:, cv2.CC_STAT_AREA]

            eligible = set(
                np.flatnonzero(
                    areas >= MIN_AREA
                ).tolist()
            )

            eligible.discard(0)

            selections = {
                "baseline": eligible
            }

            # Confidence filtering, only after morphology
            if method == "morphology":

                sums = np.bincount(
                    labels.ravel(),
                    weights=np.asarray(prob).ravel(),
                    minlength=n
                )

                confidence = (
                    sums / np.maximum(areas, 1)
                )

                selections["confidence_080"] = {
                    cid for cid in eligible
                    if confidence[cid] >= CONFIDENCE_CUTOFF
                }

            for filter_name, selected in selections.items():

                result = score_components(
                    labels,
                    stats,
                    selected,
                    gt_records
                )

                rows.append({
                    "filename": filename,
                    "model": model_name,
                    "method": method,
                    "filter": filter_name,
                    **result
                })

    print(
        f"Compared {i}/12 — {filename}",
        flush=True
    )

# ------------------------------------------------------------
# 7. Calculate aggregate PQ
# ------------------------------------------------------------

details = pd.DataFrame(rows)

summary = details.groupby(
    ["model", "method", "filter"],
    as_index=False
)[
    ["TP", "FP", "FN", "sum_iou"]
].sum()

denominator = (
    summary["TP"]
    + 0.5 * (
        summary["FP"] + summary["FN"]
    )
)

summary["PQ"] = np.where(
    denominator > 0,
    summary["sum_iou"] / denominator,
    0.0
)

summary = summary.sort_values(
    "PQ",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# 8. Save results
# ------------------------------------------------------------

details_path = (
    WORK / "v4f_pilot_comparison_details.csv"
)

summary_path = (
    WORK / "v4f_pilot_comparison_summary.csv"
)

details.to_csv(
    details_path,
    index=False
)

summary.to_csv(
    summary_path,
    index=False
)

# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("V3 VS V4-F — 12 IMAGE PILOT COMPARISON")
print("=" * 65)

print(
    summary[
        ["model", "method", "filter",
         "TP", "FP", "FN", "PQ"]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ------------------------------------------------------------
# 10. Reproduce historical V3 reference scores
# ------------------------------------------------------------

expected_v3 = {
    ("raw", "baseline"): 0.252145,
    ("morphology", "baseline"): 0.258482,
    ("morphology", "confidence_080"): 0.267127
}

print("\nV3 REFERENCE CHECK")

all_match = True

for (method, filt), expected in expected_v3.items():

    selected_row = summary[
        (summary["model"] == "V3")
        & (summary["method"] == method)
        & (summary["filter"] == filt)
    ]

    assert len(selected_row) == 1

    actual = float(
        selected_row.iloc[0]["PQ"]
    )

    match = abs(actual - expected) < 0.00001

    all_match = all_match and match

    print(
        f"{method}/{filt}: "
        f"actual={actual:.6f}, "
        f"expected={expected:.6f}, "
        f"match={match}"
    )

if all_match:
    print("\nSUCCESS — V3 reference scores reproduced.")
else:
    print(
        "\nWARNING — V3 reference mismatch. "
        "Do not interpret V4-F performance yet. "
        "Share this output for debugging."
    )

print("\nDetails saved:", details_path)
print("Summary saved:", summary_path)
print("\nV4-F12B comparison finished.")


In [ ]:

# ============================================================
# V4-F13A — Generate V4-F predictions for 130 validation images
# No training. Reuses the epoch-1 V4-F model.
# ============================================================

import time
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

WORK = Path("/kaggle/working")
ENHANCED = WORK / "v4f_preprocessed_images"
OUT = WORK / "v4f_validation_130_cache"
OUT.mkdir(parents=True, exist_ok=True)

# Reconstruct the original physical-image validation split.
all_files = sorted(records_by_filename.keys())

_, val_files_v4 = train_test_split(
    all_files,
    test_size=0.20,
    random_state=42
)

# Exclude the 12 pilot photographs.
remaining_v4f = sorted(
    set(val_files_v4) - set(pilot_files_v4)
)

assert len(val_files_v4) == 142
assert len(pilot_files_v4) == 12
assert len(remaining_v4f) == 130

# Check that inference components are still available.
assert callable(infer_full_v4)
assert model_v4 is not None
model_v4.eval()

print("V4-F13A — 130-IMAGE VALIDATION")
print("Validation images:", len(remaining_v4f))
print("Model:", "V4-F epoch 1")
print("Inference:", "256-pixel patches, stride 128")
print()

start = time.perf_counter()
created = 0

for i, filename in enumerate(remaining_v4f, 1):

    output = OUT / (Path(filename).stem + ".npy")

    if not output.exists():
        input_path = ENHANCED / (
            Path(filename).stem + ".png"
        )

        assert input_path.exists(), filename

        probability = infer_full_v4(input_path)

        assert probability.shape == (2048, 2048)
        assert np.isfinite(probability).all()

        # Float32 preserves precision for the PQ comparison.
        np.save(
            output,
            probability.astype(np.float32)
        )
        created += 1

    if i % 10 == 0 or i == len(remaining_v4f):
        print(
            f"Processed {i}/130 | "
            f"New maps: {created} | "
            f"Elapsed: {(time.perf_counter()-start)/60:.2f} min",
            flush=True
        )

cached = list(OUT.glob("*.npy"))

assert all(
    (OUT / (Path(f).stem + ".npy")).exists()
    for f in remaining_v4f
)

print("\nSUCCESS — V4-F13A COMPLETE")
print("Validation photographs:", len(remaining_v4f))
print("Cached probability maps:", len(cached))
print("New maps generated:", created)
print(
    "Elapsed minutes:",
    round((time.perf_counter()-start)/60, 2)
)
print("Cache:", OUT)


In [ ]:

# ============================================================
# V4-F13B — FULL 130-IMAGE PQ COMPARISON
# V3 vs V4-F
# Raw / Morphology / Morphology + Confidence
# No training or GPU inference.
# ============================================================

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageDraw
from collections import defaultdict

WORK = Path("/kaggle/working")

CACHES = {
    "V3": WORK / "v3_validation_130_cache",
    "V4F": WORK / "v4f_validation_130_cache",
}

THRESHOLD = 0.55
MIN_AREA = 800
CONFIDENCE_CUTOFF = 0.80

# ------------------------------------------------------------
# 1. Restore annotation mappings, if necessary
# ------------------------------------------------------------

records_by_filename = defaultdict(list)

for rec in image_records:
    records_by_filename[rec["file_name"]].append(rec)

# Rebuild the annotation dictionary from the loaded COCO JSON.
annotations_by_image = defaultdict(list)

for ann in coco["annotations"]:
    annotations_by_image[ann["image_id"]].append(ann)


def get_ground_truth(rec):
    width = int(rec["width"])
    height = int(rec["height"])
    masks = []

    for ann in annotations_by_image[rec["id"]]:
        canvas = Image.new("L", (width, height), 0)
        draw = ImageDraw.Draw(canvas)

        for polygon in ann["segmentation"]:
            points = np.asarray(
                polygon, dtype=np.float32
            ).reshape(-1, 2)

            if len(points) >= 3:
                draw.polygon(
                    [tuple(p) for p in points],
                    fill=1
                )

        masks.append(np.asarray(canvas, dtype=bool))

    return masks


# ------------------------------------------------------------
# 2. Identical post-processing for both models
# ------------------------------------------------------------

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (3, 3)
)


def morphology_open_close(binary):
    opened = cv2.morphologyEx(
        binary, cv2.MORPH_OPEN, kernel
    )
    return cv2.morphologyEx(
        opened, cv2.MORPH_CLOSE, kernel
    )


# ------------------------------------------------------------
# 3. Historical reference-style scorer
# ------------------------------------------------------------

def score_components(labels, stats, selected, gt_records):
    n = len(stats)
    areas = stats[:, cv2.CC_STAT_AREA]

    totals = {
        "TP": 0,
        "FP": 0,
        "FN": 0,
        "sum_iou": 0.0
    }

    for gt_masks in gt_records:
        matched_preds = set()
        matched_gts = set()

        for gt_id, gt in enumerate(gt_masks):
            intersections = np.bincount(
                labels[gt].ravel(),
                minlength=n
            )

            ids = np.flatnonzero(intersections)
            ids = ids[ids != 0]

            gt_area = int(gt.sum())

            for cid in ids:
                cid = int(cid)

                if cid not in selected:
                    continue

                intersection = int(intersections[cid])

                union = (
                    int(areas[cid])
                    + gt_area
                    - intersection
                )

                if union <= 0:
                    continue

                iou = intersection / union

                if iou > 0.5:
                    totals["TP"] += 1
                    totals["sum_iou"] += float(iou)
                    matched_preds.add(cid)
                    matched_gts.add(gt_id)

        totals["FP"] += len(selected) - len(matched_preds)
        totals["FN"] += len(gt_masks) - len(matched_gts)

    return totals


# ------------------------------------------------------------
# 4. Verify all probability maps before scoring
# ------------------------------------------------------------

assert len(remaining_v4f) == 130

for filename in remaining_v4f:
    assert filename in records_by_filename, filename

    for model_name, cache_dir in CACHES.items():
        path = cache_dir / (Path(filename).stem + ".npy")
        assert path.exists(), (
            f"Missing {model_name} probability map: {path}"
        )

print("All 130 V3 and V4-F probability maps verified.")
print("Starting PQ comparison...\n", flush=True)


# ------------------------------------------------------------
# 5. Evaluate all 130 photographs
# ------------------------------------------------------------

rows = []

for index, filename in enumerate(remaining_v4f, 1):

    gt_records = [
        get_ground_truth(rec)
        for rec in records_by_filename[filename]
    ]

    for model_name, cache_dir in CACHES.items():

        path = cache_dir / (Path(filename).stem + ".npy")
        prob = np.load(path, mmap_mode="r")

        assert prob.shape == (2048, 2048)
        assert np.isfinite(prob).all()

        binary = (prob > THRESHOLD).astype(np.uint8)

        for method in ("raw", "morphology"):

            processed = (
                binary
                if method == "raw"
                else morphology_open_close(binary)
            )

            n, labels, stats, _ = (
                cv2.connectedComponentsWithStats(
                    processed, connectivity=8
                )
            )

            areas = stats[:, cv2.CC_STAT_AREA]

            eligible = set(
                np.flatnonzero(areas >= MIN_AREA).tolist()
            )
            eligible.discard(0)

            selections = {"baseline": eligible}

            if method == "morphology":

                sums = np.bincount(
                    labels.ravel(),
                    weights=np.asarray(prob).ravel(),
                    minlength=n
                )

                mean_confidence = (
                    sums / np.maximum(areas, 1)
                )

                selections["confidence_080"] = {
                    cid for cid in eligible
                    if mean_confidence[cid] >= CONFIDENCE_CUTOFF
                }

            for filter_name, selected in selections.items():

                result = score_components(
                    labels, stats, selected, gt_records
                )

                rows.append({
                    "filename": filename,
                    "model": model_name,
                    "method": method,
                    "filter": filter_name,
                    **result
                })

    if index % 10 == 0:
        print(
            f"Compared {index}/130 photographs",
            flush=True
        )


# ------------------------------------------------------------
# 6. Aggregate reference-style PQ
# ------------------------------------------------------------

details = pd.DataFrame(rows)

summary = details.groupby(
    ["model", "method", "filter"],
    as_index=False
)[["TP", "FP", "FN", "sum_iou"]].sum()

denominator = (
    summary["TP"]
    + 0.5 * (summary["FP"] + summary["FN"])
)

summary["PQ"] = np.where(
    denominator > 0,
    summary["sum_iou"] / denominator,
    0.0
)

summary = summary.sort_values(
    "PQ", ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# 7. Save results
# ------------------------------------------------------------

DETAILS_PATH = WORK / "v4f_validation_130_details.csv"
SUMMARY_PATH = WORK / "v4f_validation_130_summary.csv"

details.to_csv(DETAILS_PATH, index=False)
summary.to_csv(SUMMARY_PATH, index=False)


# ------------------------------------------------------------
# 8. Display comparison
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("V3 VS V4-F — 130-IMAGE VALIDATION")
print("=" * 70)

print(
    summary[
        ["model", "method", "filter",
         "TP", "FP", "FN", "PQ"]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)


# ------------------------------------------------------------
# 9. Verify historical V3 scores
# ------------------------------------------------------------

expected_v3 = {
    ("raw", "baseline"): 0.273674,
    ("morphology", "baseline"): 0.279330,
    ("morphology", "confidence_080"): 0.282300
}

print("\nV3 REFERENCE REPRODUCTION CHECK")

all_match = True

for (method, filt), expected in expected_v3.items():

    matches = summary[
        (summary["model"] == "V3")
        & (summary["method"] == method)
        & (summary["filter"] == filt)
    ]

    assert len(matches) == 1

    actual = float(matches.iloc[0]["PQ"])
    match = abs(actual - expected) < 0.00001

    all_match = all_match and match

    print(
        f"{method}/{filt}: "
        f"actual={actual:.6f} | "
        f"expected={expected:.6f} | "
        f"match={match}"
    )


# ------------------------------------------------------------
# 10. Calculate direct V4-F improvements
# ------------------------------------------------------------

if all_match:

    print("\nDIRECT V4-F VS V3 COMPARISON")

    for method, filt in expected_v3:

        v3 = summary[
            (summary.model == "V3")
            & (summary.method == method)
            & (summary["filter"] == filt)
        ].iloc[0]

        v4 = summary[
            (summary.model == "V4F")
            & (summary.method == method)
            & (summary["filter"] == filt)
        ].iloc[0]

        difference = float(v4.PQ - v3.PQ)

        print(
            f"{method}/{filt}: "
            f"V3={v3.PQ:.6f}, "
            f"V4F={v4.PQ:.6f}, "
            f"delta={difference:+.6f}"
        )

    print("\nSUCCESS — V3 reference scores reproduced.")

else:
    print(
        "\nWARNING: V3 reference scores do not match. "
        "Do not interpret V4-F results until investigated."
    )

print("\nSaved:", DETAILS_PATH)
print("Saved:", SUMMARY_PATH)
print("\nV4-F13B COMPLETE")


In [ ]:

# V4-F14 — Probability ensemble pilot
# Uses previously verified scorer from V4-F13B.
# No training, no new inference.

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")
V3_CACHE = WORK / "v3_pilot_probability_maps"
V4_CACHE = WORK / "v4f_pilot_probability_maps"

WEIGHTS = [0.0, 0.10, 0.25, 0.50]
THRESHOLD = 0.55
MIN_AREA = 800
CONFIDENCE = 0.80

assert callable(score_components)
assert callable(morphology_open_close)
assert len(pilot_files_v4) == 12

rows = []

for i, filename in enumerate(pilot_files_v4, 1):
    stem = Path(filename).stem

    p3 = np.load(
        V3_CACHE / f"{stem}.npy"
    )
    p4 = np.load(
        V4_CACHE / f"{stem}.npy"
    )

    assert p3.shape == p4.shape == (2048, 2048)

    gt_records = [
        get_ground_truth(rec)
        for rec in records_by_filename[filename]
    ]

    for weight in WEIGHTS:
        prob = (
            (1.0 - weight) * p3
            + weight * p4
        )

        binary = (prob > THRESHOLD).astype(np.uint8)
        processed = morphology_open_close(binary)

        n, labels, stats, _ = (
            cv2.connectedComponentsWithStats(
                processed, connectivity=8
            )
        )

        areas = stats[:, cv2.CC_STAT_AREA]

        eligible = np.flatnonzero(
            areas >= MIN_AREA
        )
        eligible = eligible[eligible != 0]

        sums = np.bincount(
            labels.ravel(),
            weights=prob.ravel(),
            minlength=n
        )

        mean_conf = sums / np.maximum(areas, 1)

        selected = {
            int(cid) for cid in eligible
            if mean_conf[cid] >= CONFIDENCE
        }

        result = score_components(
            labels, stats, selected, gt_records
        )

        rows.append({
            "filename": filename,
            "v4_weight": weight,
            **result
        })

    print(f"Evaluated {i}/12", flush=True)

details = pd.DataFrame(rows)

summary = details.groupby(
    "v4_weight", as_index=False
)[["TP", "FP", "FN", "sum_iou"]].sum()

denom = (
    summary.TP
    + 0.5 * (summary.FP + summary.FN)
)

summary["PQ"] = summary.sum_iou / denom

summary = summary.sort_values(
    "PQ", ascending=False
)

details.to_csv(
    WORK / "v4f14_ensemble_pilot_details.csv",
    index=False
)

summary.to_csv(
    WORK / "v4f14_ensemble_pilot_summary.csv",
    index=False
)

print("\nV4-F14 ENSEMBLE PILOT RESULTS")
print(
    summary[
        ["v4_weight", "TP", "FP", "FN", "PQ"]
    ].to_string(index=False)
)

baseline = summary.loc[
    np.isclose(summary.v4_weight, 0.0), "PQ"
].iloc[0]

assert abs(baseline - 0.267127) < 0.00001, (
    f"V3 baseline mismatch: {baseline:.6f}"
)

print("\nSUCCESS — Historical V3 pilot reproduced.")
print("Best pilot PQ:", round(summary.PQ.max(), 6))


In [ ]:

# ============================================================
# V4-F15 — V3 + V4-F ensemble: 130-image validation
# Test V4 weights: 0%, 10%, 25%, 50%
# No training or GPU inference.
# ============================================================

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")
V3_CACHE = WORK / "v3_validation_130_cache"
V4_CACHE = WORK / "v4f_validation_130_cache"

WEIGHTS = [0.0, 0.10, 0.25, 0.50]
THRESHOLD = 0.55
MIN_AREA = 800
CONFIDENCE = 0.80

assert len(remaining_v4f) == 130
assert callable(score_components)
assert callable(morphology_open_close)

# Check that every prediction is available.
for filename in remaining_v4f:
    stem = Path(filename).stem
    assert (V3_CACHE / f"{stem}.npy").exists()
    assert (V4_CACHE / f"{stem}.npy").exists()

print("V4-F15 — ENSEMBLE VALIDATION")
print("Photographs:", len(remaining_v4f))
print("Weights:", WEIGHTS)
print()

rows = []

for i, filename in enumerate(remaining_v4f, 1):
    stem = Path(filename).stem

    p3 = np.load(
        V3_CACHE / f"{stem}.npy",
        mmap_mode="r"
    )
    p4 = np.load(
        V4_CACHE / f"{stem}.npy",
        mmap_mode="r"
    )

    assert p3.shape == p4.shape == (2048, 2048)

    gt_records = [
        get_ground_truth(rec)
        for rec in records_by_filename[filename]
    ]

    # Include images with zero predictions.
    for weight in WEIGHTS:
        if weight == 0:
            prob = np.asarray(p3)
        else:
            prob = (
                (1.0 - weight) * np.asarray(p3)
                + weight * np.asarray(p4)
            )

        binary = (
            prob > THRESHOLD
        ).astype(np.uint8)

        processed = morphology_open_close(binary)

        n, labels, stats, _ = (
            cv2.connectedComponentsWithStats(
                processed,
                connectivity=8
            )
        )

        areas = stats[:, cv2.CC_STAT_AREA]

        eligible = np.flatnonzero(
            areas >= MIN_AREA
        )
        eligible = eligible[eligible != 0]

        # Mean probability inside each component.
        sums = np.bincount(
            labels.ravel(),
            weights=prob.ravel(),
            minlength=n
        )

        mean_confidence = (
            sums / np.maximum(areas, 1)
        )

        selected = {
            int(cid) for cid in eligible
            if mean_confidence[cid] >= CONFIDENCE
        }

        result = score_components(
            labels,
            stats,
            selected,
            gt_records
        )

        rows.append({
            "filename": filename,
            "v4_weight": weight,
            **result
        })

    if i % 10 == 0:
        print(
            f"Evaluated {i}/130",
            flush=True
        )

# ------------------------------------------------------------
# Aggregate PQ
# ------------------------------------------------------------

details = pd.DataFrame(rows)

summary = details.groupby(
    "v4_weight",
    as_index=False
)[["TP", "FP", "FN", "sum_iou"]].sum()

denominator = (
    summary.TP
    + 0.5 * (summary.FP + summary.FN)
)

summary["PQ"] = np.where(
    denominator > 0,
    summary.sum_iou / denominator,
    0.0
)

baseline = summary.loc[
    np.isclose(summary.v4_weight, 0.0),
    "PQ"
].iloc[0]

summary["delta_vs_V3"] = (
    summary.PQ - baseline
)

summary = summary.sort_values(
    "PQ",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

DETAILS = WORK / "v4f15_ensemble_130_details.csv"
SUMMARY = WORK / "v4f15_ensemble_130_summary.csv"

details.to_csv(DETAILS, index=False)
summary.to_csv(SUMMARY, index=False)

print("\n" + "=" * 65)
print("V4-F15 — ENSEMBLE RESULTS ON 130 PHOTOGRAPHS")
print("=" * 65)

print(
    summary[
        ["v4_weight", "TP", "FP",
         "FN", "PQ", "delta_vs_V3"]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# Historical reference check.
expected = 0.282300

assert abs(float(baseline) - expected) < 0.00001, (
    f"Baseline mismatch: {baseline:.6f}, "
    f"expected {expected:.6f}"
)

print("\nSUCCESS — V3 reference PQ reproduced.")
print("Best ensemble PQ:", f"{summary.PQ.max():.6f}")
print("Saved:", DETAILS)
print("Saved:", SUMMARY)
print("V4-F15 COMPLETE")


In [ ]:

# V4-F16 — Back up evaluation results only
# No training, inference, or large probability-map uploads.

import json
import shutil
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi

WORK = Path("/kaggle/working")
BACKUP = WORK / "v4f_results_backup"
BACKUP.mkdir(exist_ok=True)

patterns = [
    "v4e_*.csv",
    "v4f_pilot_comparison_*.csv",
    "v4f_validation_130_*.csv",
    "v4f14_ensemble_pilot_*.csv",
    "v4f15_ensemble_130_*.csv",
    "v4f_training/v4f_training_history.csv",
    "v4f_disk_annotation_check.csv",
    "v4f_training_disk_check.csv"
]

copied = []

for pattern in patterns:
    for source in WORK.glob(pattern):
        if source.is_file():
            shutil.copy2(source, BACKUP / source.name)
            copied.append(source.name)

assert copied, "No experiment results found."

metadata = {
    "title": "Solar Filament V4 Evaluation Results",
    "id": "asaadwahag/solar-filament-v4-evaluation-results",
    "licenses": [{"name": "CC0-1.0"}]
}

(BACKUP / "dataset-metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

print("Results to back up:", len(set(copied)))
for name in sorted(set(copied)):
    print(" -", name)

api = KaggleApi()
api.authenticate()

api.dataset_create_new(
    str(BACKUP),
    public=False,
    quiet=False
)

print("\nSUCCESS — V4 evaluation results backed up.")


In [ ]:

# V5-A1 — Inspect V3 predictions on one validation image
# No training, no inference. Uses existing cached maps.

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

WORK = Path("/kaggle/working")

filename = remaining_v4f[0]
stem = Path(filename).stem

image = cv2.imread(
    str(IMAGE_DIR / filename),
    cv2.IMREAD_GRAYSCALE
)

prob = np.load(
    WORK / "v3_validation_130_cache" / f"{stem}.npy"
)

assert image.shape == prob.shape == (2048, 2048)

# Best V3 post-processing
binary = (prob > 0.55).astype(np.uint8)
processed = morphology_open_close(binary)

n, labels, stats, _ = cv2.connectedComponentsWithStats(
    processed, connectivity=8
)

areas = stats[:, cv2.CC_STAT_AREA]

sums = np.bincount(
    labels.ravel(),
    weights=prob.ravel(),
    minlength=n
)

confidence = sums / np.maximum(areas, 1)

selected = [
    cid for cid in range(1, n)
    if areas[cid] >= 800
    and confidence[cid] >= 0.80
]

# Union of expert annotations
gt_union = np.zeros(image.shape, dtype=np.uint8)

for rec in records_by_filename[filename]:
    for gt in get_ground_truth(rec):
        gt_union[gt] = 1

# Red = predicted filament; green = annotation
overlay = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
overlay = overlay.astype(np.float32)

prediction_mask = np.isin(labels, selected)

overlay[prediction_mask, 0] = (
    0.5 * overlay[prediction_mask, 0] + 127.5
)
overlay[gt_union > 0, 1] = (
    0.5 * overlay[gt_union > 0, 1] + 127.5
)

overlay = overlay.astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].imshow(image, cmap="gray")
axes[0].set_title("Original solar image")

axes[1].imshow(overlay)
axes[1].set_title(
    "V3 predictions (red) / Expert annotations (green)"
)

for ax in axes:
    ax.axis("off")

plt.suptitle(
    f"V5-A1 — {filename} | Predicted components: {len(selected)}"
)
plt.tight_layout()
plt.show()

print("Photograph:", filename)
print("Predicted components:", len(selected))
print("Expert annotation records:",
      len(records_by_filename[filename]))


In [ ]:

# V5-A1 FIX — Restore original image directory

from pathlib import Path
import cv2

IMAGE_DIR = (
    Path("/kaggle/input/competitions")
    / "filament-segmentation-2026"
    / "MAGFiLO_1.0_Kaggle_2026"
    / "train"
    / "train_images"
)

# Check the original file location
filename = remaining_v4f[0]
image_path = IMAGE_DIR / filename

if not image_path.exists():
    # Locate the directory in case Kaggle mounted it differently
    candidates = list(
        Path("/kaggle/input").rglob(filename)
    )

    assert candidates, (
        f"Cannot find original image: {filename}"
    )

    image_path = candidates[0]
    IMAGE_DIR = image_path.parent

assert image_path.exists()

image = cv2.imread(
    str(image_path),
    cv2.IMREAD_GRAYSCALE
)

assert image is not None
assert image.shape == (2048, 2048)

print("SUCCESS — Original image directory restored")
print("IMAGE_DIR:", IMAGE_DIR)
print("Image:", filename)
print("Shape:", image.shape)


In [ ]:

# V5-A2 — Diagnose every predicted component
# No training or new inference.

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

filename = "20110119082634Lh.jpeg"
stem = Path(filename).stem

image = cv2.imread(
    str(IMAGE_DIR / filename), cv2.IMREAD_GRAYSCALE
)
prob = np.load(
    Path("/kaggle/working/v3_validation_130_cache")
    / f"{stem}.npy"
)

assert image is not None
assert image.shape == prob.shape

# Exact best V3 post-processing
binary = (prob > 0.55).astype(np.uint8)
processed = morphology_open_close(binary)

n, labels, stats, centroids = cv2.connectedComponentsWithStats(
    processed, connectivity=8
)

areas = stats[:, cv2.CC_STAT_AREA]
sums = np.bincount(
    labels.ravel(),
    weights=prob.ravel(),
    minlength=n
)
conf = sums / np.maximum(areas, 1)

selected = sorted(
    cid for cid in range(1, n)
    if areas[cid] >= 800 and conf[cid] >= 0.80
)

# Keep GT instances separate; do not use GT union for IoU.
gt_masks = [
    gt for rec in records_by_filename[filename]
    for gt in get_ground_truth(rec)
]

gt_union = np.zeros(image.shape, dtype=bool)
for gt in gt_masks:
    gt_union |= gt

rows = []
fig, axes = plt.subplots(
    1, max(len(selected), 1),
    figsize=(4 * max(len(selected), 1), 4)
)
axes = np.atleast_1d(axes)

for j, cid in enumerate(selected):
    x, y, w, h, area = map(int, stats[cid])
    pred = labels == cid

    overlaps = []
    for gt in gt_masks:
        intersection = np.count_nonzero(pred & gt)
        union = area + int(gt.sum()) - intersection
        overlaps.append(
            intersection / union if union else 0.0
        )

    best_iou = max(overlaps, default=0.0)
    best_gt = (
        int(np.argmax(overlaps)) + 1 if overlaps else None
    )
    status = "TP match" if best_iou > 0.5 else "FP"

    rows.append({
        "component": j + 1,
        "area": area,
        "mean_confidence": float(conf[cid]),
        "best_GT": best_gt,
        "best_IoU": best_iou,
        "status": status
    })

    # Zoom around each prediction.
    pad = max(60, int(max(w, h) * 0.4))
    x1, y1 = max(0, x-pad), max(0, y-pad)
    x2 = min(image.shape[1], x+w+pad)
    y2 = min(image.shape[0], y+h+pad)

    crop = image[y1:y2, x1:x2]
    pred_crop = pred[y1:y2, x1:x2]
    gt_crop = gt_union[y1:y2, x1:x2]

    rgb = cv2.cvtColor(crop, cv2.COLOR_GRAY2RGB)
    rgb = rgb.astype(np.float32)
    rgb[pred_crop, 0] = (
        0.5 * rgb[pred_crop, 0] + 127.5
    )
    rgb[gt_crop, 1] = (
        0.5 * rgb[gt_crop, 1] + 127.5
    )

    axes[j].imshow(rgb.astype(np.uint8))
    axes[j].set_title(
        f"Component {j+1}: {status}\n"
        f"IoU={best_iou:.3f}, conf={conf[cid]:.3f}"
    )
    axes[j].axis("off")

plt.tight_layout()
plt.show()

results = pd.DataFrame(rows)
print("\nV5-A2 COMPONENT DIAGNOSIS")
print(results.round(4).to_string(index=False))
print("\nPredicted components:", len(selected))
print("Expert GT instances:", len(gt_masks))
print("Components with IoU > 0.5:",
      int((results.best_IoU > 0.5).sum()))

results.to_csv(
    "/kaggle/working/v5a2_component_diagnosis.csv",
    index=False
)


In [ ]:

# V5-A3 — Identify images with the most V3 false positives
# No training, no inference, no rescoring.

import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")

details = pd.read_csv(
    WORK / "v4f_validation_130_details.csv"
)

v3_errors = details[
    (details["model"] == "V3") &
    (details["method"] == "morphology") &
    (details["filter"] == "confidence_080")
].copy()

assert len(v3_errors) == 130, (
    f"Expected 130 photographs; found {len(v3_errors)}"
)

v3_errors = v3_errors.sort_values(
    ["FP", "FN"], ascending=False
)

print("V5-A3 — TOP 10 PHOTOGRAPHS BY FALSE POSITIVES")
print(
    v3_errors[
        ["filename", "TP", "FP", "FN"]
    ].head(10).to_string(index=False)
)

print("\nTotal false positives:",
      int(v3_errors["FP"].sum()))

assert int(v3_errors["FP"].sum()) == 916

# Choose the photograph with the highest FP count.
diagnostic_filename = v3_errors.iloc[0]["filename"]

print("\nSELECTED FOR V5-A4:")
print(diagnostic_filename)
print("False positives:", int(v3_errors.iloc[0]["FP"]))
print("True positives:", int(v3_errors.iloc[0]["TP"]))
print("False negatives:", int(v3_errors.iloc[0]["FN"]))

print("\nSUCCESS — Diagnostic photograph selected.")


In [ ]:

# ============================================================
# V5-A4 — Diagnose high-confidence false-positive candidates
# Original V3 | Same PQ matching and post-processing
# No training, no inference
# ============================================================

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

WORK = Path("/kaggle/working")
filename = "20150614162514Mh.jpeg"
stem = Path(filename).stem

image = cv2.imread(
    str(IMAGE_DIR / filename), cv2.IMREAD_GRAYSCALE
)
prob = np.load(
    WORK / "v3_validation_130_cache" / f"{stem}.npy",
    mmap_mode="r"
)

assert image is not None
assert image.shape == prob.shape == (2048, 2048)

# 1. Restore V3's best post-processing.
binary = (prob > 0.55).astype(np.uint8)
processed = morphology_open_close(binary)

n, labels, stats, _ = cv2.connectedComponentsWithStats(
    processed, connectivity=8
)

areas = stats[:, cv2.CC_STAT_AREA]
sums = np.bincount(
    labels.ravel(),
    weights=np.asarray(prob).ravel(),
    minlength=n
)
conf = sums / np.maximum(areas, 1)

selected = [
    cid for cid in range(1, n)
    if areas[cid] >= 800 and conf[cid] >= 0.80
]
selected_set = set(selected)

# 2. Load expert masks, preserving annotator records.
gt_records = [
    get_ground_truth(rec)
    for rec in records_by_filename[filename]
]

# Union is for visualisation ONLY.
gt_union = np.zeros(image.shape, dtype=bool)
for masks in gt_records:
    for gt in masks:
        gt_union |= gt

# 3. Identify matches per annotation record.
matched_by_record = [set() for _ in gt_records]
best_iou = {cid: 0.0 for cid in selected}

total_tp = 0
total_fn = 0

for ridx, gt_masks in enumerate(gt_records):
    matched_gts = set()

    for gid, gt in enumerate(gt_masks):
        gt_area = int(gt.sum())
        intersections = np.bincount(
            labels[gt].ravel(), minlength=n
        )

        for cid in np.flatnonzero(intersections):
            cid = int(cid)

            if cid not in selected_set:
                continue

            intersection = int(intersections[cid])
            union = int(areas[cid]) + gt_area - intersection

            if union <= 0:
                continue

            iou = intersection / union
            best_iou[cid] = max(best_iou[cid], iou)

            if iou > 0.5:
                total_tp += 1
                matched_gts.add(gid)
                matched_by_record[ridx].add(cid)

    total_fn += len(gt_masks) - len(matched_gts)

total_fp = sum(
    len(selected_set - matched)
    for matched in matched_by_record
)

print("REFERENCE-STYLE SCORING CHECK")
print("Annotation records:", len(gt_records))
print("Selected components:", len(selected))
print("TP:", total_tp)
print("FP:", total_fp)
print("FN:", total_fn)

assert (total_tp, total_fp, total_fn) == (11, 43, 18), (
    "Reference counts differ; inspect before proceeding."
)

# 4. Summarise components individually.
rows = []

for cid in selected:
    x, y, w, h, area = map(int, stats[cid])

    matched_records = sum(
        cid in matched for matched in matched_by_record
    )

    rows.append({
        "component_id": cid,
        "area": area,
        "confidence": float(conf[cid]),
        "best_iou": best_iou[cid],
        "matched_records": matched_records,
        "unmatched_records": len(gt_records) - matched_records,
        "x": x, "y": y, "w": w, "h": h
    })

df = pd.DataFrame(rows)

# Strongest potential hard negatives: never matched by any
# annotator, despite high model confidence.
candidates = df[
    df["matched_records"] == 0
].sort_values("confidence", ascending=False)

print("\nCOMPONENT SUMMARY")
print("Unique selected components:", len(df))
print("Never matched:", len(candidates))
print(
    "Matched by at least one record:",
    int((df.matched_records > 0).sum())
)

print("\nTOP 8 UNMATCHED COMPONENTS")
print(
    candidates[
        ["component_id", "area", "confidence",
         "best_iou", "unmatched_records"]
    ].head(8).round(4).to_string(index=False)
)

# 5. Show eight crops.
top = candidates.head(8)
count = len(top)

if count:
    fig, axes = plt.subplots(
        2, 4, figsize=(17, 9)
    )
    axes = axes.flatten()

    for ax in axes:
        ax.axis("off")

    for ax, (_, row) in zip(axes, top.iterrows()):
        cid = int(row.component_id)
        x, y, w, h = [
            int(row[k]) for k in ("x", "y", "w", "h")
        ]

        pad = max(70, int(max(w, h) * 0.5))
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(image.shape[1], x + w + pad)
        y2 = min(image.shape[0], y + h + pad)

        crop = image[y1:y2, x1:x2]
        pred = labels[y1:y2, x1:x2] == cid
        gt = gt_union[y1:y2, x1:x2]

        rgb = cv2.cvtColor(crop, cv2.COLOR_GRAY2RGB)
        rgb = rgb.astype(np.float32)

        rgb[pred, 0] = rgb[pred, 0] * 0.5 + 127.5
        rgb[gt, 1] = rgb[gt, 1] * 0.5 + 127.5

        ax.imshow(rgb.astype(np.uint8))
        ax.set_title(
            f"ID {cid} | IoU {row.best_iou:.3f}\n"
            f"Confidence {row.confidence:.3f} | "
            f"Area {int(row.area)}",
            fontsize=10
        )
        ax.axis("off")

    plt.suptitle(
        f"V5-A4 — Unmatched V3 predictions: {filename}\n"
        "Red = selected prediction, "
        "green = union of expert annotations",
        fontsize=13
    )
    plt.tight_layout()
    plt.show()
else:
    print("No components unmatched across all annotators.")

output = WORK / "v5a4_component_diagnosis.csv"
df.to_csv(output, index=False)

print("\nSaved:", output)
print("SUCCESS — V5-A4 complete.")


In [ ]:

# V5-A5 — Segmentation threshold diagnostic
# One difficult image; no training or inference.

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

filename = "20150614162514Mh.jpeg"
prob = np.load(
    Path("/kaggle/working/v3_validation_130_cache")
    / (Path(filename).stem + ".npy"),
    mmap_mode="r"
)

gt_records = [
    get_ground_truth(rec)
    for rec in records_by_filename[filename]
]

rows = []

for threshold in [0.45, 0.50, 0.55, 0.60]:
    binary = (prob > threshold).astype(np.uint8)
    processed = morphology_open_close(binary)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        processed, connectivity=8
    )

    areas = stats[:, cv2.CC_STAT_AREA]

    sums = np.bincount(
        labels.ravel(),
        weights=np.asarray(prob).ravel(),
        minlength=n
    )
    mean_confidence = sums / np.maximum(areas, 1)

    selected = {
        cid for cid in range(1, n)
        if areas[cid] >= 800
        and mean_confidence[cid] >= 0.80
    }

    result = score_components(
        labels, stats, selected, gt_records
    )

    denominator = (
        result["TP"]
        + 0.5 * (result["FP"] + result["FN"])
    )

    pq = (
        result["sum_iou"] / denominator
        if denominator > 0 else 0.0
    )

    rows.append({
        "threshold": threshold,
        "components": len(selected),
        "TP": result["TP"],
        "FP": result["FP"],
        "FN": result["FN"],
        "PQ": pq
    })

results = pd.DataFrame(rows)

print("\nV5-A5 — THRESHOLD DIAGNOSTIC")
print(
    results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

baseline = results.loc[
    np.isclose(results.threshold, 0.55)
].iloc[0]

assert (
    int(baseline.TP),
    int(baseline.FP),
    int(baseline.FN)
) == (11, 43, 18), "Historical reference mismatch"

results.to_csv(
    "/kaggle/working/v5a5_threshold_diagnostic.csv",
    index=False
)

print("\nSUCCESS — V5-A5 complete.")


In [ ]:

# V5-A6 — Analyse unmatched V3 components over 130 images
# Uses V3 best post-processing. No training or inference.

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")
CACHE = WORK / "v3_validation_130_cache"

rows = []

for i, filename in enumerate(remaining_v4f, 1):
    prob = np.load(
        CACHE / (Path(filename).stem + ".npy"),
        mmap_mode="r"
    )

    binary = (prob > 0.55).astype(np.uint8)
    processed = morphology_open_close(binary)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        processed, connectivity=8
    )

    areas = stats[:, cv2.CC_STAT_AREA]

    sums = np.bincount(
        labels.ravel(),
        weights=np.asarray(prob).ravel(),
        minlength=n
    )
    confidence = sums / np.maximum(areas, 1)

    selected = [
        cid for cid in range(1, n)
        if areas[cid] >= 800
        and confidence[cid] >= 0.80
    ]

    gt_records = [
        get_ground_truth(rec)
        for rec in records_by_filename[filename]
    ]

    # Best IoU and matching per annotation record.
    best_iou = {cid: 0.0 for cid in selected}
    matched_records = {cid: 0 for cid in selected}

    for gt_masks in gt_records:
        matched = set()

        for gt in gt_masks:
            intersections = np.bincount(
                labels[gt].ravel(),
                minlength=n
            )

            gt_area = int(gt.sum())

            for cid in np.flatnonzero(intersections):
                cid = int(cid)

                if cid not in best_iou:
                    continue

                intersection = int(intersections[cid])
                union = int(areas[cid]) + gt_area - intersection

                if union <= 0:
                    continue

                iou = intersection / union
                best_iou[cid] = max(best_iou[cid], iou)

                if iou > 0.5:
                    matched.add(cid)

        for cid in matched:
            matched_records[cid] += 1

    for cid in selected:
        iou = best_iou[cid]

        category = (
            "Matched"
            if matched_records[cid] > 0
            else "Partial overlap (0.1–0.5)"
            if iou >= 0.1
            else "Near-zero overlap (<0.1)"
        )

        rows.append({
            "filename": filename,
            "component_id": cid,
            "area": int(areas[cid]),
            "confidence": float(confidence[cid]),
            "best_iou": float(iou),
            "annotation_records": len(gt_records),
            "matched_records": matched_records[cid],
            "unmatched_records": (
                len(gt_records) - matched_records[cid]
            ),
            "category": category
        })

    if i % 20 == 0:
        print(f"Analysed {i}/130", flush=True)

details = pd.DataFrame(rows)

summary = details.groupby(
    "category", as_index=False
).agg(
    components=("component_id", "count"),
    fp_contributions=("unmatched_records", "sum"),
    median_confidence=("confidence", "median"),
    median_area=("area", "median")
)

details.to_csv(
    WORK / "v5a6_component_error_details.csv",
    index=False
)
summary.to_csv(
    WORK / "v5a6_component_error_summary.csv",
    index=False
)

print("\nV5-A6 — UNMATCHED COMPONENT ANALYSIS")
print(summary.to_string(index=False))

print("\nTotal FP contributions:",
      int(details.unmatched_records.sum()))

assert int(details.unmatched_records.sum()) == 916, (
    "Historical FP count mismatch"
)

print("\nSUCCESS — V5-A6 complete.")


In [ ]:

# V5-A7 — Characterise partially overlapping components
# No training, no inference, no mask recomputation.

import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")

df = pd.read_csv(
    WORK / "v5a6_component_error_details.csv"
)

# Components never matched by any annotator but
# overlapping at least one GT instance.
partial = df[
    (df["matched_records"] == 0) &
    (df["best_iou"] >= 0.1) &
    (df["best_iou"] <= 0.5)
].copy()

assert len(partial) == 321, len(partial)

bins = [0.1, 0.2, 0.3, 0.4, 0.5]
labels = ["0.10–0.20", "0.20–0.30",
          "0.30–0.40", "0.40–0.50"]

partial["iou_band"] = pd.cut(
    partial["best_iou"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

summary = (
    partial.groupby("iou_band", observed=False)
    .agg(
        components=("best_iou", "size"),
        fp_contributions=("unmatched_records", "sum"),
        median_area=("area", "median"),
        median_confidence=("confidence", "median")
    )
    .reset_index()
)

summary["share_pct"] = (
    summary["components"] / len(partial) * 100
)

print("\nV5-A7 — PARTIAL-OVERLAP IoU DISTRIBUTION")
print(summary.round(3).to_string(index=False))

print("\nComponents with IoU >= 0.40:",
      int((partial.best_iou >= 0.4).sum()))
print("Components with IoU < 0.30:",
      int((partial.best_iou < 0.3).sum()))
print("Median IoU:", round(partial.best_iou.median(), 4))

output = WORK / "v5a7_partial_iou_distribution.csv"
summary.to_csv(output, index=False)

print("\nSaved:", output)
print("SUCCESS — V5-A7 complete.")


In [ ]:

# V5-A8 — Diagnose shape mismatch for 20 near-TP components
# No training or neural-network inference.

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")
CACHE = WORK / "v3_validation_130_cache"

df = pd.read_csv(
    WORK / "v5a6_component_error_details.csv"
)

candidates = df[
    (df.matched_records == 0) &
    (df.best_iou >= 0.40) &
    (df.best_iou <= 0.50)
].sort_values("best_iou", ascending=False).head(20)

assert len(candidates) == 20

rows = []

for filename, group in candidates.groupby("filename"):
    prob = np.load(
        CACHE / (Path(filename).stem + ".npy"),
        mmap_mode="r"
    )

    binary = (prob > 0.55).astype(np.uint8)
    processed = morphology_open_close(binary)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        processed, connectivity=8
    )

    gt_masks = [
        gt
        for rec in records_by_filename[filename]
        for gt in get_ground_truth(rec)
    ]

    for _, item in group.iterrows():
        cid = int(item.component_id)
        assert 0 < cid < n

        pred_area = int(stats[cid, cv2.CC_STAT_AREA])
        best = None

        for gt in gt_masks:
            gt_area = int(gt.sum())
            intersection = int(
                np.count_nonzero(labels[gt] == cid)
            )
            union = pred_area + gt_area - intersection
            iou = intersection / union if union else 0

            if best is None or iou > best["IoU"]:
                best = {
                    "IoU": iou,
                    "GT_area": gt_area,
                    "intersection": intersection
                }

        if best is None:
            continue

        rows.append({
            "filename": filename,
            "component_id": cid,
            "IoU": best["IoU"],
            "pred_area": pred_area,
            "GT_area": best["GT_area"],
            "area_ratio": pred_area / max(best["GT_area"], 1),
            "GT_coverage": (
                best["intersection"] /
                max(best["GT_area"], 1)
            ),
            "pred_precision": (
                best["intersection"] /
                max(pred_area, 1)
            )
        })

results = pd.DataFrame(rows).sort_values(
    "IoU", ascending=False
)

print("\nV5-A8 — NEAR-THRESHOLD SHAPE DIAGNOSIS")
print(
    results[
        ["component_id", "IoU", "area_ratio",
         "GT_coverage", "pred_precision"]
    ].round(3).to_string(index=False)
)

print("\nMEDIAN STATISTICS")
print(
    results[
        ["IoU", "area_ratio", "GT_coverage", "pred_precision"]
    ].median().round(3).to_string()
)

print("\nPredictions smaller than their best GT:",
      int((results.area_ratio < 1).sum()), "/",
      len(results))

output = WORK / "v5a8_shape_diagnosis.csv"
results.to_csv(output, index=False)

print("\nSaved:", output)
print("SUCCESS — V5-A8 complete.")


In [ ]:

# V5-A9 — Global boundary correction pilot
# No training or model inference.

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")
CACHE = WORK / "v3_pilot_probability_maps"

kernel3 = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (3, 3)
)

operations = ["baseline", "erode3", "dilate3"]
rows = []

for index, filename in enumerate(pilot_files_v4, 1):
    prob = np.load(
        CACHE / (Path(filename).stem + ".npy"),
        mmap_mode="r"
    )

    binary = (prob > 0.55).astype(np.uint8)
    base = morphology_open_close(binary)

    gt_records = [
        get_ground_truth(rec)
        for rec in records_by_filename[filename]
    ]

    for operation in operations:
        if operation == "erode3":
            mask = cv2.erode(base, kernel3, iterations=1)
        elif operation == "dilate3":
            mask = cv2.dilate(base, kernel3, iterations=1)
        else:
            mask = base

        n, labels, stats, _ = (
            cv2.connectedComponentsWithStats(
                mask, connectivity=8
            )
        )

        areas = stats[:, cv2.CC_STAT_AREA]

        sums = np.bincount(
            labels.ravel(),
            weights=np.asarray(prob).ravel(),
            minlength=n
        )
        confidence = sums / np.maximum(areas, 1)

        selected = {
            cid for cid in range(1, n)
            if areas[cid] >= 800
            and confidence[cid] >= 0.80
        }

        result = score_components(
            labels, stats, selected, gt_records
        )

        rows.append({
            "filename": filename,
            "operation": operation,
            **result
        })

    print(f"Evaluated {index}/12", flush=True)

details = pd.DataFrame(rows)

summary = details.groupby(
    "operation", as_index=False
)[["TP", "FP", "FN", "sum_iou"]].sum()

denominator = (
    summary.TP + 0.5 * (summary.FP + summary.FN)
)

summary["PQ"] = np.where(
    denominator > 0,
    summary.sum_iou / denominator,
    0.0
)

summary = summary.sort_values(
    "PQ", ascending=False
)

baseline = float(
    summary.loc[
        summary.operation == "baseline", "PQ"
    ].iloc[0]
)

assert abs(baseline - 0.267127) < 0.00001, (
    f"Historical V3 mismatch: {baseline:.6f}"
)

details.to_csv(
    WORK / "v5a9_boundary_pilot_details.csv",
    index=False
)
summary.to_csv(
    WORK / "v5a9_boundary_pilot_summary.csv",
    index=False
)

print("\nV5-A9 — BOUNDARY CORRECTION PILOT")
print(
    summary[
        ["operation", "TP", "FP", "FN", "PQ"]
    ].to_string(index=False)
)

print("\nSUCCESS — V3 baseline reproduced.")
print("V5-A9 complete.")


In [ ]:

# V5-B1 — GPU memory feasibility test for 512x512 crops
# No optimizer step, no training, no checkpoint changes.

import torch
import torch.nn as nn

assert torch.cuda.is_available()
assert "UNetV3" in globals()

device = torch.device("cuda")
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

test_model = UNetV3().to(device)

checkpoint = (
    "/kaggle/working/v3_training/v3_unet_best.pth"
)

test_model.load_state_dict(
    torch.load(
        checkpoint,
        map_location=device,
        weights_only=True
    ),
    strict=True
)

test_model.train()

batch_size = 2

try:
    x = torch.rand(
        batch_size, 1, 512, 512,
        device=device
    )

    y = torch.zeros_like(x)

    logits = test_model(x)

    criterion = nn.BCEWithLogitsLoss()
    loss = criterion(logits, y)

    loss.backward()

    torch.cuda.synchronize()

    print("SUCCESS — 512x512 training pass completed")
    print("Batch size:", batch_size)
    print("Output shape:", tuple(logits.shape))
    print(
        "Peak allocated GPU memory:",
        round(
            torch.cuda.max_memory_allocated() / 1e9,
            2
        ),
        "GB"
    )
    print(
        "Peak reserved GPU memory:",
        round(
            torch.cuda.max_memory_reserved() / 1e9,
            2
        ),
        "GB"
    )

except torch.cuda.OutOfMemoryError:
    print("GPU out of memory at batch size 2")

finally:
    del test_model
    if "x" in locals():
        del x
    if "y" in locals():
        del y
    if "logits" in locals():
        del logits
    if "loss" in locals():
        del loss

    import gc
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:

# ============================================================
# V5-B2 — Original-image 512x512 dynamic crop dataset
# No training or checkpoint changes.
# ============================================================

import random
import numpy as np
import cv2
import torch

from pathlib import Path
from PIL import Image, ImageDraw
from torch.utils.data import Dataset, DataLoader

PATCH_V5 = 512
SEED_V5 = 42

class SolarDataset512(Dataset):
    def __init__(self, records, image_dir, training=True):
        self.records = records
        self.image_dir = Path(image_dir)
        self.training = training

    def __len__(self):
        return len(self.records) * 2

    def __getitem__(self, index):
        rec = self.records[index // 2]
        filename = rec["file_name"]

        rng = (
            random
            if self.training
            else random.Random(SEED_V5 + index)
        )

        image = cv2.imread(
            str(self.image_dir / filename),
            cv2.IMREAD_GRAYSCALE
        )
        if image is None:
            raise FileNotFoundError(filename)

        h, w = image.shape
        patch = PATCH_V5

        anns = annotations_by_image[rec["id"]]

        # Even indices: filament-centred crop.
        # Odd indices: random crop.
        if index % 2 == 0 and anns:
            ann = rng.choice(anns)
            x, y, bw, bh = ann["bbox"]

            cx = x + bw / 2 + rng.randint(-128, 128)
            cy = y + bh / 2 + rng.randint(-128, 128)

            x0 = int(np.clip(cx - patch / 2, 0, w - patch))
            y0 = int(np.clip(cy - patch / 2, 0, h - patch))
        else:
            x0 = rng.randint(0, w - patch)
            y0 = rng.randint(0, h - patch)

        crop = image[y0:y0+patch, x0:x0+patch].copy()

        # Rasterise polygons in crop-local coordinates.
        mask_image = Image.new("L", (patch, patch), 0)
        draw = ImageDraw.Draw(mask_image)

        for ann in anns:
            for polygon in ann["segmentation"]:
                pts = np.asarray(
                    polygon, dtype=np.float32
                ).reshape(-1, 2)

                if len(pts) < 3:
                    continue

                # Skip polygons whose bounding box lies
                # entirely outside the crop.
                if (
                    pts[:, 0].max() < x0 or
                    pts[:, 0].min() >= x0 + patch or
                    pts[:, 1].max() < y0 or
                    pts[:, 1].min() >= y0 + patch
                ):
                    continue

                pts = pts.copy()
                pts[:, 0] -= x0
                pts[:, 1] -= y0

                draw.polygon(
                    [tuple(p) for p in pts],
                    fill=1
                )

        mask = np.asarray(mask_image, dtype=np.uint8).copy()

        if self.training:
            if rng.random() < 0.5:
                crop = np.fliplr(crop)
                mask = np.fliplr(mask)

            if rng.random() < 0.5:
                crop = np.flipud(crop)
                mask = np.flipud(mask)

            k = rng.randint(0, 3)
            crop = np.rot90(crop, k)
            mask = np.rot90(mask, k)

        x_tensor = torch.from_numpy(
            np.ascontiguousarray(crop, dtype=np.float32)
        ).unsqueeze(0) / 255.0

        y_tensor = torch.from_numpy(
            np.ascontiguousarray(mask, dtype=np.float32)
        ).unsqueeze(0)

        return x_tensor, y_tensor


# ------------------------------------------------------------
# Reuse the original physical-image split
# ------------------------------------------------------------

assert len(train_records) == 922
assert len(val_records) == 232
assert set(r["file_name"] for r in train_records).isdisjoint(
    set(r["file_name"] for r in val_records)
)

train_dataset_v5 = SolarDataset512(
    train_records, IMAGE_DIR, training=True
)

val_dataset_v5 = SolarDataset512(
    val_records, IMAGE_DIR, training=False
)

train_loader_v5 = DataLoader(
    train_dataset_v5,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader_v5 = DataLoader(
    val_dataset_v5,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

images, masks = next(iter(train_loader_v5))

assert images.shape == (2, 1, 512, 512)
assert masks.shape == (2, 1, 512, 512)
assert masks.min() >= 0 and masks.max() <= 1
assert images.min() >= 0 and images.max() <= 1

print("V5-B2 DATASET READY")
print("Training records:", len(train_records))
print("Validation records:", len(val_records))
print("Training crops per epoch:", len(train_dataset_v5))
print("Validation crops:", len(val_dataset_v5))
print("Batch image shape:", tuple(images.shape))
print("Batch mask shape:", tuple(masks.shape))
print("Foreground percentage:",
      round(float(masks.mean()) * 100, 3))
print("Original-image inputs:", True)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

# ============================================================
# V5-B3 — Larger-context fine-tuning from V3
# 512x512 crops | 5 epochs | Resumable checkpoints
# ============================================================

import os
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path

WORK = Path("/kaggle/working")
OUT = WORK / "v5b_training"
OUT.mkdir(parents=True, exist_ok=True)

V3_CHECKPOINT = WORK / "v3_training/v3_unet_best.pth"
BEST_PATH = OUT / "v5b_unet_best.pth"
LATEST_PATH = OUT / "v5b_latest_state.pth"
HISTORY_PATH = OUT / "v5b_training_history.csv"

assert V3_CHECKPOINT.exists(), "V3 checkpoint missing"
assert "UNetV3" in globals(), "Restore V3 architecture first"
assert "train_loader_v5" in globals(), "Run V5-B2 first"
assert "val_loader_v5" in globals(), "Run V5-B2 first"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

EPOCHS = 5
LR = 3e-5

# Create an entirely separate V5 model.
model_v5 = UNetV3().to(DEVICE)

weights = torch.load(
    V3_CHECKPOINT,
    map_location=DEVICE,
    weights_only=True
)

model_v5.load_state_dict(weights, strict=True)
del weights

optimizer_v5 = torch.optim.AdamW(
    model_v5.parameters(),
    lr=LR,
    weight_decay=1e-4
)

criterion_bce_v5 = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([20.0], device=DEVICE)
)

def loss_v5(logits, targets):
    bce_loss = criterion_bce_v5(logits, targets)

    probs = torch.sigmoid(logits)
    intersection = (
        probs * targets
    ).sum(dim=(1, 2, 3))

    denominator = (
        probs.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
    )

    dice_loss = 1 - (
        (2 * intersection + 1)
        / (denominator + 1)
    ).mean()

    return bce_loss + dice_loss


@torch.inference_mode()
def evaluate_v5(model, loader, threshold=0.45):
    model.eval()

    total_dice = 0.0
    samples = 0

    for images, masks in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        predictions = (
            torch.sigmoid(model(images)) > threshold
        ).float()

        intersection = (
            predictions * masks
        ).sum(dim=(1, 2, 3))

        denominator = (
            predictions.sum(dim=(1, 2, 3))
            + masks.sum(dim=(1, 2, 3))
        )

        dice = (
            (2 * intersection + 1)
            / (denominator + 1)
        )

        total_dice += float(dice.sum().item())
        samples += len(images)

    return total_dice / max(samples, 1)


def atomic_save_v5(obj, path):
    temp = path.with_suffix(".tmp")
    torch.save(obj, temp)
    os.replace(temp, path)


# ------------------------------------------------------------
# Resume from a previous completed epoch when available.
# ------------------------------------------------------------

start_epoch = 1
best_dice = -1.0
history = []

if LATEST_PATH.exists():
    state = torch.load(
        LATEST_PATH,
        map_location=DEVICE,
        weights_only=False
    )

    model_v5.load_state_dict(state["model"])
    optimizer_v5.load_state_dict(state["optimizer"])

    start_epoch = int(state["epoch"]) + 1
    best_dice = float(state["best_dice"])

    if "python_rng" in state:
        random.setstate(state["python_rng"])

    if "numpy_rng" in state:
        np.random.set_state(state["numpy_rng"])

    if "torch_rng" in state:
        torch.set_rng_state(state["torch_rng"].cpu())

    if (
        state.get("cuda_rng") is not None
        and torch.cuda.is_available()
    ):
        torch.cuda.set_rng_state_all(state["cuda_rng"])

    if HISTORY_PATH.exists():
        history = pd.read_csv(
            HISTORY_PATH
        ).to_dict("records")

    print(
        f"Resuming from epoch {start_epoch}",
        flush=True
    )


# ------------------------------------------------------------
# Train and validate.
# ------------------------------------------------------------

print("\nV5-B3 TRAINING")
print("Device:", DEVICE)
print("Crop size: 512 × 512")
print("Batch size:", train_loader_v5.batch_size)
print("Epochs:", EPOCHS)
print("Learning rate:", LR)
print("Starting epoch:", start_epoch)
print(flush=True)

for epoch in range(start_epoch, EPOCHS + 1):
    start_time = time.perf_counter()

    model_v5.train()
    running_loss = 0.0
    total_samples = 0

    for batch_index, (images, masks) in enumerate(
        train_loader_v5, 1
    ):
        images = images.to(
            DEVICE, non_blocking=True
        )
        masks = masks.to(
            DEVICE, non_blocking=True
        )

        optimizer_v5.zero_grad(set_to_none=True)

        logits = model_v5(images)
        loss = loss_v5(logits, masks)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss: epoch {epoch}, "
                f"batch {batch_index}"
            )

        loss.backward()
        optimizer_v5.step()

        batch_size = len(images)
        running_loss += (
            float(loss.item()) * batch_size
        )
        total_samples += batch_size

        if batch_index % 200 == 0:
            print(
                f"Epoch {epoch} | "
                f"Batch {batch_index}/{len(train_loader_v5)}",
                flush=True
            )

    train_loss = running_loss / total_samples

    val_dice = evaluate_v5(
        model_v5,
        val_loader_v5,
        threshold=0.45
    )

    if val_dice > best_dice:
        best_dice = val_dice
        atomic_save_v5(
            model_v5.state_dict(),
            BEST_PATH
        )
        improved = True
    else:
        improved = False

    elapsed = time.perf_counter() - start_time

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_dice": val_dice,
        "best_dice": best_dice,
        "learning_rate": LR,
        "elapsed_sec": elapsed
    })

    pd.DataFrame(history).to_csv(
        HISTORY_PATH, index=False
    )

    state = {
        "epoch": epoch,
        "model": model_v5.state_dict(),
        "optimizer": optimizer_v5.state_dict(),
        "best_dice": best_dice,
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        )
    }

    atomic_save_v5(state, LATEST_PATH)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Loss {train_loss:.4f} | "
        f"Val Dice {val_dice:.4f} | "
        f"Best {best_dice:.4f} | "
        f"Improved {improved} | "
        f"Time {elapsed:.1f}s",
        flush=True
    )


# ------------------------------------------------------------
# Final report.
# ------------------------------------------------------------

print("\nSUCCESS — V5-B3 TRAINING COMPLETE")
print("Best validation Dice:", round(best_dice, 6))
print("Best checkpoint:", BEST_PATH)
print("Resumable state:", LATEST_PATH)
print("Training history:", HISTORY_PATH)

assert BEST_PATH.exists()
assert LATEST_PATH.exists()


In [ ]:

# ============================================================
# V5-B4 — Back up 512x512 fine-tuned V5 model
# No training or GPU inference.
# ============================================================

import json
import shutil
from pathlib import Path

import pandas as pd
import torch
from kaggle.api.kaggle_api_extended import KaggleApi

WORK = Path("/kaggle/working")
SOURCE = WORK / "v5b_training"
BACKUP = WORK / "v5b_checkpoint_backup"

BACKUP.mkdir(parents=True, exist_ok=True)

filenames = [
    "v5b_unet_best.pth",
    "v5b_latest_state.pth",
    "v5b_training_history.csv"
]

for name in filenames:
    src = SOURCE / name
    assert src.exists() and src.stat().st_size > 0, name

    shutil.copy2(src, BACKUP / name)
    print("Copied:", name)

# Verify the saved best model and training history.
history = pd.read_csv(
    SOURCE / "v5b_training_history.csv"
)

best_row = history.loc[
    history["val_dice"].idxmax()
]

assert int(best_row["epoch"]) == 4

weights = torch.load(
    SOURCE / "v5b_unet_best.pth",
    map_location="cpu",
    weights_only=True
)
assert isinstance(weights, dict)
del weights

experiment = {
    "architecture": "UNetV3",
    "initialized_from": "v3_unet_best.pth",
    "input": "original grayscale JPEG",
    "crop_size": 512,
    "batch_size": 2,
    "epochs": 5,
    "learning_rate": 3e-5,
    "best_epoch": int(best_row["epoch"]),
    "best_crop_dice": float(best_row["val_dice"]),
    "full_image_PQ": None,
    "note": "Not evaluated on full-image PQ yet"
}

(BACKUP / "experiment_metadata.json").write_text(
    json.dumps(experiment, indent=2)
)

dataset_metadata = {
    "title": "Solar Filament V5B Checkpoints",
    "id": "asaadwahag/solar-filament-v5b-checkpoints",
    "licenses": [{"name": "other"}]
}

(BACKUP / "dataset-metadata.json").write_text(
    json.dumps(dataset_metadata, indent=2)
)

api = KaggleApi()
api.authenticate()

api.dataset_create_new(
    str(BACKUP),
    public=False,
    quiet=False
)

print("\nSUCCESS — V5-B4 BACKUP COMPLETE")
print("Dataset: asaadwahag/solar-filament-v5b-checkpoints")
print("Best epoch:", int(best_row["epoch"]))
print("Best Dice:", round(float(best_row["val_dice"]), 6))


In [ ]:

from pathlib import Path

backup = Path("/kaggle/working/v5b_checkpoint_backup")

for name in [
    "v5b_unet_best.pth",
    "v5b_latest_state.pth",
    "v5b_training_history.csv",
    "experiment_metadata.json"
]:
    p = backup / name
    print(name, "OK" if p.exists() else "MISSING")

print("\nCheck the Kaggle dataset page to confirm upload.")


In [ ]:

# ============================================================
# V5-B5A — V5 PILOT INFERENCE (12 IMAGES)
# Uses best V5 checkpoint from epoch 4.
# No training. Does not modify V3 or V5 checkpoints.
# ============================================================

import time
import numpy as np
import cv2
import torch
from pathlib import Path

WORK = Path("/kaggle/working")
CACHE = WORK / "v5b_pilot_probability_maps"
CACHE.mkdir(parents=True, exist_ok=True)

CHECKPOINT = WORK / "v5b_training/v5b_unet_best.pth"
assert CHECKPOINT.exists()

assert "UNetV3" in globals()
assert "pilot_files_v4" in globals()
assert len(pilot_files_v4) == 12
assert "IMAGE_DIR" in globals()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Restore epoch-4 model, not epoch-5 model.
model_v5 = UNetV3().to(DEVICE)

model_v5.load_state_dict(
    torch.load(
        CHECKPOINT,
        map_location=DEVICE,
        weights_only=True
    ),
    strict=True
)

model_v5.eval()

@torch.inference_mode()
def predict_v5_full(image_path, patch=256, stride=128, batch=8):
    image = cv2.imread(
        str(image_path), cv2.IMREAD_GRAYSCALE
    )

    assert image is not None, image_path
    assert image.shape == (2048, 2048)

    image = image.astype(np.float32) / 255.0
    h, w = image.shape

    positions = [
        (y, x)
        for y in range(0, h-patch+1, stride)
        for x in range(0, w-patch+1, stride)
    ]

    accumulated = np.zeros((h, w), dtype=np.float32)
    counts = np.zeros((h, w), dtype=np.float32)

    for start in range(0, len(positions), batch):
        batch_positions = positions[start:start+batch]

        patches = np.stack([
            image[y:y+patch, x:x+patch]
            for y, x in batch_positions
        ])

        inputs = torch.from_numpy(
            patches.copy()
        ).unsqueeze(1).to(DEVICE)

        predictions = torch.sigmoid(
            model_v5(inputs)
        ).cpu().numpy()[:, 0]

        for prediction, (y, x) in zip(
            predictions, batch_positions
        ):
            accumulated[y:y+patch, x:x+patch] += prediction
            counts[y:y+patch, x:x+patch] += 1

    assert np.all(counts > 0)
    return accumulated / counts


start_time = time.perf_counter()

for i, filename in enumerate(pilot_files_v4, 1):
    output = CACHE / (Path(filename).stem + ".npy")

    if not output.exists():
        image_path = IMAGE_DIR / filename
        assert image_path.exists(), filename

        probability = predict_v5_full(image_path)

        assert probability.shape == (2048, 2048)
        assert np.isfinite(probability).all()

        np.save(output, probability)

    print(
        f"Processed {i}/12 — {filename}",
        flush=True
    )

print("\nSUCCESS — V5-B5A PILOT INFERENCE COMPLETE")
print("Pilot images:", len(pilot_files_v4))
print(
    "Cached maps:",
    sum(
        (CACHE / (Path(f).stem + ".npy")).exists()
        for f in pilot_files_v4
    )
)
print(
    "Elapsed minutes:",
    round((time.perf_counter()-start_time)/60, 2)
)
print("Cache:", CACHE)


In [ ]:

# V5-B5B — Final pilot comparison
# V3 vs V5-B, 12 photographs
# No training or inference

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working")

CACHES = {
    "V3": WORK / "v3_pilot_probability_maps",
    "V5B": WORK / "v5b_pilot_probability_maps"
}

assert len(pilot_files_v4) == 12
assert callable(get_ground_truth)
assert callable(score_components)
assert callable(morphology_open_close)

rows = []

for index, filename in enumerate(pilot_files_v4, 1):

    gt_records = [
        get_ground_truth(rec)
        for rec in records_by_filename[filename]
    ]

    for model_name, cache in CACHES.items():

        path = cache / (Path(filename).stem + ".npy")
        assert path.exists(), path

        prob = np.load(path, mmap_mode="r")
        assert prob.shape == (2048, 2048)

        binary = (prob > 0.55).astype(np.uint8)
        processed = morphology_open_close(binary)

        n, labels, stats, _ = (
            cv2.connectedComponentsWithStats(
                processed, connectivity=8
            )
        )

        areas = stats[:, cv2.CC_STAT_AREA]

        sums = np.bincount(
            labels.ravel(),
            weights=np.asarray(prob).ravel(),
            minlength=n
        )

        confidence = sums / np.maximum(areas, 1)

        selected = {
            cid for cid in range(1, n)
            if areas[cid] >= 800
            and confidence[cid] >= 0.80
        }

        result = score_components(
            labels, stats, selected, gt_records
        )

        rows.append({
            "filename": filename,
            "model": model_name,
            **result
        })

    print(f"Compared {index}/12", flush=True)

details = pd.DataFrame(rows)

summary = details.groupby(
    "model", as_index=False
)[["TP", "FP", "FN", "sum_iou"]].sum()

denominator = (
    summary["TP"] +
    0.5 * (summary["FP"] + summary["FN"])
)

summary["PQ"] = np.where(
    denominator > 0,
    summary["sum_iou"] / denominator,
    0.0
)

summary = summary.sort_values(
    "PQ", ascending=False
)

details.to_csv(
    WORK / "v5b_pilot_comparison_details.csv",
    index=False
)

summary.to_csv(
    WORK / "v5b_pilot_comparison_summary.csv",
    index=False
)

print("\nV5-B5B — V3 VS V5 PILOT")
print(
    summary[
        ["model", "TP", "FP", "FN", "PQ"]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

v3_pq = float(
    summary.loc[
        summary.model == "V3", "PQ"
    ].iloc[0]
)

assert abs(v3_pq - 0.267127) < 0.00001, (
    f"V3 reference mismatch: {v3_pq:.6f}"
)

v5_pq = float(
    summary.loc[
        summary.model == "V5B", "PQ"
    ].iloc[0]
)

print("\nV3 reference reproduced: YES")
print("V5 minus V3:", f"{v5_pq-v3_pq:+.6f}")
print("SUCCESS — V5-B5B complete.")


In [ ]:

from pathlib import Path

root = Path("/kaggle/working")

files = [
    root / "v5b_training/v5b_unet_best.pth",
    root / "v5b_training/v5b_latest_state.pth",
    root / "v5b_pilot_comparison_summary.csv",
]

for path in files:
    print(
        "FOUND" if path.is_file() else "MISSING",
        path,
    )

print(
    "Pilot probability maps:",
    len(list((root / "v5b_pilot_probability_maps").glob("*.npy")))
)


In [ ]:

import torch

path = "/kaggle/working/v5b_training/v5b_unet_best.pth"

checkpoint = torch.load(
    path,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("Number of entries:", len(checkpoint))
    print("First 10 keys:", list(checkpoint.keys())[:10])

print("CUDA available:", torch.cuda.is_available())


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F

# Original V3/V5-B architecture
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNetV3(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = DoubleConv(1, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.bottleneck = DoubleConv(128, 256)

        self.pool = nn.MaxPool2d(2)

        self.dec3 = DoubleConv(384, 128)
        self.dec2 = DoubleConv(192, 64)
        self.dec1 = DoubleConv(96, 32)

        self.output = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))

        d3 = F.interpolate(
            b, size=e3.shape[2:],
            mode="bilinear", align_corners=False
        )
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = F.interpolate(
            d3, size=e2.shape[2:],
            mode="bilinear", align_corners=False
        )
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = F.interpolate(
            d2, size=e1.shape[2:],
            mode="bilinear", align_corners=False
        )
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.output(d1)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model_v5_test = UNetV3().to(device)

weights = torch.load(
    "/kaggle/working/v5b_training/v5b_unet_best.pth",
    map_location="cpu",
    weights_only=True
)

model_v5_test.load_state_dict(weights, strict=True)
model_v5_test.eval()

with torch.inference_mode():
    dummy = torch.zeros(1, 1, 512, 512, device=device)
    output = model_v5_test(dummy)

print("Checkpoint loaded: SUCCESS")
print("Output shape:", tuple(output.shape))
print("Parameters:", sum(p.numel() for p in model_v5_test.parameters()))

del dummy, output


In [ ]:

import json
import time
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split

ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

# Reconstruct the original pilot selection
annotation_path = next(ROOT.rglob(
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
))

with open(annotation_path) as f:
    coco = json.load(f)

image_dir = annotation_path.parent / "train_images"

filenames = sorted({
    rec["file_name"] for rec in coco["images"]
})

_, val_files = train_test_split(
    filenames, test_size=0.20, random_state=42
)

rng = np.random.default_rng(42)
pilot_files = sorted(
    rng.choice(sorted(val_files), 12, replace=False).tolist()
)

# Confirm these are the previously used pilot images
old_cache = WORK / "v5b_pilot_probability_maps"

expected = {Path(f).stem for f in pilot_files}
existing = {p.stem for p in old_cache.glob("*.npy")}

assert expected == existing, (
    "Pilot files differ from the original cache. Stop here."
)

@torch.inference_mode()
def predict_v5_512(image_path):
    image = np.asarray(
        Image.open(image_path).convert("L"),
        dtype=np.float32
    ) / 255.0

    assert image.shape == (2048, 2048)

    patch = 512
    stride = 256
    batch_size = 2

    positions = [
        (y, x)
        for y in range(0, 2048-patch+1, stride)
        for x in range(0, 2048-patch+1, stride)
    ]

    total = np.zeros_like(image)
    counts = np.zeros_like(image)

    for start in range(0, len(positions), batch_size):
        current = positions[start:start+batch_size]

        patches = np.stack([
            image[y:y+patch, x:x+patch]
            for y, x in current
        ])

        inputs = torch.from_numpy(
            patches.copy()
        ).unsqueeze(1).to(device)

        probs = torch.sigmoid(
            model_v5_test(inputs)
        ).cpu().numpy()[:, 0]

        for prob, (y, x) in zip(probs, current):
            total[y:y+patch, x:x+patch] += prob
            counts[y:y+patch, x:x+patch] += 1

    assert np.all(counts > 0)
    return total / counts

new_cache = WORK / "v5b_pilot_512_probability_maps"
new_cache.mkdir(exist_ok=True)

filename = pilot_files[0]
output_path = new_cache / (Path(filename).stem + ".npy")

start = time.perf_counter()

if not output_path.exists():
    probability = predict_v5_512(image_dir / filename)
    assert probability.shape == (2048, 2048)
    assert np.isfinite(probability).all()
    np.save(output_path, probability)

print("Image:", filename)
print("New prediction:", output_path)
print("Saved:", output_path.exists())
print("Inference patches: 49")
print("Elapsed seconds:", round(time.perf_counter()-start, 1))


In [ ]:

import time
import numpy as np
from pathlib import Path

start = time.perf_counter()
completed = 0
skipped = 0

model_v5_test.eval()

for i, filename in enumerate(pilot_files, start=1):
    image_path = image_dir / filename
    output_path = new_cache / (
        Path(filename).stem + ".npy"
    )

    if output_path.exists():
        skipped += 1
        print(f"[{i}/12] Already saved: {filename}")
        continue

    probability = predict_v5_512(image_path)

    assert probability.shape == (2048, 2048)
    assert np.isfinite(probability).all()

    np.save(output_path, probability)

    completed += 1
    print(f"[{i}/12] Saved: {filename}")

saved_files = list(new_cache.glob("*.npy"))

print("\n--- V5-B 512x512 Inference Summary ---")
print("New predictions:", completed)
print("Previously saved:", skipped)
print("Total saved:", len(saved_files))
print("Expected:", len(pilot_files))
print("Elapsed seconds:", round(time.perf_counter()-start, 1))

assert len(saved_files) == len(pilot_files), (
    "Some pilot predictions are missing."
)


In [ ]:

# V5-B — FINAL 512 INFERENCE PILOT COMPARISON
# No training, no inference, no submission.

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageDraw
from collections import defaultdict

WORK = Path("/kaggle/working")

CACHES = {
    "V3_256": WORK / "v3_pilot_probability_maps",
    "V5B_256": WORK / "v5b_pilot_probability_maps",
    "V5B_512": WORK / "v5b_pilot_512_probability_maps",
}

# Build annotation mappings from the loaded COCO JSON.
records = defaultdict(list)
annotations = defaultdict(list)

for rec in coco["images"]:
    records[rec["file_name"]].append(rec)

for ann in coco["annotations"]:
    annotations[ann["image_id"]].append(ann)

# Reproduce the original PIL ground-truth masks.
def pilot_gt_masks(rec):
    width = int(rec["width"])
    height = int(rec["height"])
    masks = []

    for ann in annotations[rec["id"]]:
        canvas = Image.new("L", (width, height), 0)
        draw = ImageDraw.Draw(canvas)

        for polygon in ann["segmentation"]:
            points = np.asarray(
                polygon, dtype=np.float32
            ).reshape(-1, 2)

            if len(points) >= 3:
                draw.polygon(
                    [tuple(p) for p in points], fill=1
                )

        masks.append(np.asarray(canvas, dtype=bool))

    return masks

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (3, 3)
)

# The same morphology as the original comparison.
def pilot_morphology(binary):
    opened = cv2.morphologyEx(
        binary, cv2.MORPH_OPEN, kernel
    )
    return cv2.morphologyEx(
        opened, cv2.MORPH_CLOSE, kernel
    )

# Reference-style instance matcher from our original notebook.
def pilot_score(labels, stats, selected, gt_records):
    n = len(stats)
    areas = stats[:, cv2.CC_STAT_AREA]
    totals = dict(TP=0, FP=0, FN=0, sum_iou=0.0)

    for gt_masks in gt_records:
        matched_preds = set()
        matched_gts = set()

        for gt_id, gt in enumerate(gt_masks):
            intersections = np.bincount(
                labels[gt].ravel(), minlength=n
            )
            ids = np.flatnonzero(intersections)

            gt_area = int(gt.sum())

            for cid in ids:
                cid = int(cid)
                if cid == 0 or cid not in selected:
                    continue

                intersection = int(intersections[cid])
                union = int(areas[cid]) + gt_area - intersection

                if union > 0:
                    iou = intersection / union

                    if iou > 0.5:
                        totals["TP"] += 1
                        totals["sum_iou"] += iou
                        matched_preds.add(cid)
                        matched_gts.add(gt_id)

        totals["FP"] += len(selected) - len(matched_preds)
        totals["FN"] += len(gt_masks) - len(matched_gts)

    return totals

# Score all three existing probability-map caches.
assert len(pilot_files) == 12

rows = []

for index, filename in enumerate(pilot_files, 1):
    gt_records = [
        pilot_gt_masks(rec)
        for rec in records[filename]
    ]
    assert gt_records, filename

    for model_name, cache in CACHES.items():
        path = cache / (Path(filename).stem + ".npy")
        assert path.exists(), path

        prob = np.load(path, mmap_mode="r")
        assert prob.shape == (2048, 2048)
        assert np.isfinite(prob).all()

        binary = (prob > 0.55).astype(np.uint8)
        processed = pilot_morphology(binary)

        n, labels, stats, _ = cv2.connectedComponentsWithStats(
            processed, connectivity=8
        )

        areas = stats[:, cv2.CC_STAT_AREA]

        sums = np.bincount(
            labels.ravel(),
            weights=np.asarray(prob).ravel(),
            minlength=n
        )
        confidence = sums / np.maximum(areas, 1)

        selected = {
            cid for cid in range(1, n)
            if areas[cid] >= 800
            and confidence[cid] >= 0.80
        }

        result = pilot_score(
            labels, stats, selected, gt_records
        )

        rows.append({
            "filename": filename,
            "model": model_name,
            **result
        })

    print(f"Scored {index}/12", flush=True)

details = pd.DataFrame(rows)

summary = details.groupby(
    "model", as_index=False
)[["TP", "FP", "FN", "sum_iou"]].sum()

denom = (
    summary["TP"]
    + 0.5 * (summary["FP"] + summary["FN"])
)

summary["PQ"] = np.where(
    denom > 0, summary["sum_iou"] / denom, 0.0
)

# Confirm the historical reference is unchanged.
v3_pq = float(
    summary.loc[
        summary["model"] == "V3_256", "PQ"
    ].iloc[0]
)

assert abs(v3_pq - 0.267127) < 0.00001, (
    f"V3 reference mismatch: {v3_pq:.6f}"
)

details.to_csv(
    WORK / "v5b_512_pilot_details.csv", index=False
)
summary.to_csv(
    WORK / "v5b_512_pilot_summary.csv", index=False
)

print("\nFINAL PILOT COMPARISON")
print(
    summary[["model", "TP", "FP", "FN", "PQ"]]
    .sort_values("PQ", ascending=False)
    .to_string(index=False)
)
print("\nV3 reference reproduced: YES")


In [ ]:

# V5-B — 130-image validation at 512x512
# Existing V3 vs V5-B; no training or Kaggle submission.

import time
from pathlib import Path
import numpy as np
import pandas as pd
import cv2

WORK = Path("/kaggle/working")

V3_CACHE = WORK / "v3_validation_130_cache"
RESULT = WORK / "v5b_512_validation_130_details.csv"
SUMMARY = WORK / "v5b_512_validation_130_summary.csv"

# Same 130 photographs as the original V3 validation.
remaining_130 = sorted(set(val_files) - set(pilot_files))
assert len(remaining_130) == 130

# Ensure all original V3 predictions exist.
missing_v3 = [
    name for name in remaining_130
    if not (V3_CACHE / (Path(name).stem + ".npy")).is_file()
]
assert not missing_v3, (
    f"Missing {len(missing_v3)} V3 probability maps: "
    f"{missing_v3[:3]}"
)

# Check functions already established in this session.
assert callable(predict_v5_512)
assert callable(pilot_gt_masks)
assert callable(pilot_morphology)
assert callable(pilot_score)

model_v5_test.eval()

# Resume safely if the cell is interrupted.
if RESULT.exists():
    previous = pd.read_csv(RESULT)
    assert set(previous["model"]).issubset(
        {"V3_256", "V5B_512"}
    )
    rows = previous.to_dict("records")
else:
    rows = []

completed = {
    name for name in remaining_130
    if {
        row["model"]
        for row in rows
        if row["filename"] == name
    } == {"V3_256", "V5B_512"}
}

print("Validation images:", len(remaining_130))
print("Previously completed:", len(completed))
print("Remaining:", len(remaining_130) - len(completed))

start = time.perf_counter()

def score_probability_map(prob, gt_records):
    assert prob.shape == (2048, 2048)
    assert np.isfinite(prob).all()

    binary = (prob > 0.55).astype(np.uint8)
    processed = pilot_morphology(binary)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        processed, connectivity=8
    )

    areas = stats[:, cv2.CC_STAT_AREA]

    sums = np.bincount(
        labels.ravel(),
        weights=np.asarray(prob).ravel(),
        minlength=n
    )
    confidence = sums / np.maximum(areas, 1)

    selected = {
        cid for cid in range(1, n)
        if areas[cid] >= 800
        and confidence[cid] >= 0.80
    }

    return pilot_score(labels, stats, selected, gt_records)


for index, filename in enumerate(remaining_130, 1):

    if filename in completed:
        continue

    # All expert annotation records for the photograph.
    gt_records = [
        pilot_gt_masks(rec)
        for rec in records[filename]
    ]
    assert gt_records, filename

    # Existing V3 prediction — no additional V3 inference.
    v3_path = V3_CACHE / (Path(filename).stem + ".npy")
    v3_prob = np.load(v3_path, mmap_mode="r")

    v3_result = score_probability_map(
        v3_prob, gt_records
    )

    del v3_prob

    # New V5-B prediction using 512x512 patches.
    v5_prob = predict_v5_512(image_dir / filename)

    v5_result = score_probability_map(
        v5_prob, gt_records
    )

    del v5_prob, gt_records

    # Commit both model results together.
    rows.extend([
        {
            "filename": filename,
            "model": "V3_256",
            **v3_result
        },
        {
            "filename": filename,
            "model": "V5B_512",
            **v5_result
        }
    ])

    # Save after every completed image.
    temp = RESULT.with_suffix(".tmp")
    pd.DataFrame(rows).to_csv(temp, index=False)
    temp.replace(RESULT)

    completed.add(filename)

    if len(completed) % 10 == 0:
        print(
            f"Completed {len(completed)}/130 | "
            f"Elapsed {time.perf_counter()-start:.1f}s",
            flush=True
        )


details = pd.read_csv(RESULT)

assert len(details) == 260
assert details.groupby("filename")["model"].nunique().eq(2).all()

summary = details.groupby(
    "model", as_index=False
)[["TP", "FP", "FN", "sum_iou"]].sum()

denominator = (
    summary["TP"] +
    0.5 * (summary["FP"] + summary["FN"])
)

summary["PQ"] = summary["sum_iou"] / denominator

summary = summary.sort_values(
    "PQ", ascending=False
)

summary.to_csv(SUMMARY, index=False)

print("\n" + "=" * 55)
print("FINAL — 130-IMAGE VALIDATION")
print("=" * 55)

print(
    summary[["model", "TP", "FP", "FN", "PQ"]]
    .to_string(index=False, float_format=lambda x: f"{x:.6f}")
)

v3_pq = float(
    summary.loc[summary.model == "V3_256", "PQ"].iloc[0]
)

print("\nHistorical V3 reference: 0.282300")
print("Current V3 reference:", round(v3_pq, 6))
print(
    "Reference reproduced:",
    abs(v3_pq - 0.282300) < 0.00001
)

v5_pq = float(
    summary.loc[summary.model == "V5B_512", "PQ"].iloc[0]
)

print("V5-B minus V3:", f"{v5_pq - v3_pq:+.6f}")
print("Result saved:", SUMMARY)


In [ ]:

# V5-B — TEST SET SUBMISSION GENERATION
# No training. No change to existing checkpoints.
# Resumable: one CSV part per test photograph.

import time
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from pycocotools import mask as mask_utils

WORK = Path("/kaggle/working")
PARTS = WORK / "v5b_512_test_parts"
PARTS.mkdir(parents=True, exist_ok=True)

OUTPUT = WORK / "submission_v5b_512.csv"

# Find the official competition test images.
test_dirs = [
    p for p in Path("/kaggle/input").rglob("test_images")
    if p.is_dir()
]
assert len(test_dirs) == 1, (
    f"Expected one test folder, found {len(test_dirs)}"
)

test_images = sorted(test_dirs[0].glob("*.jpeg"))
assert len(test_images) == 180, (
    f"Expected 180 test images, found {len(test_images)}"
)

# Reuse the validated 512x512 inference function.
assert callable(predict_v5_512)
assert not model_v5_test.training

THRESHOLD = 0.55
MIN_AREA = 800
CONFIDENCE = 0.80

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (3, 3)
)

def create_v5b_submission_rows(image_path):
    prob = predict_v5_512(image_path)

    assert prob.shape == (2048, 2048)
    assert np.isfinite(prob).all()

    binary = (prob > THRESHOLD).astype(np.uint8)

    opened = cv2.morphologyEx(
        binary, cv2.MORPH_OPEN, kernel
    )
    processed = cv2.morphologyEx(
        opened, cv2.MORPH_CLOSE, kernel
    )

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        processed, connectivity=8
    )

    areas = stats[:, cv2.CC_STAT_AREA]

    sums = np.bincount(
        labels.ravel(),
        weights=prob.ravel(),
        minlength=n
    )

    confidence = sums / np.maximum(areas, 1)

    rows = []
    number = 0

    for cid in range(1, n):
        if areas[cid] < MIN_AREA:
            continue
        if confidence[cid] < CONFIDENCE:
            continue

        instance_mask = np.asfortranarray(
            (labels == cid).astype(np.uint8)
        )

        rle = mask_utils.encode(
            instance_mask
        )["counts"].decode("ascii")

        number += 1

        rows.append({
            "filament_id": f"{image_path.stem}_{number}",
            "segmentation_rle": rle
        })

    return rows


start_time = time.perf_counter()

for i, image_path in enumerate(test_images, 1):
    part_path = PARTS / (image_path.stem + ".csv")

    if part_path.exists():
        print(f"{i}/180 — already saved", flush=True)
        continue

    predictions = create_v5b_submission_rows(
        image_path
    )

    pd.DataFrame(
        predictions,
        columns=["filament_id", "segmentation_rle"]
    ).to_csv(part_path, index=False)

    if i % 10 == 0 or i == 1:
        print(
            f"{i}/180 — {image_path.name} — "
            f"{len(predictions)} instances",
            flush=True
        )


# Combine all photograph-level CSV files.
frames = []

for image_path in test_images:
    part_path = PARTS / (image_path.stem + ".csv")
    assert part_path.exists(), part_path

    part = pd.read_csv(part_path)

    if len(part):
        assert list(part.columns) == [
            "filament_id", "segmentation_rle"
        ]
        frames.append(part)

assert frames, "No filament predictions were generated."

submission = pd.concat(frames, ignore_index=True)

assert submission["filament_id"].is_unique
assert submission.notna().all().all()

submission.to_csv(OUTPUT, index=False)

print("\n" + "=" * 55)
print("V5-B 512x512 SUBMISSION GENERATED")
print("=" * 55)
print("Test images processed:", len(test_images))
print("Predicted filament instances:", len(submission))
print("CSV file:", OUTPUT)
print(
    "Elapsed minutes:",
    round((time.perf_counter() - start_time) / 60, 2)
)


In [ ]:

import pandas as pd
from pathlib import Path

df = pd.read_csv(
    "/kaggle/working/submission_v5b_512.csv"
)

valid_stems = {p.stem for p in test_images}

image_ids = df["filament_id"].str.rsplit(
    "_", n=1
).str[0]

predicted_stems = set(image_ids)

missing = sorted(valid_stems - predicted_stems)
unexpected = sorted(predicted_stems - valid_stems)

print("Total test images:", len(valid_stems))
print("Images with predictions:", len(predicted_stems))
print("Images without predictions:", len(missing))
print("Unexpected image IDs:", unexpected)
print("Images without predictions:", missing)

print("\nChecking saved per-image CSV files:")
for stem in missing:
    part = PARTS / f"{stem}.csv"
    if part.exists():
        rows = len(pd.read_csv(part))
        print(stem, "| saved rows:", rows)
    else:
        print(stem, "| PART FILE MISSING")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from pycocotools import mask as mask_utils

path = Path("/kaggle/working/submission_v5b_512.csv")
df = pd.read_csv(path)

assert list(df.columns) == [
    "filament_id", "segmentation_rle"
]
assert len(df) == 1224
assert df["filament_id"].is_unique
assert df.notna().all().all()

valid_stems = {p.stem for p in test_images}

image_ids = df["filament_id"].str.rsplit(
    "_", n=1
).str[0]

assert set(image_ids).issubset(valid_stems)

# Every test image must have a completed part file,
# even when the model predicts zero filaments.
for stem in valid_stems:
    part = PARTS / f"{stem}.csv"
    assert part.is_file(), f"Missing part: {stem}"

# Verify each encoded mask and its image dimensions.
for i, rle_text in enumerate(df["segmentation_rle"]):
    rle = {
        "size": [2048, 2048],
        "counts": rle_text.encode("ascii")
    }

    area = int(mask_utils.area(rle))
    assert area >= 800, (
        f"Invalid mask area in row {i}: {area}"
    )

print("CSV format: VALID")
print("Test images processed:", len(valid_stems))
print("Images with predictions:", image_ids.nunique())
print("Images with zero predictions:",
      len(valid_stems - set(image_ids)))
print("Unique filament IDs:", df.filament_id.nunique())
print("All RLE masks: VALID")
print("Ready for Kaggle submission: YES")


In [ ]:
from IPython.display import FileLink, display
from pathlib import Path

file = Path("/kaggle/working/submission_v5b_512.csv")

assert file.exists(), "Submission file not found!"

print("File found:", file.name)
print("Size:", round(file.stat().st_size / 1024 / 1024, 2), "MB")

display(FileLink(str(file)))